# ER-CyRIS Siklus 1 — STEP 2A
## S0 Baseline Supervised (XGBoost + Random Forest) pada 4 Dataset

Fokus tahap ini hanya **S0 supervised**.

Belum menjalankan Isolation Forest, LOF, DBSCAN, S1–S5, atau SHAP.

Dataset:
1. CICIDS2018
2. HDFS (representasi richer 22 fitur yang sudah lolos audit konflik label)
3. BGL
4. UNSW-NB15

Desain mempertahankan inti eksperimen Siklus 1 asli:
- split 80/20,
- XGBoost,
- Random Forest,
- Precision, Recall, F1, PR-AUC, FAR, Alert Rate.

Perbaikan metodologis:
- preprocessing di-fit hanya pada training,
- feature-vector identik tidak boleh tersebar antara train dan test,
- HDFS memakai representasi 22 fitur yang sudah diverifikasi,
- UNSW-NB15 dibaca sebagai file 49 kolom tanpa header,
- hasil disimpan terpisah sebagai confirmatory multi-dataset replication.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, zlib, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

print("✅ Environment ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Environment ready


In [6]:
BASE = '/content/drive/MyDrive/ercyris_siklus2/'
OUT = Path('/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/')
OUT.mkdir(parents=True, exist_ok=True)

CFG = {
    'CICIDS': BASE + '02-14-2018.csv',
    'BGL': BASE + 'BGL.log',
    'UNSW': BASE + 'UNSW-NB15_1.csv',
    'HDFS_CACHE': (
        '/content/drive/MyDrive/ercyris_siklus1_confirmatory/cache/'
        'HDFS_richer_22f_confirmatory.parquet'
    ),
    'MAX_CICIDS': 500_000,
    'MAX_BGL': 50_000,
    'TEST_SIZE': 0.20,
    'SEED': 42,
}

for k in ['CICIDS','BGL','UNSW','HDFS_CACHE']:
    p = CFG[k]
    print(f"{k:12s} {'✅' if os.path.exists(p) else '❌'} {p}")

if not os.path.exists(CFG['HDFS_CACHE']):
    raise FileNotFoundError(
        "HDFS cache belum ditemukan. Kembali ke notebook STEP 1 dan simpan "
        "X_hdfs2 + y_hdfs2 ke HDFS_richer_22f_confirmatory.parquet."
    )

CICIDS       ✅ /content/drive/MyDrive/ercyris_siklus2/02-14-2018.csv
BGL          ✅ /content/drive/MyDrive/ercyris_siklus2/BGL.log
UNSW         ✅ /content/drive/MyDrive/ercyris_siklus2/UNSW-NB15_1.csv
HDFS_CACHE   ✅ /content/drive/MyDrive/ercyris_siklus1_confirmatory/cache/HDFS_richer_22f_confirmatory.parquet


In [3]:
def clean_cols(cols):
    out = []
    for c in cols:
        c = str(c).strip().lower().replace(' ', '_').replace('/', '_per_')
        c = re.sub(r'[^a-z0-9_]+', '_', c).strip('_')
        out.append(c)
    return out

def stable_hash_1000(text):
    return zlib.crc32(str(text).encode('utf-8')) % 1000

def feature_groups(X):
    return pd.util.hash_pandas_object(
        X.reset_index(drop=True),
        index=False
    ).astype('uint64').values

def split_no_duplicate_leakage(X, y, test_size=0.20, seed=42):
    y = np.asarray(y, dtype=int)
    groups = feature_groups(X)
    n_unique = len(np.unique(groups))

    if n_unique == len(X):
        idx = np.arange(len(X))
        tr, te = train_test_split(
            idx,
            test_size=test_size,
            random_state=seed,
            stratify=y
        )
        method = 'stratified_unique'
    else:
        best = None
        overall_rate = y.mean()

        for rs in range(seed, seed + 200):
            gss = GroupShuffleSplit(
                n_splits=1,
                test_size=test_size,
                random_state=rs
            )
            tr0, te0 = next(gss.split(X, y, groups))

            if len(np.unique(y[tr0])) < 2 or len(np.unique(y[te0])) < 2:
                continue

            test_frac = len(te0) / len(X)
            test_rate = y[te0].mean()

            score = (
                abs(test_frac - test_size)
                + 2.0 * abs(test_rate - overall_rate)
            )

            if best is None or score < best[0]:
                best = (score, tr0, te0, rs)

        if best is None:
            raise RuntimeError("Tidak ditemukan group-aware split yang valid.")

        _, tr, te, chosen_seed = best
        method = f'group_aware_seed_{chosen_seed}'

        shared = np.intersect1d(
            np.unique(groups[tr]),
            np.unique(groups[te])
        )
        assert len(shared) == 0, "Feature-group leakage masih terjadi."

    audit = {
        'method': method,
        'n_total': len(X),
        'n_train': len(tr),
        'n_test': len(te),
        'train_pos_rate': float(y[tr].mean()),
        'test_pos_rate': float(y[te].mean()),
        'overall_pos_rate': float(y.mean()),
        'unique_feature_vectors': int(n_unique),
        'duplicate_rows': int(len(X) - n_unique),
    }
    return np.asarray(tr), np.asarray(te), audit

def make_preprocessor(Xtr):
    numeric_cols = list(Xtr.select_dtypes(include=np.number).columns)
    categorical_cols = [c for c in Xtr.columns if c not in numeric_cols]

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ])

    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ))
    ])

    pre = ColumnTransformer(
        transformers=[
            ('num', num_pipe, numeric_cols),
            ('cat', cat_pipe, categorical_cols),
        ],
        remainder='drop',
        sparse_threshold=0
    )

    pipe = Pipeline([
        ('columns', pre),
        ('scale', MinMaxScaler())
    ])
    return pipe

def calc_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0,1]
    ).ravel()

    N = tn + fp + fn + tp

    return {
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'PR_AUC': average_precision_score(y_true, y_score),
        'FAR': fp / max(fp + tn, 1),
        'Alert_Rate': (tp + fp) / max(N, 1),
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn),
        'TP': int(tp),
    }

In [4]:
def load_cicids():
    print("\n📂 CICIDS2018")

    df = pd.read_csv(CFG['CICIDS'], low_memory=False)

    if len(df) > CFG['MAX_CICIDS']:
        df = df.sample(
            n=CFG['MAX_CICIDS'],
            random_state=CFG['SEED']
        )

    df.columns = clean_cols(df.columns)

    label_col = next(c for c in df.columns if 'label' in c)
    y = (
        df[label_col].astype(str).str.strip().str.upper()
        != 'BENIGN'
    ).astype(int).values

    drop = [label_col] + [
        c for c in df.columns
        if any(k in c for k in [
            'timestamp','flow_id',
            'src_ip','dst_ip',
            'source_ip','destination_ip'
        ])
    ]

    X = (
        df.drop(columns=list(set(drop)), errors='ignore')
        .replace([np.inf,-np.inf], np.nan)
        .reset_index(drop=True)
    )

    tmp = X.copy()
    tmp['__label__'] = y
    feat = list(X.columns)

    before = len(tmp)
    tmp = tmp.drop_duplicates(
        subset=feat,
        keep='first'
    ).reset_index(drop=True)

    y = tmp.pop('__label__').astype(int).values
    X = tmp

    print(
        f"   {before:,} -> {len(X):,} rows | "
        f"positive={y.sum():,} ({100*y.mean():.3f}%)"
    )
    return X, y

In [5]:
def load_hdfs():
    print("\n📂 HDFS richer 22f")

    df = pd.read_parquet(CFG['HDFS_CACHE'])

    if '__label__' not in df.columns:
        raise ValueError("Kolom __label__ tidak ditemukan pada HDFS cache.")

    y = df.pop('__label__').astype(int).values
    X = df.reset_index(drop=True)

    print(
        f"   {len(X):,} rows | {X.shape[1]} features | "
        f"positive={y.sum():,} ({100*y.mean():.3f}%)"
    )
    return X, y

In [6]:
def load_bgl():
    print("\n📂 BGL")

    SM = {
        'INFO':1,'DEBUG':0,'WARN':2,'WARNING':2,'ERROR':3,'SEVERE':4,
        'FATAL':4,'APPFPO':3,'KERN':2,'KERNSVC':3,'APPFPW':2
    }

    AKW = [
        'fatal','error','fail','exception','abort','illegal',
        'segfault','panic','crash','out of memory','overflow'
    ]

    rows = []

    with open(CFG['BGL'], 'r', errors='replace') as f:
        for seq,line in enumerate(tqdm(f, desc='parse BGL', leave=False)):
            p = line.split(None, 8)
            if len(p) < 9:
                continue

            label = 0 if p[0] == '-' else 1
            sev = SM.get(p[6].upper(),1)
            msg = p[8].lower()

            try:
                hour = int(p[3].split(':')[0]) if ':' in p[3] else 12
            except:
                hour = 12

            rows.append({
                'sev': sev,
                'is_err': int(sev >= 3),
                'is_warn': int(sev == 2),
                'hour': hour,
                'off_hr': int(hour < 6),
                'peak_hr': int(8 <= hour <= 18),
                'err_kw': int(any(k in msg for k in AKW)),
                'has_num': int(bool(re.search(r'\b\d{4,}\b', msg))),
                'msg_len': min(len(msg),500),
                'comp_len': len(p[5]),
                'node_h': stable_hash_1000(p[4]),
                '__label__': label,
                '__seq__': seq
            })

    df = pd.DataFrame(rows)

    if len(df) > CFG['MAX_BGL']:
        selected = []
        for cls in [0,1]:
            g = df[df['__label__'] == cls]
            n = int(round(
                CFG['MAX_BGL'] * len(g) / len(df)
            ))
            selected.append(
                g.sample(
                    n=min(n,len(g)),
                    random_state=CFG['SEED']
                )
            )

        df = (
            pd.concat(selected)
            .sort_values('__seq__')
            .reset_index(drop=True)
        )

    y = df.pop('__label__').astype(int).values
    df.drop(columns=['__seq__'], inplace=True)
    X = df

    print(
        f"   {len(X):,} rows | {X.shape[1]} features | "
        f"positive={y.sum():,} ({100*y.mean():.3f}%)"
    )
    return X, y

In [7]:
UNSW_COLUMNS = [
    'srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes',
    'sttl','dttl','sloss','dloss','service','sload','dload','spkts','dpkts',
    'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth',
    'res_bdy_len','sjit','djit','stime','ltime','sintpkt','dintpkt',
    'tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl',
    'ct_flw_http_mthd','is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst',
    'ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
    'ct_dst_src_ltm','attack_cat','label'
]

def load_unsw():
    print("\n📂 UNSW-NB15")

    raw = pd.read_csv(
        CFG['UNSW'],
        header=None,
        low_memory=False
    )

    if raw.shape[1] != len(UNSW_COLUMNS):
        raise ValueError(
            f"UNSW columns={raw.shape[1]}, expected={len(UNSW_COLUMNS)}"
        )

    raw.columns = UNSW_COLUMNS

    y = (
        pd.to_numeric(raw['label'], errors='coerce')
        .fillna(0)
        .astype(int)
        .values
    )

    X = raw.drop(
        columns=[
            'label','attack_cat',
            'srcip','dstip',
            'stime','ltime'
        ],
        errors='ignore'
    ).copy()

    X.columns = clean_cols(X.columns)
    X.replace([np.inf,-np.inf], np.nan, inplace=True)

    print(
        f"   {len(X):,} rows | {X.shape[1]} features | "
        f"positive={y.sum():,} ({100*y.mean():.3f}%)"
    )
    return X.reset_index(drop=True), y

In [8]:
DATASETS = {
    'CICIDS2018': load_cicids(),
    'HDFS': load_hdfs(),
    'BGL': load_bgl(),
    'UNSW-NB15': load_unsw(),
}

print("\n✅ All four datasets loaded.")


📂 CICIDS2018
   500,000 -> 330,726 rows | positive=45,029 (13.615%)

📂 HDFS richer 22f
   575,061 rows | 22 features | positive=16,838 (2.928%)

📂 BGL


parse BGL: 0it [00:00, ?it/s]

   50,000 rows | 11 features | positive=3,670 (7.340%)

📂 UNSW-NB15
   700,001 rows | 43 features | positive=22,215 (3.174%)

✅ All four datasets loaded.


## Split Audit

Sebelum modeling, notebook mencatat:
- jumlah train/test,
- prevalensi positive train/test,
- jumlah feature vector unik,
- duplicate rows,
- metode split.

Untuk HDFS, BGL, dan UNSW-NB15, feature vector identik tidak boleh muncul sekaligus pada train dan test.

In [9]:
SPLITS = {}
split_rows = []

for name,(X,y) in DATASETS.items():
    print(f"\n🔀 Splitting {name} ...")

    tr, te, audit = split_no_duplicate_leakage(
        X, y,
        test_size=CFG['TEST_SIZE'],
        seed=CFG['SEED']
    )

    SPLITS[name] = (tr,te)
    split_rows.append({'dataset': name, **audit})
    print(audit)

split_audit_df = pd.DataFrame(split_rows)
display(split_audit_df)

split_audit_df.to_csv(
    OUT / 'S0_split_audit.csv',
    index=False
)

print("✅ Split audit saved.")


🔀 Splitting CICIDS2018 ...
{'method': 'stratified_unique', 'n_total': 330726, 'n_train': 264580, 'n_test': 66146, 'train_pos_rate': 0.1361516365560511, 'test_pos_rate': 0.13615335772382306, 'overall_pos_rate': 0.13615198079376886, 'unique_feature_vectors': 330726, 'duplicate_rows': 0}

🔀 Splitting HDFS ...
{'method': 'group_aware_seed_226', 'n_total': 575061, 'n_train': 460028, 'n_test': 115033, 'train_pos_rate': 0.02925691479649065, 'test_pos_rate': 0.029374179583250025, 'overall_pos_rate': 0.029280371995318757, 'unique_feature_vectors': 525274, 'duplicate_rows': 49787}

🔀 Splitting BGL ...
{'method': 'group_aware_seed_88', 'n_total': 50000, 'n_train': 39989, 'n_test': 10011, 'train_pos_rate': 0.07337017679861962, 'test_pos_rate': 0.07351912895814604, 'overall_pos_rate': 0.0734, 'unique_feature_vectors': 24119, 'duplicate_rows': 25881}

🔀 Splitting UNSW-NB15 ...
{'method': 'group_aware_seed_109', 'n_total': 700001, 'n_train': 559968, 'n_test': 140033, 'train_pos_rate': 0.031735742042

,dataset,method,n_total,n_train,n_test,train_pos_rate,test_pos_rate,overall_pos_rate,unique_feature_vectors,duplicate_rows
0,CICIDS2018,stratified_unique,330726,264580,66146,0.136152,0.136153,0.136152,330726,0
1,HDFS,group_aware_seed_226,575061,460028,115033,0.029257,0.029374,0.029280,525274,49787
2,BGL,group_aware_seed_88,50000,39989,10011,0.073370,0.073519,0.073400,24119,25881
3,UNSW-NB15,group_aware_seed_109,700001,559968,140033,0.031736,0.031735,0.031736,636833,63168


✅ Split audit saved.


## S0 Supervised Baseline

Model:
- XGBoost
- Random Forest

Keduanya dilatih pada seluruh training partition dan dievaluasi pada test partition yang sudah bebas feature-group leakage.

In [10]:
RESULTS = []

for name,(X,y) in DATASETS.items():

    print(f"\n{'='*74}")
    print(f"S0 SUPERVISED — {name}")
    print('='*74)

    tr,te = SPLITS[name]

    Xtr_raw = X.iloc[tr].copy()
    Xte_raw = X.iloc[te].copy()
    ytr = y[tr]
    yte = y[te]

    pre = make_preprocessor(Xtr_raw)

    t0 = time.perf_counter()
    Xtr = pre.fit_transform(Xtr_raw)
    Xte = pre.transform(Xte_raw)
    prep_sec = time.perf_counter() - t0

    print(
        f"Preprocessed: train={Xtr.shape}, test={Xte.shape}, "
        f"time={prep_sec:.1f}s"
    )

    models = {
        'XGBoost': XGBClassifier(
            random_state=CFG['SEED'],
            n_jobs=-1,
            eval_metric='logloss',
            tree_method='hist',
            verbosity=0
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=100,
            random_state=CFG['SEED'],
            n_jobs=-1
        )
    }

    for model_name, model in models.items():

        print(f"  ▶ {model_name}")

        t0 = time.perf_counter()
        model.fit(Xtr, ytr)
        train_sec = time.perf_counter() - t0

        t1 = time.perf_counter()
        y_pred = model.predict(Xte)
        y_score = model.predict_proba(Xte)[:,1]
        infer_sec = time.perf_counter() - t1

        m = calc_metrics(yte, y_pred, y_score)

        RESULTS.append({
            'dataset': name,
            'scenario': 'S0',
            'model': model_name,
            'n_train': len(tr),
            'n_test': len(te),
            'train_positive_rate': float(ytr.mean()),
            'test_positive_rate': float(yte.mean()),
            'preprocessing_sec': prep_sec,
            'train_sec': train_sec,
            'inference_sec': infer_sec,
            **m
        })

        print(
            f"     F1={m['F1']:.6f} | "
            f"Precision={m['Precision']:.6f} | "
            f"Recall={m['Recall']:.6f} | "
            f"PR-AUC={m['PR_AUC']:.6f} | "
            f"FAR={m['FAR']:.6f}"
        )

results_df = pd.DataFrame(RESULTS)
display(results_df)

results_df.to_csv(
    OUT / 'S0_supervised_results.csv',
    index=False
)

print("\n✅ STEP 2A completed and saved:")
print(OUT / 'S0_supervised_results.csv')


S0 SUPERVISED — CICIDS2018
Preprocessed: train=(264580, 78), test=(66146, 78), time=3.2s
  ▶ XGBoost
     F1=1.000000 | Precision=1.000000 | Recall=1.000000 | PR-AUC=1.000000 | FAR=0.000000
  ▶ RandomForest
     F1=0.999944 | Precision=1.000000 | Recall=0.999889 | PR-AUC=1.000000 | FAR=0.000000

S0 SUPERVISED — HDFS
Preprocessed: train=(460028, 22), test=(115033, 22), time=1.3s
  ▶ XGBoost
     F1=0.997039 | Precision=0.997630 | Recall=0.996449 | PR-AUC=0.999630 | FAR=0.000072
  ▶ RandomForest
     F1=0.997336 | Precision=0.997336 | Recall=0.997336 | PR-AUC=0.999592 | FAR=0.000081

S0 SUPERVISED — BGL
Preprocessed: train=(39989, 11), test=(10011, 11), time=0.0s
  ▶ XGBoost
     F1=0.926997 | Precision=0.939944 | Recall=0.914402 | PR-AUC=0.958336 | FAR=0.004636
  ▶ RandomForest
     F1=0.896033 | Precision=0.902204 | Recall=0.889946 | PR-AUC=0.904855 | FAR=0.007655

S0 SUPERVISED — UNSW-NB15
Preprocessed: train=(559968, 43), test=(140033, 43), time=5.4s
  ▶ XGBoost
     F1=0.953953 | P

,dataset,scenario,model,n_train,n_test,train_positive_rate,test_positive_rate,preprocessing_sec,train_sec,inference_sec,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S0,XGBoost,264580,66146,0.136152,0.136153,3.212609,2.312729,0.073902,1.000000,1.000000,1.000000,1.000000,0.000000,0.136153,57140,0,0,9006
1,CICIDS2018,S0,RandomForest,264580,66146,0.136152,0.136153,3.212609,3.866549,0.107964,1.000000,0.999889,0.999944,1.000000,0.000000,0.136138,57140,0,1,9005
2,HDFS,S0,XGBoost,460028,115033,0.029257,0.029374,1.317803,0.881450,0.071448,0.997630,0.996449,0.997039,0.999630,0.000072,0.029339,111646,8,12,3367
3,HDFS,S0,RandomForest,460028,115033,0.029257,0.029374,1.317803,9.273465,0.194245,0.997336,0.997336,0.997336,0.999592,0.000081,0.029374,111645,9,9,3370
4,BGL,S0,XGBoost,39989,10011,0.073370,0.073519,0.033450,0.271497,0.014310,0.939944,0.914402,0.926997,0.958336,0.004636,0.071521,9232,43,63,673
5,BGL,S0,RandomForest,39989,10011,0.073370,0.073519,0.033450,0.359478,0.070035,0.902204,0.889946,0.896033,0.904855,0.007655,0.072520,9204,71,81,655
6,UNSW-NB15,S0,XGBoost,559968,140033,0.031736,0.031735,5.400948,1.897633,0.091395,0.952242,0.955671,0.953953,0.989617,0.001571,0.031850,135376,213,197,4247
7,UNSW-NB15,S0,RandomForest,559968,140033,0.031736,0.031735,5.400948,12.710698,0.172784,0.957229,0.966922,0.962051,0.995544,0.001416,0.032057,135397,192,147,4297



✅ STEP 2A completed and saved:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S0_supervised_results.csv


## STOP

Jangan lanjut ke model unsupervised atau S1–S5.

Kirim ke ChatGPT dua tabel:
1. `S0_split_audit.csv`
2. `S0_supervised_results.csv`

Setelah diperiksa, baru dilanjutkan ke STEP 2B: Isolation Forest, LOF, dan DBSCAN.

In [11]:
# ============================================================
# STEP 2B-1 — S0 ISOLATION FOREST
# TRAIN-ONLY THRESHOLD, FULL HELD-OUT TEST
# ============================================================

from sklearn.ensemble import IsolationForest

IF_RESULTS = []

for name, (X, y) in DATASETS.items():

    print(f"\n{'='*74}")
    print(f"S0 ISOLATION FOREST — {name}")
    print('='*74)

    tr, te = SPLITS[name]

    Xtr_raw = X.iloc[tr].copy()
    Xte_raw = X.iloc[te].copy()

    ytr = y[tr]
    yte = y[te]

    # --------------------------------------------------------
    # Preprocessing fit ONLY pada training
    # --------------------------------------------------------
    pre_if = make_preprocessor(Xtr_raw)

    t0 = time.perf_counter()

    Xtr = pre_if.fit_transform(Xtr_raw)
    Xte = pre_if.transform(Xte_raw)

    prep_sec = time.perf_counter() - t0

    # --------------------------------------------------------
    # Isolation Forest
    # Tidak menggunakan label untuk fitting / threshold
    # --------------------------------------------------------
    model = IsolationForest(
        n_estimators=200,
        max_samples=min(4096, len(Xtr)),
        contamination='auto',
        random_state=CFG['SEED'],
        n_jobs=-1
    )

    t0 = time.perf_counter()
    model.fit(Xtr)
    train_sec = time.perf_counter() - t0

    # --------------------------------------------------------
    # Threshold calibration dari TRAINING SCORE saja
    # fixed unsupervised operating point = top 5%
    # --------------------------------------------------------
    rng = np.random.RandomState(CFG['SEED'])

    cal_n = min(50000, len(Xtr))

    cal_idx = rng.choice(
        len(Xtr),
        size=cal_n,
        replace=False
    )

    # score besar = lebih anomalous
    cal_score = -model.score_samples(
        Xtr[cal_idx]
    )

    threshold = float(
        np.quantile(cal_score, 0.95)
    )

    # --------------------------------------------------------
    # HELD-OUT TEST
    # --------------------------------------------------------
    t1 = time.perf_counter()

    test_score = -model.score_samples(Xte)

    y_pred = (
        test_score > threshold
    ).astype(int)

    infer_sec = time.perf_counter() - t1

    m = calc_metrics(
        yte,
        y_pred,
        test_score
    )

    row = {
        'dataset': name,
        'scenario': 'S0',
        'model': 'IsolationForest',
        'threshold_source':
            '95th_percentile_training_score',
        'threshold': threshold,

        'n_train': len(tr),
        'n_calibration': cal_n,
        'n_test': len(te),

        'train_positive_rate':
            float(ytr.mean()),

        'test_positive_rate':
            float(yte.mean()),

        'preprocessing_sec':
            prep_sec,

        'train_sec':
            train_sec,

        'inference_sec':
            infer_sec,

        **m
    }

    IF_RESULTS.append(row)

    print(
        f"Precision={m['Precision']:.6f} | "
        f"Recall={m['Recall']:.6f} | "
        f"F1={m['F1']:.6f} | "
        f"PR-AUC={m['PR_AUC']:.6f} | "
        f"FAR={m['FAR']:.6f} | "
        f"Alert={m['Alert_Rate']:.6f}"
    )


if_df = pd.DataFrame(IF_RESULTS)

display(if_df)

if_df.to_csv(
    OUT / 'S0_isolationforest_results.csv',
    index=False
)

print(
    "\n✅ Saved:",
    OUT / 'S0_isolationforest_results.csv'
)


S0 ISOLATION FOREST — CICIDS2018
Precision=0.000000 | Recall=0.000000 | F1=0.000000 | PR-AUC=0.114855 | FAR=0.059118 | Alert=0.051069

S0 ISOLATION FOREST — HDFS
Precision=0.495676 | Recall=0.865049 | F1=0.630229 | PR-AUC=0.614789 | FAR=0.026636 | Alert=0.051264

S0 ISOLATION FOREST — BGL
Precision=0.043152 | Recall=0.031250 | F1=0.036249 | PR-AUC=0.096167 | FAR=0.054987 | Alert=0.053241

S0 ISOLATION FOREST — UNSW-NB15
Precision=0.222512 | Recall=0.364311 | F1=0.276280 | PR-AUC=0.229014 | FAR=0.041722 | Alert=0.051959


,dataset,scenario,model,threshold_source,threshold,n_train,n_calibration,n_test,train_positive_rate,test_positive_rate,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S0,IsolationForest,95th_percentile_training_score,0.452999,264580,50000,66146,0.136152,0.136153,...,0.000000,0.000000,0.000000,0.114855,0.059118,0.051069,53762,3378,9006,0
1,HDFS,S0,IsolationForest,95th_percentile_training_score,0.530310,460028,50000,115033,0.029257,0.029374,...,0.495676,0.865049,0.630229,0.614789,0.026636,0.051264,108680,2974,456,2923
2,BGL,S0,IsolationForest,95th_percentile_training_score,0.587495,39989,39989,10011,0.073370,0.073519,...,0.043152,0.031250,0.036249,0.096167,0.054987,0.053241,8765,510,713,23
3,UNSW-NB15,S0,IsolationForest,95th_percentile_training_score,0.458734,559968,50000,140033,0.031736,0.031735,...,0.222512,0.364311,0.276280,0.229014,0.041722,0.051959,129932,5657,2825,1619



✅ Saved: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S0_isolationforest_results.csv


In [12]:
# ============================================================
# STEP 2B-2 — S0 LOCAL OUTLIER FACTOR
# scalable novelty-mode held-out evaluation
# ============================================================

from sklearn.neighbors import LocalOutlierFactor

LOF_RESULTS = []

LOF_REFERENCE_MAX = 50_000
LOF_CALIB_MAX   = 20_000
LOF_K           = 20

for name, (X, y) in DATASETS.items():

    print(f"\n{'='*74}")
    print(f"S0 LOF — {name}")
    print('='*74)

    tr, te = SPLITS[name]

    Xtr_raw = X.iloc[tr].copy()
    Xte_raw = X.iloc[te].copy()

    ytr = y[tr]
    yte = y[te]

    # --------------------------------------------------------
    # Preprocessing fit ONLY on training
    # --------------------------------------------------------
    pre_lof = make_preprocessor(Xtr_raw)

    t0 = time.perf_counter()

    Xtr = pre_lof.fit_transform(Xtr_raw)
    Xte = pre_lof.transform(Xte_raw)

    prep_sec = time.perf_counter() - t0

    # --------------------------------------------------------
    # Reference/calibration split FROM TRAIN ONLY
    # no label used for selection
    # --------------------------------------------------------
    rng = np.random.RandomState(CFG['SEED'])

    perm = rng.permutation(len(Xtr))

    ref_n = min(
        LOF_REFERENCE_MAX,
        max(1000, int(0.7 * len(Xtr)))
    )

    ref_idx = perm[:ref_n]

    remaining = perm[ref_n:]

    if len(remaining) == 0:
        raise RuntimeError(
            f"{name}: tidak ada data tersisa untuk calibration."
        )

    cal_n = min(
        LOF_CALIB_MAX,
        len(remaining)
    )

    cal_idx = remaining[:cal_n]

    X_ref = Xtr[ref_idx]
    X_cal = Xtr[cal_idx]

    # --------------------------------------------------------
    # LOF novelty=True:
    # fit hanya pada reference training
    # --------------------------------------------------------
    k = min(
        LOF_K,
        max(2, len(X_ref) - 1)
    )

    model = LocalOutlierFactor(
        n_neighbors=k,
        novelty=True,
        contamination='auto',
        n_jobs=-1
    )

    t0 = time.perf_counter()

    model.fit(X_ref)

    train_sec = time.perf_counter() - t0

    # --------------------------------------------------------
    # TRAIN-ONLY CALIBRATION
    # score besar = semakin anomalous
    # --------------------------------------------------------
    cal_score = -model.score_samples(X_cal)

    threshold = float(
        np.quantile(
            cal_score,
            0.95
        )
    )

    # --------------------------------------------------------
    # HELD-OUT TEST
    # --------------------------------------------------------
    t1 = time.perf_counter()

    test_score = -model.score_samples(Xte)

    y_pred = (
        test_score > threshold
    ).astype(int)

    infer_sec = time.perf_counter() - t1

    m = calc_metrics(
        yte,
        y_pred,
        test_score
    )

    row = {
        'dataset': name,
        'scenario': 'S0',
        'model': 'LOF',

        'n_neighbors': k,

        'reference_source':
            'training_partition_only',

        'n_reference':
            len(X_ref),

        'n_calibration':
            len(X_cal),

        'threshold_source':
            '95th_percentile_training_calibration_score',

        'threshold':
            threshold,

        'n_train_total':
            len(tr),

        'n_test':
            len(te),

        'train_positive_rate':
            float(ytr.mean()),

        'test_positive_rate':
            float(yte.mean()),

        'preprocessing_sec':
            prep_sec,

        'train_sec':
            train_sec,

        'inference_sec':
            infer_sec,

        **m
    }

    LOF_RESULTS.append(row)

    print(
        f"Reference={len(X_ref):,} | "
        f"Calibration={len(X_cal):,} | "
        f"k={k}"
    )

    print(
        f"Precision={m['Precision']:.6f} | "
        f"Recall={m['Recall']:.6f} | "
        f"F1={m['F1']:.6f} | "
        f"PR-AUC={m['PR_AUC']:.6f} | "
        f"FAR={m['FAR']:.6f} | "
        f"Alert={m['Alert_Rate']:.6f}"
    )


lof_df = pd.DataFrame(
    LOF_RESULTS
)

display(lof_df)

lof_df.to_csv(
    OUT / 'S0_lof_results.csv',
    index=False
)

print(
    "\n✅ Saved:",
    OUT / 'S0_lof_results.csv'
)


S0 LOF — CICIDS2018
Reference=50,000 | Calibration=20,000 | k=20
Precision=0.006803 | Recall=0.002443 | F1=0.003595 | PR-AUC=0.119942 | FAR=0.056213 | Alert=0.048892

S0 LOF — HDFS
Reference=50,000 | Calibration=20,000 | k=20
Precision=0.164217 | Recall=0.284404 | F1=0.208211 | PR-AUC=0.213167 | FAR=0.043805 | Alert=0.050872

S0 LOF — BGL
Reference=27,992 | Calibration=11,997 | k=20
Precision=0.037485 | Recall=0.042120 | F1=0.039667 | PR-AUC=0.061172 | FAR=0.085822 | Alert=0.082609

S0 LOF — UNSW-NB15
Reference=50,000 | Calibration=20,000 | k=20
Precision=0.118093 | Recall=0.189469 | F1=0.145499 | PR-AUC=0.068093 | FAR=0.046375 | Alert=0.050917


,dataset,scenario,model,n_neighbors,reference_source,n_reference,n_calibration,threshold_source,threshold,n_train_total,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S0,LOF,20,training_partition_only,50000,20000,95th_percentile_training_calibration_score,3.922599,264580,...,0.006803,0.002443,0.003595,0.119942,0.056213,0.048892,53928,3212,8984,22
1,HDFS,S0,LOF,20,training_partition_only,50000,20000,95th_percentile_training_calibration_score,1.698734,460028,...,0.164217,0.284404,0.208211,0.213167,0.043805,0.050872,106763,4891,2418,961
2,BGL,S0,LOF,20,training_partition_only,27992,11997,95th_percentile_training_calibration_score,1.364833,39989,...,0.037485,0.042120,0.039667,0.061172,0.085822,0.082609,8479,796,705,31
3,UNSW-NB15,S0,LOF,20,training_partition_only,50000,20000,95th_percentile_training_calibration_score,1.357404,559968,...,0.118093,0.189469,0.145499,0.068093,0.046375,0.050917,129301,6288,3602,842



✅ Saved: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S0_lof_results.csv


In [14]:
# ============================================================
# STEP 2B-3 — S0 DBSCAN
# inductive held-out evaluation
# ============================================================

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

DBSCAN_RESULTS = []

DB_REFERENCE_MAX = 15_000
DB_CALIB_MAX     = 10_000
DB_MIN_SAMPLES   = 10
DB_EPS_QUANTILE  = 0.95
SCORE_BATCH      = 10_000


def nearest_core_score(nn_core, X, batch_size=10000):
    """
    Anomaly score = distance to nearest DBSCAN core point.
    Batched agar aman untuk HDFS / UNSW.
    """
    out = []

    for start in range(0, len(X), batch_size):
        end = min(start + batch_size, len(X))

        d, _ = nn_core.kneighbors(
            X[start:end],
            n_neighbors=1
        )

        out.append(
            d[:, 0]
        )

    return np.concatenate(out)


for name, (X, y) in DATASETS.items():

    print(f"\n{'='*74}")
    print(f"S0 DBSCAN INDUCTIVE — {name}")
    print('='*74)

    tr, te = SPLITS[name]

    Xtr_raw = X.iloc[tr].copy()
    Xte_raw = X.iloc[te].copy()

    ytr = y[tr]
    yte = y[te]

    # --------------------------------------------------------
    # 1. PREPROCESSING — FIT TRAIN ONLY
    # --------------------------------------------------------
    pre_db = make_preprocessor(
        Xtr_raw
    )

    t0 = time.perf_counter()

    Xtr = pre_db.fit_transform(
        Xtr_raw
    )

    Xte = pre_db.transform(
        Xte_raw
    )

    prep_sec = (
        time.perf_counter() - t0
    )

    # --------------------------------------------------------
    # 2. REFERENCE + CALIBRATION
    #    training only, no labels
    # --------------------------------------------------------
    rng = np.random.RandomState(
        CFG['SEED']
    )

    perm = rng.permutation(
        len(Xtr)
    )

    ref_n = min(
        DB_REFERENCE_MAX,
        max(
            2000,
            int(0.60 * len(Xtr))
        )
    )

    ref_idx = perm[:ref_n]

    remaining = perm[ref_n:]

    if len(remaining) == 0:
        raise RuntimeError(
            f"{name}: tidak ada calibration subset."
        )

    cal_n = min(
        DB_CALIB_MAX,
        len(remaining)
    )

    cal_idx = remaining[:cal_n]

    X_ref = Xtr[ref_idx]
    X_cal = Xtr[cal_idx]

    # --------------------------------------------------------
    # 3. ESTIMATE EPS FROM TRAINING REFERENCE
    #
    # min_samples = 10
    # DBSCAN counts the sample itself, sehingga
    # neighbour ke-10 sesuai untuk k-distance.
    # --------------------------------------------------------
    k = min(
        DB_MIN_SAMPLES,
        len(X_ref)
    )

    t0 = time.perf_counter()

    nn_eps = NearestNeighbors(
        n_neighbors=k,
        metric='euclidean',
        n_jobs=-1
    )

    nn_eps.fit(
        X_ref
    )

    dist_ref, _ = nn_eps.kneighbors(
        X_ref
    )

    kth_distance = (
        dist_ref[:, -1]
    )

    eps = float(
        np.quantile(
            kth_distance,
            DB_EPS_QUANTILE
        )
    )

    print(
        f"Reference={len(X_ref):,} | "
        f"Calibration={len(X_cal):,} | "
        f"eps={eps:.6f}"
    )

    # --------------------------------------------------------
    # 4. DBSCAN FIT ON TRAINING REFERENCE ONLY
    # --------------------------------------------------------
    db = DBSCAN(
        eps=eps,
        min_samples=DB_MIN_SAMPLES,
        metric='euclidean',
        n_jobs=-1
    )

    db.fit(
        X_ref
    )

    train_sec = (
        time.perf_counter() - t0
    )

    n_clusters = len(
        set(db.labels_) - {-1}
    )

    n_noise_ref = int(
        np.sum(db.labels_ == -1)
    )

    n_core = len(
        db.core_sample_indices_
    )

    print(
        f"clusters={n_clusters:,} | "
        f"core={n_core:,} | "
        f"reference_noise={n_noise_ref:,}"
    )

    # --------------------------------------------------------
    # 5. SAFETY CHECK
    # --------------------------------------------------------
    if n_core == 0:

        print(
            "⚠️ Tidak ada core point; dataset dilewati."
        )

        DBSCAN_RESULTS.append({
            'dataset': name,
            'scenario': 'S0',
            'model': 'DBSCAN_inductive',
            'status': 'NO_CORE_POINTS',
            'n_reference': len(X_ref),
            'n_calibration': len(X_cal),
            'n_test': len(Xte),
            'eps': eps,
            'n_clusters': n_clusters,
            'n_core': 0,
            'reference_noise': n_noise_ref
        })

        continue

    # --------------------------------------------------------
    # 6. BUILD INDUCTIVE SCORER
    #
    # Test data NEVER refits DBSCAN.
    # Score = nearest distance to training core point.
    # --------------------------------------------------------
    core_points = X_ref[
        db.core_sample_indices_
    ]

    nn_core = NearestNeighbors(
        n_neighbors=1,
        metric='euclidean',
        n_jobs=-1
    )

    nn_core.fit(
        core_points
    )

    # --------------------------------------------------------
    # 7. TRAIN-ONLY CALIBRATION OF ANOMALY THRESHOLD
    # --------------------------------------------------------
    cal_score = nearest_core_score(
        nn_core,
        X_cal,
        SCORE_BATCH
    )

    threshold = float(
        np.quantile(
            cal_score,
            0.95
        )
    )

    # --------------------------------------------------------
    # 8. HELD-OUT TEST
    # --------------------------------------------------------
    t1 = time.perf_counter()

    test_score = nearest_core_score(
        nn_core,
        Xte,
        SCORE_BATCH
    )

    y_pred = (
        test_score > threshold
    ).astype(int)

    infer_sec = (
        time.perf_counter() - t1
    )

    # --------------------------------------------------------
    # 9. METRICS
    # --------------------------------------------------------
    m = calc_metrics(
        yte,
        y_pred,
        test_score
    )

    row = {
        'dataset': name,
        'scenario': 'S0',

        'model':
            'DBSCAN_inductive',

        'status':
            'OK',

        'reference_source':
            'training_partition_only',

        'n_reference':
            len(X_ref),

        'n_calibration':
            len(X_cal),

        'n_test':
            len(Xte),

        'min_samples':
            DB_MIN_SAMPLES,

        'eps_quantile':
            DB_EPS_QUANTILE,

        'eps':
            eps,

        'n_clusters':
            n_clusters,

        'n_core':
            n_core,

        'reference_noise':
            n_noise_ref,

        'threshold_source':
            '95th_percentile_training_calibration_core_distance',

        'threshold':
            threshold,

        'train_positive_rate':
            float(ytr.mean()),

        'test_positive_rate':
            float(yte.mean()),

        'preprocessing_sec':
            prep_sec,

        'train_sec':
            train_sec,

        'inference_sec':
            infer_sec,

        **m
    }

    DBSCAN_RESULTS.append(
        row
    )

    print(
        f"Precision={m['Precision']:.6f} | "
        f"Recall={m['Recall']:.6f} | "
        f"F1={m['F1']:.6f} | "
        f"PR-AUC={m['PR_AUC']:.6f} | "
        f"FAR={m['FAR']:.6f} | "
        f"Alert={m['Alert_Rate']:.6f}"
    )


dbscan_df = pd.DataFrame(
    DBSCAN_RESULTS
)

display(
    dbscan_df
)

dbscan_df.to_csv(
    OUT / 'S0_dbscan_inductive_results.csv',
    index=False
)

print(
    "\n✅ Saved:",
    OUT / 'S0_dbscan_inductive_results.csv'
)


S0 DBSCAN INDUCTIVE — CICIDS2018
Reference=15,000 | Calibration=10,000 | eps=0.128265
clusters=43 | core=14,250 | reference_noise=630
Precision=0.012627 | Recall=0.004553 | F1=0.006692 | PR-AUC=0.106004 | FAR=0.056108 | Alert=0.049088

S0 DBSCAN INDUCTIVE — HDFS
Reference=15,000 | Calibration=10,000 | eps=0.057760
clusters=42 | core=14,250 | reference_noise=557
Precision=0.277945 | Recall=0.491566 | F1=0.355104 | PR-AUC=0.396631 | FAR=0.038646 | Alert=0.051950

S0 DBSCAN INDUCTIVE — BGL
Reference=15,000 | Calibration=10,000 | eps=0.038592
clusters=12 | core=14,250 | reference_noise=530
Precision=0.055449 | Recall=0.039402 | F1=0.046068 | PR-AUC=0.092351 | FAR=0.053261 | Alert=0.052243

S0 DBSCAN INDUCTIVE — UNSW-NB15
Reference=15,000 | Calibration=10,000 | eps=0.461388
clusters=10 | core=14,250 | reference_noise=421
Precision=0.208126 | Recall=0.319307 | F1=0.251998 | PR-AUC=0.141467 | FAR=0.039819 | Alert=0.048689


,dataset,scenario,model,status,reference_source,n_reference,n_calibration,n_test,min_samples,eps_quantile,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S0,DBSCAN_inductive,OK,training_partition_only,15000,10000,66146,10,0.95,...,0.012627,0.004553,0.006692,0.106004,0.056108,0.049088,53934,3206,8965,41
1,HDFS,S0,DBSCAN_inductive,OK,training_partition_only,15000,10000,115033,10,0.95,...,0.277945,0.491566,0.355104,0.396631,0.038646,0.051950,107339,4315,1718,1661
2,BGL,S0,DBSCAN_inductive,OK,training_partition_only,15000,10000,10011,10,0.95,...,0.055449,0.039402,0.046068,0.092351,0.053261,0.052243,8781,494,707,29
3,UNSW-NB15,S0,DBSCAN_inductive,OK,training_partition_only,15000,10000,140033,10,0.95,...,0.208126,0.319307,0.251998,0.141467,0.039819,0.048689,130190,5399,3025,1419



✅ Saved: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S0_dbscan_inductive_results.csv


In [15]:
# ============================================================
# STEP 3 — S1 IMBALANCE ROBUSTNESS
# Multi-dataset confirmatory replication
# XGBoost + Random Forest only
# ============================================================

import gc

S1_RATIOS = [1.0, 0.10, 0.02, 0.01]
S1_RESULTS = []

# ------------------------------------------------------------
# Helper:
# target ratio = positive / negative
#
# Tidak melakukan oversampling.
# Jika target ratio lebih tinggi -> negative class dikurangi.
# Jika target ratio lebih rendah -> positive class dikurangi.
# ------------------------------------------------------------
def resample_to_pos_neg_ratio(X, y, target_ratio, seed=42):

    y = np.asarray(y, dtype=int)

    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    n_pos = len(pos_idx)
    n_neg = len(neg_idx)

    current_ratio = n_pos / n_neg

    rng = np.random.RandomState(seed)

    if target_ratio >= current_ratio:

        # Keep all positive samples.
        # Reduce negative samples.
        target_neg = int(
            np.floor(n_pos / target_ratio)
        )

        target_neg = min(
            target_neg,
            n_neg
        )

        neg_sel = rng.choice(
            neg_idx,
            size=target_neg,
            replace=False
        )

        pos_sel = pos_idx

    else:

        # Keep all negative samples.
        # Reduce positive samples.
        target_pos = int(
            np.floor(n_neg * target_ratio)
        )

        target_pos = min(
            target_pos,
            n_pos
        )

        target_pos = max(
            target_pos,
            1
        )

        pos_sel = rng.choice(
            pos_idx,
            size=target_pos,
            replace=False
        )

        neg_sel = neg_idx

    selected = np.concatenate([
        neg_sel,
        pos_sel
    ])

    rng.shuffle(selected)

    X_res = X[selected]
    y_res = y[selected]

    actual_pos = int(
        np.sum(y_res == 1)
    )

    actual_neg = int(
        np.sum(y_res == 0)
    )

    actual_ratio = (
        actual_pos / actual_neg
    )

    return (
        X_res,
        y_res,
        {
            'original_pos': n_pos,
            'original_neg': n_neg,
            'original_ratio': current_ratio,

            'resampled_pos': actual_pos,
            'resampled_neg': actual_neg,
            'actual_ratio': actual_ratio
        }
    )


# ------------------------------------------------------------
# Baseline F1 dari S0
# ------------------------------------------------------------
try:
    baseline_df = results_df.copy()
except NameError:
    baseline_df = pd.read_csv(
        OUT / 'S0_supervised_results.csv'
    )

baseline_f1 = {
    (r['dataset'], r['model']): r['F1']
    for _, r in baseline_df.iterrows()
}


# ============================================================
# RUN S1
# ============================================================
for name, (X, y) in DATASETS.items():

    print(
        f"\n{'='*76}"
    )

    print(
        f"S1 IMBALANCE — {name}"
    )

    print(
        '='*76
    )

    # --------------------------------------------------------
    # SAME S0 SPLIT
    # --------------------------------------------------------
    tr, te = SPLITS[name]

    Xtr_raw = X.iloc[tr].copy()
    Xte_raw = X.iloc[te].copy()

    ytr = y[tr]
    yte = y[te]

    # --------------------------------------------------------
    # Fit preprocessing ONCE on full S0 training.
    #
    # This keeps feature processing constant across ratios,
    # isolating class-ratio effects.
    # --------------------------------------------------------
    pre_s1 = make_preprocessor(
        Xtr_raw
    )

    t0 = time.perf_counter()

    Xtr_full = pre_s1.fit_transform(
        Xtr_raw
    )

    Xte = pre_s1.transform(
        Xte_raw
    )

    preprocessing_sec = (
        time.perf_counter() - t0
    )

    print(
        f"S0 training ratio "
        f"(positive/negative) = "
        f"{ytr.sum() / np.sum(ytr==0):.6f}"
    )

    # --------------------------------------------------------
    # EACH TARGET RATIO
    # --------------------------------------------------------
    for ratio in S1_RATIOS:

        print(
            f"\n--- target positive:negative = {ratio} ---"
        )

        Xres, yres, info = (
            resample_to_pos_neg_ratio(
                Xtr_full,
                ytr,
                target_ratio=ratio,
                seed=CFG['SEED']
            )
        )

        print(
            f"Training rows={len(yres):,} | "
            f"positive={info['resampled_pos']:,} | "
            f"negative={info['resampled_neg']:,} | "
            f"actual ratio={info['actual_ratio']:.6f}"
        )

        # ----------------------------------------------------
        # SAME SUPERVISED MODELS AS S0
        # ----------------------------------------------------
        models = {

            'XGBoost':
                XGBClassifier(
                    random_state=CFG['SEED'],
                    n_jobs=-1,
                    eval_metric='logloss',
                    tree_method='hist',
                    verbosity=0
                ),

            'RandomForest':
                RandomForestClassifier(
                    n_estimators=100,
                    random_state=CFG['SEED'],
                    n_jobs=-1
                )
        }


        for model_name, model in models.items():

            print(
                f"  ▶ {model_name}"
            )

            # ------------------------------------------------
            # TRAIN
            # ------------------------------------------------
            t1 = time.perf_counter()

            model.fit(
                Xres,
                yres
            )

            train_sec = (
                time.perf_counter() - t1
            )

            # ------------------------------------------------
            # SAME UNCHANGED S0 TEST SET
            # ------------------------------------------------
            t2 = time.perf_counter()

            y_pred = model.predict(
                Xte
            )

            y_score = model.predict_proba(
                Xte
            )[:, 1]

            inference_sec = (
                time.perf_counter() - t2
            )

            m = calc_metrics(
                yte,
                y_pred,
                y_score
            )

            f1_s0 = baseline_f1[
                (name, model_name)
            ]

            delta_f1 = (
                m['F1'] - f1_s0
            )

            pct_delta_f1 = (
                100 * delta_f1 / f1_s0
                if f1_s0 > 0
                else np.nan
            )

            row = {

                'dataset':
                    name,

                'scenario':
                    'S1',

                'model':
                    model_name,

                'target_pos_neg_ratio':
                    ratio,

                'original_train_ratio':
                    info['original_ratio'],

                'actual_train_ratio':
                    info['actual_ratio'],

                'original_positive':
                    info['original_pos'],

                'original_negative':
                    info['original_neg'],

                'resampled_positive':
                    info['resampled_pos'],

                'resampled_negative':
                    info['resampled_neg'],

                'n_train_resampled':
                    len(yres),

                'n_test':
                    len(yte),

                'baseline_F1_S0':
                    f1_s0,

                'delta_F1':
                    delta_f1,

                'pct_delta_F1':
                    pct_delta_f1,

                'preprocessing_sec':
                    preprocessing_sec,

                'train_sec':
                    train_sec,

                'inference_sec':
                    inference_sec,

                **m
            }

            S1_RESULTS.append(
                row
            )

            print(
                f"     F1={m['F1']:.6f} | "
                f"ΔF1={delta_f1:+.6f} | "
                f"({pct_delta_f1:+.2f}%) | "
                f"Recall={m['Recall']:.6f} | "
                f"FAR={m['FAR']:.6f}"
            )

        gc.collect()


# ============================================================
# SAVE
# ============================================================
s1_df = pd.DataFrame(
    S1_RESULTS
)

display(
    s1_df
)

s1_df.to_csv(
    OUT / 'S1_imbalance_supervised_results.csv',
    index=False
)


# ============================================================
# SIMPLE SUMMARY
# ============================================================
summary_s1 = (
    s1_df[
        [
            'dataset',
            'model',
            'target_pos_neg_ratio',
            'actual_train_ratio',
            'F1',
            'delta_F1',
            'pct_delta_F1',
            'Recall',
            'FAR'
        ]
    ]
    .sort_values(
        [
            'dataset',
            'model',
            'target_pos_neg_ratio'
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

display(
    summary_s1
)

summary_s1.to_csv(
    OUT / 'S1_imbalance_summary.csv',
    index=False
)

print(
    "\n✅ STEP 3 S1 selesai"
)

print(
    "Saved:",
    OUT / 'S1_imbalance_supervised_results.csv'
)


S1 IMBALANCE — CICIDS2018
S0 training ratio (positive/negative) = 0.157611

--- target positive:negative = 1.0 ---
Training rows=72,046 | positive=36,023 | negative=36,023 | actual ratio=1.000000
  ▶ XGBoost
     F1=1.000000 | ΔF1=+0.000000 | (+0.00%) | Recall=1.000000 | FAR=0.000000
  ▶ RandomForest
     F1=0.999944 | ΔF1=+0.000000 | (+0.00%) | Recall=0.999889 | FAR=0.000000

--- target positive:negative = 0.1 ---
Training rows=251,412 | positive=22,855 | negative=228,557 | actual ratio=0.099997
  ▶ XGBoost
     F1=1.000000 | ΔF1=+0.000000 | (+0.00%) | Recall=1.000000 | FAR=0.000000
  ▶ RandomForest
     F1=0.999944 | ΔF1=+0.000000 | (+0.00%) | Recall=0.999889 | FAR=0.000000

--- target positive:negative = 0.02 ---
Training rows=233,128 | positive=4,571 | negative=228,557 | actual ratio=0.019999
  ▶ XGBoost
     F1=0.999833 | ΔF1=-0.000167 | (-0.02%) | Recall=0.999667 | FAR=0.000000
  ▶ RandomForest
     F1=0.999833 | ΔF1=-0.000111 | (-0.01%) | Recall=0.999667 | FAR=0.000000

--- tar

,dataset,scenario,model,target_pos_neg_ratio,original_train_ratio,actual_train_ratio,original_positive,original_negative,resampled_positive,resampled_negative,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S1,XGBoost,1.00,0.157611,1.000000,36023,228557,36023,36023,...,1.000000,1.000000,1.000000,1.000000,0.000000,0.136153,57140,0,0,9006
1,CICIDS2018,S1,RandomForest,1.00,0.157611,1.000000,36023,228557,36023,36023,...,1.000000,0.999889,0.999944,1.000000,0.000000,0.136138,57140,0,1,9005
2,CICIDS2018,S1,XGBoost,0.10,0.157611,0.099997,36023,228557,22855,228557,...,1.000000,1.000000,1.000000,1.000000,0.000000,0.136153,57140,0,0,9006
3,CICIDS2018,S1,RandomForest,0.10,0.157611,0.099997,36023,228557,22855,228557,...,1.000000,0.999889,0.999944,1.000000,0.000000,0.136138,57140,0,1,9005
4,CICIDS2018,S1,XGBoost,0.02,0.157611,0.019999,36023,228557,4571,228557,...,1.000000,0.999667,0.999833,1.000000,0.000000,0.136108,57140,0,3,9003
5,CICIDS2018,S1,RandomForest,0.02,0.157611,0.019999,36023,228557,4571,228557,...,1.000000,0.999667,0.999833,1.000000,0.000000,0.136108,57140,0,3,9003
6,CICIDS2018,S1,XGBoost,0.01,0.157611,0.009998,36023,228557,2285,228557,...,1.000000,0.998668,0.999333,0.999999,0.000000,0.135972,57140,0,12,8994
7,CICIDS2018,S1,RandomForest,0.01,0.157611,0.009998,36023,228557,2285,228557,...,1.000000,0.999334,0.999667,1.000000,0.000000,0.136063,57140,0,6,9000
8,HDFS,S1,XGBoost,1.00,0.030139,1.000000,13459,446569,13459,13459,...,0.978545,0.998816,0.988576,0.997547,0.000663,0.029983,111580,74,4,3375
9,HDFS,S1,RandomForest,1.00,0.030139,1.000000,13459,446569,13459,13459,...,0.978545,0.998816,0.988576,0.995575,0.000663,0.029983,111580,74,4,3375


,dataset,model,target_pos_neg_ratio,actual_train_ratio,F1,delta_F1,pct_delta_F1,Recall,FAR
17,BGL,RandomForest,1.00,1.000000,0.779923,-0.116110,-12.958236,0.960598,0.039892
19,BGL,RandomForest,0.10,0.100000,0.886653,-0.009380,-1.046790,0.898098,0.010135
21,BGL,RandomForest,0.02,0.019997,0.835821,-0.060212,-6.719836,0.760870,0.004744
23,BGL,RandomForest,0.01,0.009985,0.730609,-0.165424,-18.461823,0.595109,0.002695
16,BGL,XGBoost,1.00,1.000000,0.782372,-0.144625,-15.601460,0.976902,0.041294
18,BGL,XGBoost,0.10,0.100000,0.928033,0.001036,0.111721,0.919837,0.004960
20,BGL,XGBoost,0.02,0.019997,0.836090,-0.090907,-9.806612,0.755435,0.004097
22,BGL,XGBoost,0.01,0.009985,0.769231,-0.157766,-17.019088,0.645380,0.002588
1,CICIDS2018,RandomForest,1.00,1.000000,0.999944,0.000000,0.000000,0.999889,0.000000
3,CICIDS2018,RandomForest,0.10,0.099997,0.999944,0.000000,0.000000,0.999889,0.000000



✅ STEP 3 S1 selesai
Saved: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S1_imbalance_supervised_results.csv


In [16]:
# ============================================================
# STEP 4A — S2 NOISE + MISSING ROBUSTNESS
# Multi-dataset confirmatory replication
# XGBoost + Random Forest ONLY
# ============================================================

import gc

S2_NOISE_SIGMAS = [0.01, 0.05]
S2_MISSING_RATES = [0.01, 0.05]

S2_RESULTS = []

# ------------------------------------------------------------
# Baseline S0 supervised
# ------------------------------------------------------------
try:
    baseline_df = results_df.copy()
except NameError:
    baseline_df = pd.read_csv(
        OUT / 'S0_supervised_results.csv'
    )

baseline_f1 = {
    (r['dataset'], r['model']): r['F1']
    for _, r in baseline_df.iterrows()
}


# ============================================================
# HELPER — inject missing values only into eligible numeric cols
# ============================================================
def inject_missing_raw(
    Xraw,
    eligible_cols,
    rate,
    seed=42
):
    Xmiss = Xraw.copy()

    rng = np.random.RandomState(seed)

    if len(eligible_cols) == 0:
        return Xmiss

    n_rows = len(Xmiss)
    n_cols = len(eligible_cols)

    total_cells = n_rows * n_cols

    n_missing = int(
        round(total_cells * rate)
    )

    if n_missing <= 0:
        return Xmiss

    flat_idx = rng.choice(
        total_cells,
        size=n_missing,
        replace=False
    )

    row_idx, col_idx = np.unravel_index(
        flat_idx,
        (n_rows, n_cols)
    )

    for j, col in enumerate(eligible_cols):

        rows_j = row_idx[
            col_idx == j
        ]

        if len(rows_j):
            Xmiss.iloc[
                rows_j,
                Xmiss.columns.get_loc(col)
            ] = np.nan

    return Xmiss


# ============================================================
# RUN S2
# ============================================================
for name, (X, y) in DATASETS.items():

    print(
        f"\n{'='*76}"
    )

    print(
        f"S2 NOISE + MISSING — {name}"
    )

    print(
        '='*76
    )

    # --------------------------------------------------------
    # SAME S0 TRAIN / TEST PARTITIONS
    # --------------------------------------------------------
    tr, te = SPLITS[name]

    Xtr_raw = X.iloc[tr].copy()
    Xte_raw = X.iloc[te].copy()

    ytr = y[tr]
    yte = y[te]

    # --------------------------------------------------------
    # Identify numeric telemetry
    # --------------------------------------------------------
    numeric_cols = list(
        Xtr_raw
        .select_dtypes(include=np.number)
        .columns
    )

    # node_h adalah hash categorical-like pada BGL,
    # jangan diberi Gaussian noise.
    noise_cols = [
        c for c in numeric_cols
        if c not in ['node_h']
    ]

    # make_preprocessor menaruh numeric columns lebih dahulu.
    noise_positions = [
        numeric_cols.index(c)
        for c in noise_cols
    ]

    print(
        f"Numeric={len(numeric_cols)} | "
        f"noise-eligible={len(noise_cols)}"
    )

    # --------------------------------------------------------
    # PREPROCESSING FIT ONLY ON TRAIN
    # --------------------------------------------------------
    pre_s2 = make_preprocessor(
        Xtr_raw
    )

    t0 = time.perf_counter()

    Xtr_p = pre_s2.fit_transform(
        Xtr_raw
    )

    Xte_clean = pre_s2.transform(
        Xte_raw
    )

    preprocessing_sec = (
        time.perf_counter() - t0
    )

    # --------------------------------------------------------
    # SAME MODELS AS S0
    # --------------------------------------------------------
    models = {

        'XGBoost':
            XGBClassifier(
                random_state=CFG['SEED'],
                n_jobs=-1,
                eval_metric='logloss',
                tree_method='hist',
                verbosity=0
            ),

        'RandomForest':
            RandomForestClassifier(
                n_estimators=100,
                random_state=CFG['SEED'],
                n_jobs=-1
            )
    }

    # train ONCE on clean S0 training data
    for model_name, model in models.items():

        print(
            f"\n▶ Training {model_name}"
        )

        model.fit(
            Xtr_p,
            ytr
        )

    # ========================================================
    # A. GAUSSIAN NOISE
    # ========================================================
    for sigma in S2_NOISE_SIGMAS:

        print(
            f"\n--- Gaussian noise sigma={sigma} ---"
        )

        rng = np.random.RandomState(
            CFG['SEED']
            + int(sigma * 10000)
        )

        Xnoise = Xte_clean.copy()

        # noise only on continuous/numeric transformed columns
        noise = rng.normal(
            loc=0.0,
            scale=sigma,
            size=(
                len(Xnoise),
                len(noise_positions)
            )
        )

        Xnoise[:, noise_positions] = (
            Xnoise[:, noise_positions]
            + noise
        )

        # mengikuti eksperimen awal:
        # normalized telemetry dibatasi 0..1
        Xnoise[:, noise_positions] = np.clip(
            Xnoise[:, noise_positions],
            0.0,
            1.0
        )

        for model_name, model in models.items():

            y_pred = model.predict(
                Xnoise
            )

            y_score = model.predict_proba(
                Xnoise
            )[:, 1]

            m = calc_metrics(
                yte,
                y_pred,
                y_score
            )

            f1_s0 = baseline_f1[
                (name, model_name)
            ]

            delta_f1 = (
                m['F1'] - f1_s0
            )

            pct_delta_f1 = (
                100 * delta_f1 / f1_s0
                if f1_s0 > 0
                else np.nan
            )

            S2_RESULTS.append({

                'dataset':
                    name,

                'scenario':
                    'S2_noise',

                'condition':
                    f'sigma={sigma}',

                'noise_sigma':
                    sigma,

                'missing_rate':
                    np.nan,

                'model':
                    model_name,

                'baseline_F1_S0':
                    f1_s0,

                'delta_F1':
                    delta_f1,

                'pct_delta_F1':
                    pct_delta_f1,

                **m
            })

            print(
                f"{model_name:13s} | "
                f"F1={m['F1']:.6f} | "
                f"Δ={pct_delta_f1:+.2f}% | "
                f"Recall={m['Recall']:.6f} | "
                f"FAR={m['FAR']:.6f}"
            )

    # ========================================================
    # B. MISSING TELEMETRY
    # ========================================================
    for rate in S2_MISSING_RATES:

        print(
            f"\n--- Missing numeric telemetry={rate:.0%} ---"
        )

        Xmiss_raw = inject_missing_raw(

            Xte_raw,

            eligible_cols=noise_cols,

            rate=rate,

            seed=(
                CFG['SEED']
                + int(rate * 10000)
            )
        )

        # median imputation parameters came ONLY from training
        Xmiss = pre_s2.transform(
            Xmiss_raw
        )

        for model_name, model in models.items():

            y_pred = model.predict(
                Xmiss
            )

            y_score = model.predict_proba(
                Xmiss
            )[:, 1]

            m = calc_metrics(
                yte,
                y_pred,
                y_score
            )

            f1_s0 = baseline_f1[
                (name, model_name)
            ]

            delta_f1 = (
                m['F1'] - f1_s0
            )

            pct_delta_f1 = (
                100 * delta_f1 / f1_s0
                if f1_s0 > 0
                else np.nan
            )

            S2_RESULTS.append({

                'dataset':
                    name,

                'scenario':
                    'S2_missing',

                'condition':
                    f'missing={rate}',

                'noise_sigma':
                    np.nan,

                'missing_rate':
                    rate,

                'model':
                    model_name,

                'baseline_F1_S0':
                    f1_s0,

                'delta_F1':
                    delta_f1,

                'pct_delta_F1':
                    pct_delta_f1,

                **m
            })

            print(
                f"{model_name:13s} | "
                f"F1={m['F1']:.6f} | "
                f"Δ={pct_delta_f1:+.2f}% | "
                f"Recall={m['Recall']:.6f} | "
                f"FAR={m['FAR']:.6f}"
            )

    gc.collect()


# ============================================================
# SAVE
# ============================================================
s2_df = pd.DataFrame(
    S2_RESULTS
)

display(
    s2_df
)

s2_df.to_csv(
    OUT / 'S2_noise_missing_supervised_results.csv',
    index=False
)


# ============================================================
# SUMMARY
# ============================================================
s2_summary = (
    s2_df[
        [
            'dataset',
            'model',
            'scenario',
            'condition',
            'F1',
            'delta_F1',
            'pct_delta_F1',
            'Recall',
            'PR_AUC',
            'FAR'
        ]
    ]
    .sort_values(
        [
            'dataset',
            'model',
            'scenario',
            'condition'
        ]
    )
)

display(
    s2_summary
)

s2_summary.to_csv(
    OUT / 'S2_noise_missing_summary.csv',
    index=False
)

print(
    "\n✅ STEP 4A selesai."
)

print(
    "Saved:",
    OUT / 'S2_noise_missing_supervised_results.csv'
)


S2 NOISE + MISSING — CICIDS2018
Numeric=78 | noise-eligible=78

▶ Training XGBoost

▶ Training RandomForest

--- Gaussian noise sigma=0.01 ---
XGBoost       | F1=0.239610 | Δ=-76.04% | Recall=0.136354 | FAR=0.000280
RandomForest  | F1=0.054439 | Δ=-94.56% | Recall=0.027981 | FAR=0.000000

--- Gaussian noise sigma=0.05 ---
XGBoost       | F1=0.245560 | Δ=-75.44% | Recall=0.140462 | FAR=0.000560
RandomForest  | F1=0.012140 | Δ=-98.79% | Recall=0.006107 | FAR=0.000000

--- Missing numeric telemetry=1% ---
XGBoost       | F1=0.987976 | Δ=-1.20% | Recall=0.976238 | FAR=0.000000
RandomForest  | F1=0.999944 | Δ=+0.00% | Recall=0.999889 | FAR=0.000000

--- Missing numeric telemetry=5% ---
XGBoost       | F1=0.951394 | Δ=-4.86% | Recall=0.907395 | FAR=0.000018
RandomForest  | F1=0.999500 | Δ=-0.04% | Recall=0.999001 | FAR=0.000000

S2 NOISE + MISSING — HDFS
Numeric=22 | noise-eligible=22

▶ Training XGBoost

▶ Training RandomForest

--- Gaussian noise sigma=0.01 ---
XGBoost       | F1=0.799063

,dataset,scenario,condition,noise_sigma,missing_rate,model,baseline_F1_S0,delta_F1,pct_delta_F1,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S2_noise,sigma=0.01,0.01,NaN,XGBoost,1.000000,-0.760390,-76.039024,0.987138,0.136354,0.239610,0.719196,0.000280,0.018807,57124,16,7778,1228
1,CICIDS2018,S2_noise,sigma=0.01,0.01,NaN,RandomForest,0.999944,-0.945505,-94.555757,1.000000,0.027981,0.054439,0.781659,0.000000,0.003810,57140,0,8754,252
2,CICIDS2018,S2_noise,sigma=0.05,0.05,NaN,XGBoost,1.000000,-0.754440,-75.444045,0.975328,0.140462,0.245560,0.655178,0.000560,0.019608,57108,32,7741,1265
3,CICIDS2018,S2_noise,sigma=0.05,0.05,NaN,RandomForest,0.999944,-0.987805,-98.785939,1.000000,0.006107,0.012140,0.593532,0.000000,0.000831,57140,0,8951,55
4,CICIDS2018,S2_missing,missing=0.01,NaN,0.01,XGBoost,1.000000,-0.012024,-1.202382,1.000000,0.976238,0.987976,0.999935,0.000000,0.132918,57140,0,214,8792
5,CICIDS2018,S2_missing,missing=0.01,NaN,0.01,RandomForest,0.999944,0.000000,0.000000,1.000000,0.999889,0.999944,1.000000,0.000000,0.136138,57140,0,1,9005
6,CICIDS2018,S2_missing,missing=0.05,NaN,0.05,XGBoost,1.000000,-0.048606,-4.860586,0.999878,0.907395,0.951394,0.999423,0.000018,0.123560,57139,1,834,8172
7,CICIDS2018,S2_missing,missing=0.05,NaN,0.05,RandomForest,0.999944,-0.000444,-0.044442,1.000000,0.999001,0.999500,1.000000,0.000000,0.136017,57140,0,9,8997
8,HDFS,S2_noise,sigma=0.01,0.01,NaN,XGBoost,0.997039,-0.197976,-19.856424,0.685068,0.958568,0.799063,0.943371,0.013336,0.041101,110165,1489,140,3239
9,HDFS,S2_noise,sigma=0.01,0.01,NaN,RandomForest,0.997336,-0.350375,-35.131029,0.480980,0.987866,0.646962,0.980322,0.032260,0.060331,108052,3602,41,3338


,dataset,model,scenario,condition,F1,delta_F1,pct_delta_F1,Recall,PR_AUC,FAR
21,BGL,RandomForest,S2_missing,missing=0.01,0.879076,-0.016957,-1.892424,0.879076,0.898308,0.009596
23,BGL,RandomForest,S2_missing,missing=0.05,0.829630,-0.066403,-7.410800,0.836957,0.881494,0.014340
17,BGL,RandomForest,S2_noise,sigma=0.01,0.364532,-0.531501,-59.317114,0.351902,0.303654,0.045930
19,BGL,RandomForest,S2_noise,sigma=0.05,0.196933,-0.699100,-78.021675,0.165761,0.142494,0.041078
20,BGL,XGBoost,S2_missing,missing=0.01,0.907724,-0.019273,-2.079121,0.902174,0.941745,0.006792
22,BGL,XGBoost,S2_missing,missing=0.05,0.859079,-0.067919,-7.326737,0.861413,0.888250,0.011429
16,BGL,XGBoost,S2_noise,sigma=0.01,0.215943,-0.711054,-76.705056,0.145380,0.227745,0.015957
18,BGL,XGBoost,S2_noise,sigma=0.05,0.111902,-0.815095,-87.928514,0.074728,0.149941,0.020701
5,CICIDS2018,RandomForest,S2_missing,missing=0.01,0.999944,0.000000,0.000000,0.999889,1.000000,0.000000
7,CICIDS2018,RandomForest,S2_missing,missing=0.05,0.999500,-0.000444,-0.044442,0.999001,1.000000,0.000000



✅ STEP 4A selesai.
Saved: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S2_noise_missing_supervised_results.csv


In [17]:
# ============================================================
# STEP 4B-1 — S2 ISOLATION FOREST
# Clean-training -> fixed train-calibrated threshold
# -> perturbed held-out test
# ============================================================

from sklearn.ensemble import IsolationForest
import numpy as np
import pandas as pd
import time
import gc

S2_IF_RESULTS = []

S2_NOISE_SIGMAS = [0.01, 0.05]
S2_MISSING_RATES = [0.01, 0.05]


# ------------------------------------------------------------
# Load S0 Isolation Forest baseline
# ------------------------------------------------------------
s0_if = pd.read_csv(
    OUT / 'S0_isolationforest_results.csv'
)

baseline_if_f1 = {
    r['dataset']: r['F1']
    for _, r in s0_if.iterrows()
}


# ------------------------------------------------------------
# helper missing
# ------------------------------------------------------------
def inject_missing_raw_if(
    Xraw,
    eligible_cols,
    rate,
    seed=42
):
    Xmiss = Xraw.copy()

    if len(eligible_cols) == 0:
        return Xmiss

    rng = np.random.RandomState(seed)

    n_rows = len(Xmiss)
    n_cols = len(eligible_cols)

    total_cells = n_rows * n_cols

    n_missing = int(
        round(total_cells * rate)
    )

    flat_idx = rng.choice(
        total_cells,
        size=n_missing,
        replace=False
    )

    row_idx, col_idx = np.unravel_index(
        flat_idx,
        (n_rows, n_cols)
    )

    for j, col in enumerate(eligible_cols):

        rows_j = row_idx[
            col_idx == j
        ]

        if len(rows_j):

            Xmiss.iloc[
                rows_j,
                Xmiss.columns.get_loc(col)
            ] = np.nan

    return Xmiss


# ============================================================
# RUN
# ============================================================
for name, (X, y) in DATASETS.items():

    print(f"\n{'='*76}")
    print(f"S2 ISOLATION FOREST — {name}")
    print('='*76)

    tr, te = SPLITS[name]

    Xtr_raw = X.iloc[tr].copy()
    Xte_raw = X.iloc[te].copy()

    ytr = y[tr]
    yte = y[te]

    # --------------------------------------------------------
    # numeric telemetry eligible for perturbation
    # --------------------------------------------------------
    numeric_cols = list(
        Xtr_raw
        .select_dtypes(include=np.number)
        .columns
    )

    noise_cols = [
        c for c in numeric_cols
        if c not in ['node_h']
    ]

    noise_positions = [
        numeric_cols.index(c)
        for c in noise_cols
    ]

    # --------------------------------------------------------
    # train-only preprocessing
    # --------------------------------------------------------
    pre_if = make_preprocessor(
        Xtr_raw
    )

    Xtr = pre_if.fit_transform(
        Xtr_raw
    )

    Xte_clean = pre_if.transform(
        Xte_raw
    )

    # --------------------------------------------------------
    # clean-training Isolation Forest
    # --------------------------------------------------------
    model = IsolationForest(
        n_estimators=200,
        max_samples=min(
            4096,
            len(Xtr)
        ),
        contamination='auto',
        random_state=CFG['SEED'],
        n_jobs=-1
    )

    t0 = time.perf_counter()

    model.fit(
        Xtr
    )

    train_sec = (
        time.perf_counter()
        - t0
    )

    # --------------------------------------------------------
    # fixed threshold from CLEAN TRAINING only
    # --------------------------------------------------------
    rng_cal = np.random.RandomState(
        CFG['SEED']
    )

    cal_n = min(
        50_000,
        len(Xtr)
    )

    cal_idx = rng_cal.choice(
        len(Xtr),
        size=cal_n,
        replace=False
    )

    cal_score = -model.score_samples(
        Xtr[cal_idx]
    )

    threshold = float(
        np.quantile(
            cal_score,
            0.95
        )
    )

    print(
        f"threshold={threshold:.6f} | "
        f"calibration={cal_n:,}"
    )

    # ========================================================
    # A. NOISE
    # ========================================================
    for sigma in S2_NOISE_SIGMAS:

        rng = np.random.RandomState(
            CFG['SEED']
            + int(sigma * 10000)
        )

        Xpert = Xte_clean.copy()

        noise = rng.normal(
            0.0,
            sigma,
            size=(
                len(Xpert),
                len(noise_positions)
            )
        )

        Xpert[:, noise_positions] += noise

        Xpert[:, noise_positions] = np.clip(
            Xpert[:, noise_positions],
            0.0,
            1.0
        )

        t1 = time.perf_counter()

        score = -model.score_samples(
            Xpert
        )

        y_pred = (
            score > threshold
        ).astype(int)

        infer_sec = (
            time.perf_counter()
            - t1
        )

        m = calc_metrics(
            yte,
            y_pred,
            score
        )

        base_f1 = baseline_if_f1[name]

        delta = (
            m['F1']
            - base_f1
        )

        pct = (
            100 * delta / base_f1
            if base_f1 > 0
            else np.nan
        )

        S2_IF_RESULTS.append({

            'dataset':
                name,

            'scenario':
                'S2_noise',

            'condition':
                f'sigma={sigma}',

            'model':
                'IsolationForest',

            'threshold':
                threshold,

            'baseline_F1_S0':
                base_f1,

            'delta_F1':
                delta,

            'pct_delta_F1':
                pct,

            'train_sec':
                train_sec,

            'inference_sec':
                infer_sec,

            **m
        })

        print(
            f"Noise {sigma}: "
            f"F1={m['F1']:.6f} | "
            f"Recall={m['Recall']:.6f} | "
            f"FAR={m['FAR']:.6f}"
        )


    # ========================================================
    # B. MISSING
    # ========================================================
    for rate in S2_MISSING_RATES:

        Xmiss_raw = inject_missing_raw_if(

            Xte_raw,

            eligible_cols=noise_cols,

            rate=rate,

            seed=(
                CFG['SEED']
                + int(rate * 10000)
            )
        )

        Xpert = pre_if.transform(
            Xmiss_raw
        )

        t1 = time.perf_counter()

        score = -model.score_samples(
            Xpert
        )

        y_pred = (
            score > threshold
        ).astype(int)

        infer_sec = (
            time.perf_counter()
            - t1
        )

        m = calc_metrics(
            yte,
            y_pred,
            score
        )

        base_f1 = baseline_if_f1[name]

        delta = (
            m['F1']
            - base_f1
        )

        pct = (
            100 * delta / base_f1
            if base_f1 > 0
            else np.nan
        )

        S2_IF_RESULTS.append({

            'dataset':
                name,

            'scenario':
                'S2_missing',

            'condition':
                f'missing={rate}',

            'model':
                'IsolationForest',

            'threshold':
                threshold,

            'baseline_F1_S0':
                base_f1,

            'delta_F1':
                delta,

            'pct_delta_F1':
                pct,

            'train_sec':
                train_sec,

            'inference_sec':
                infer_sec,

            **m
        })

        print(
            f"Missing {rate}: "
            f"F1={m['F1']:.6f} | "
            f"Recall={m['Recall']:.6f} | "
            f"FAR={m['FAR']:.6f}"
        )

    gc.collect()


# ============================================================
# SAVE
# ============================================================
s2_if_df = pd.DataFrame(
    S2_IF_RESULTS
)

display(
    s2_if_df
)

s2_if_df.to_csv(
    OUT / 'S2_isolationforest_results.csv',
    index=False
)

print(
    "\n✅ Saved:",
    OUT / 'S2_isolationforest_results.csv'
)


S2 ISOLATION FOREST — CICIDS2018
threshold=0.452999 | calibration=50,000
Noise 0.01: F1=0.239707 | Recall=0.999778 | FAR=0.999562
Noise 0.05: F1=0.239674 | Recall=1.000000 | FAR=1.000000
Missing 0.01: F1=0.000000 | Recall=0.000000 | FAR=0.060676
Missing 0.05: F1=0.000000 | Recall=0.000000 | FAR=0.065821

S2 ISOLATION FOREST — HDFS
threshold=0.530310 | calibration=50,000
Noise 0.01: F1=0.322675 | Recall=0.978100 | FAR=0.123605
Noise 0.05: F1=0.061130 | Recall=1.000000 | FAR=0.929595
Missing 0.01: F1=0.624947 | Recall=0.867416 | FAR=0.027496
Missing 0.05: F1=0.587224 | Recall=0.869192 | FAR=0.033022

S2 ISOLATION FOREST — BGL
threshold=0.587495 | calibration=39,989
Noise 0.01: F1=0.032555 | Recall=0.031250 | FAR=0.070512
Noise 0.05: F1=0.063007 | Recall=0.089674 | FAR=0.139407
Missing 0.01: F1=0.036078 | Recall=0.031250 | FAR=0.055633
Missing 0.05: F1=0.041602 | Recall=0.036685 | FAR=0.057682

S2 ISOLATION FOREST — UNSW-NB15
threshold=0.458734 | calibration=50,000
Noise 0.01: F1=0.09669

,dataset,scenario,condition,model,threshold,baseline_F1_S0,delta_F1,pct_delta_F1,train_sec,inference_sec,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S2_noise,sigma=0.01,IsolationForest,0.452999,0.000000,0.239707,NaN,1.769154,0.879514,0.136179,0.999778,0.239707,0.110369,0.999562,0.999592,25,57115,2,9004
1,CICIDS2018,S2_noise,sigma=0.05,IsolationForest,0.452999,0.000000,0.239674,NaN,1.769154,0.892631,0.136153,1.000000,0.239674,0.109122,1.000000,1.000000,0,57140,0,9006
2,CICIDS2018,S2_missing,missing=0.01,IsolationForest,0.452999,0.000000,0.000000,NaN,1.769154,0.812590,0.000000,0.000000,0.000000,0.115982,0.060676,0.052414,53673,3467,9006,0
3,CICIDS2018,S2_missing,missing=0.05,IsolationForest,0.452999,0.000000,0.000000,NaN,1.769154,0.807742,0.000000,0.000000,0.000000,0.120807,0.065821,0.056859,53379,3761,9006,0
4,HDFS,S2_noise,sigma=0.01,IsolationForest,0.530310,0.630229,-0.307553,-48.800300,2.821173,1.239977,0.193207,0.978100,0.322675,0.607419,0.123605,0.148705,97853,13801,74,3305
5,HDFS,S2_noise,sigma=0.05,IsolationForest,0.530310,0.630229,-0.569098,-90.300319,2.821173,1.397049,0.031529,1.000000,0.061130,0.404822,0.929595,0.931663,7861,103793,0,3379
6,HDFS,S2_missing,missing=0.01,IsolationForest,0.530310,0.630229,-0.005282,-0.838085,2.821173,1.072727,0.488419,0.867416,0.624947,0.612782,0.027496,0.052168,108584,3070,448,2931
7,HDFS,S2_missing,missing=0.05,IsolationForest,0.530310,0.630229,-0.043005,-6.823670,2.821173,1.107838,0.443388,0.869192,0.587224,0.591133,0.033022,0.057583,107967,3687,442,2937
8,BGL,S2_noise,sigma=0.01,IsolationForest,0.587495,0.036249,-0.003694,-10.191083,0.584434,0.190644,0.033973,0.031250,0.032555,0.091080,0.070512,0.067626,8621,654,713,23
9,BGL,S2_noise,sigma=0.05,IsolationForest,0.587495,0.036249,0.026758,73.817578,0.584434,0.193040,0.048565,0.089674,0.063007,0.072350,0.139407,0.135751,7982,1293,670,66



✅ Saved: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S2_isolationforest_results.csv


In [18]:
# ============================================================
# STEP 4B-2 — S2 LOF
# clean reference + clean train calibration
# fixed threshold -> perturbed held-out test
# ============================================================

from sklearn.neighbors import LocalOutlierFactor

S2_LOF_RESULTS = []

S2_NOISE_SIGMAS = [0.01, 0.05]
S2_MISSING_RATES = [0.01, 0.05]

LOF_REFERENCE_MAX = 50_000
LOF_CALIB_MAX = 20_000
LOF_K = 20


# ------------------------------------------------------------
# LOAD S0 LOF BASELINE
# ------------------------------------------------------------
s0_lof = pd.read_csv(
    OUT / 'S0_lof_results.csv'
)

baseline_lof_f1 = {
    r['dataset']: r['F1']
    for _, r in s0_lof.iterrows()
}


# ------------------------------------------------------------
# HELPER MISSING
# ------------------------------------------------------------
def inject_missing_raw_lof(
    Xraw,
    eligible_cols,
    rate,
    seed=42
):

    Xmiss = Xraw.copy()

    if len(eligible_cols) == 0:
        return Xmiss

    rng = np.random.RandomState(seed)

    n_rows = len(Xmiss)
    n_cols = len(eligible_cols)

    total_cells = n_rows * n_cols

    n_missing = int(
        round(
            total_cells * rate
        )
    )

    flat_idx = rng.choice(
        total_cells,
        size=n_missing,
        replace=False
    )

    row_idx, col_idx = np.unravel_index(
        flat_idx,
        (n_rows, n_cols)
    )

    for j, col in enumerate(
        eligible_cols
    ):

        rows_j = row_idx[
            col_idx == j
        ]

        if len(rows_j):

            Xmiss.iloc[
                rows_j,
                Xmiss.columns.get_loc(col)
            ] = np.nan

    return Xmiss


# ============================================================
# RUN
# ============================================================
for name, (X, y) in DATASETS.items():

    print(
        f"\n{'='*76}"
    )

    print(
        f"S2 LOF — {name}"
    )

    print(
        '='*76
    )

    tr, te = SPLITS[name]

    Xtr_raw = X.iloc[tr].copy()
    Xte_raw = X.iloc[te].copy()

    ytr = y[tr]
    yte = y[te]

    # --------------------------------------------------------
    # perturbation-eligible numeric telemetry
    # --------------------------------------------------------
    numeric_cols = list(
        Xtr_raw
        .select_dtypes(include=np.number)
        .columns
    )

    noise_cols = [
        c for c in numeric_cols
        if c not in ['node_h']
    ]

    noise_positions = [
        numeric_cols.index(c)
        for c in noise_cols
    ]

    # --------------------------------------------------------
    # PREPROCESS CLEAN TRAIN ONLY
    # --------------------------------------------------------
    pre_lof = make_preprocessor(
        Xtr_raw
    )

    Xtr = pre_lof.fit_transform(
        Xtr_raw
    )

    Xte_clean = pre_lof.transform(
        Xte_raw
    )

    # --------------------------------------------------------
    # REFERENCE + CALIBRATION
    # CLEAN TRAINING ONLY
    # --------------------------------------------------------
    rng = np.random.RandomState(
        CFG['SEED']
    )

    perm = rng.permutation(
        len(Xtr)
    )

    ref_n = min(
        LOF_REFERENCE_MAX,
        max(
            1000,
            int(0.70 * len(Xtr))
        )
    )

    ref_idx = perm[:ref_n]

    remaining = perm[ref_n:]

    cal_n = min(
        LOF_CALIB_MAX,
        len(remaining)
    )

    cal_idx = remaining[:cal_n]

    X_ref = Xtr[ref_idx]
    X_cal = Xtr[cal_idx]

    k = min(
        LOF_K,
        max(
            2,
            len(X_ref)-1
        )
    )

    # --------------------------------------------------------
    # FIT CLEAN REFERENCE
    # --------------------------------------------------------
    model = LocalOutlierFactor(
        n_neighbors=k,
        novelty=True,
        contamination='auto',
        n_jobs=-1
    )

    t0 = time.perf_counter()

    model.fit(
        X_ref
    )

    train_sec = (
        time.perf_counter()
        - t0
    )

    # --------------------------------------------------------
    # FIXED CLEAN-TRAIN CALIBRATION
    # --------------------------------------------------------
    cal_score = -model.score_samples(
        X_cal
    )

    threshold = float(
        np.quantile(
            cal_score,
            0.95
        )
    )

    print(
        f"reference={len(X_ref):,} | "
        f"calibration={len(X_cal):,} | "
        f"threshold={threshold:.6f}"
    )


    # ========================================================
    # A. GAUSSIAN NOISE
    # ========================================================
    for sigma in S2_NOISE_SIGMAS:

        rng_noise = np.random.RandomState(
            CFG['SEED']
            + int(
                sigma * 10000
            )
        )

        Xpert = Xte_clean.copy()

        noise = rng_noise.normal(
            0.0,
            sigma,
            size=(
                len(Xpert),
                len(noise_positions)
            )
        )

        Xpert[
            :,
            noise_positions
        ] += noise

        Xpert[
            :,
            noise_positions
        ] = np.clip(
            Xpert[
                :,
                noise_positions
            ],
            0.0,
            1.0
        )

        t1 = time.perf_counter()

        score = -model.score_samples(
            Xpert
        )

        y_pred = (
            score > threshold
        ).astype(int)

        infer_sec = (
            time.perf_counter()
            - t1
        )

        m = calc_metrics(
            yte,
            y_pred,
            score
        )

        base_f1 = baseline_lof_f1[
            name
        ]

        delta = (
            m['F1']
            - base_f1
        )

        pct = (
            100 * delta / base_f1
            if base_f1 > 0
            else np.nan
        )

        S2_LOF_RESULTS.append({

            'dataset':
                name,

            'scenario':
                'S2_noise',

            'condition':
                f'sigma={sigma}',

            'model':
                'LOF',

            'threshold':
                threshold,

            'n_reference':
                len(X_ref),

            'n_calibration':
                len(X_cal),

            'baseline_F1_S0':
                base_f1,

            'delta_F1':
                delta,

            'pct_delta_F1':
                pct,

            'train_sec':
                train_sec,

            'inference_sec':
                infer_sec,

            **m
        })

        print(
            f"Noise {sigma}: "
            f"F1={m['F1']:.6f} | "
            f"Recall={m['Recall']:.6f} | "
            f"FAR={m['FAR']:.6f} | "
            f"Alert={m['Alert_Rate']:.6f}"
        )


    # ========================================================
    # B. MISSING
    # ========================================================
    for rate in S2_MISSING_RATES:

        Xmiss_raw = inject_missing_raw_lof(

            Xte_raw,

            eligible_cols=noise_cols,

            rate=rate,

            seed=(
                CFG['SEED']
                + int(
                    rate * 10000
                )
            )
        )

        Xpert = pre_lof.transform(
            Xmiss_raw
        )

        t1 = time.perf_counter()

        score = -model.score_samples(
            Xpert
        )

        y_pred = (
            score > threshold
        ).astype(int)

        infer_sec = (
            time.perf_counter()
            - t1
        )

        m = calc_metrics(
            yte,
            y_pred,
            score
        )

        base_f1 = baseline_lof_f1[
            name
        ]

        delta = (
            m['F1']
            - base_f1
        )

        pct = (
            100 * delta / base_f1
            if base_f1 > 0
            else np.nan
        )

        S2_LOF_RESULTS.append({

            'dataset':
                name,

            'scenario':
                'S2_missing',

            'condition':
                f'missing={rate}',

            'model':
                'LOF',

            'threshold':
                threshold,

            'n_reference':
                len(X_ref),

            'n_calibration':
                len(X_cal),

            'baseline_F1_S0':
                base_f1,

            'delta_F1':
                delta,

            'pct_delta_F1':
                pct,

            'train_sec':
                train_sec,

            'inference_sec':
                infer_sec,

            **m
        })

        print(
            f"Missing {rate}: "
            f"F1={m['F1']:.6f} | "
            f"Recall={m['Recall']:.6f} | "
            f"FAR={m['FAR']:.6f} | "
            f"Alert={m['Alert_Rate']:.6f}"
        )


# ============================================================
# SAVE
# ============================================================
s2_lof_df = pd.DataFrame(
    S2_LOF_RESULTS
)

display(
    s2_lof_df
)

s2_lof_df.to_csv(
    OUT / 'S2_lof_results.csv',
    index=False
)

print(
    "\n✅ Saved:",
    OUT / 'S2_lof_results.csv'
)


S2 LOF — CICIDS2018
reference=50,000 | calibration=20,000 | threshold=3.922599
Noise 0.01: F1=0.274982 | Recall=0.995781 | FAR=0.826951 | Alert=0.849938
Noise 0.05: F1=0.249850 | Recall=0.995781 | FAR=0.941775 | Alert=0.949128
Missing 0.01: F1=0.189316 | Recall=0.222518 | FAR=0.177826 | Alert=0.183911
Missing 0.05: F1=0.289190 | Recall=0.688985 | FAR=0.484809 | Alert=0.512608

S2 LOF — HDFS
reference=50,000 | calibration=20,000 | threshold=1.698734
Noise 0.01: F1=0.040052 | Recall=0.625629 | FAR=0.896242 | Alert=0.888293
Noise 0.05: F1=0.052393 | Recall=0.915064 | FAR=0.999158 | Alert=0.996688
Missing 0.01: F1=0.132783 | Recall=0.342705 | FAR=0.115580 | Alert=0.122252
Missing 0.05: F1=0.083108 | Recall=0.531222 | FAR=0.340543 | Alert=0.346144

S2 LOF — BGL
reference=27,992 | calibration=11,997 | threshold=1.364833
Noise 0.01: F1=0.124808 | Recall=0.771739 | FAR=0.840755 | Alert=0.835681
Noise 0.05: F1=0.138108 | Recall=0.987772 | FAR=0.977358 | Alert=0.978124
Missing 0.01: F1=0.048990

,dataset,scenario,condition,model,threshold,n_reference,n_calibration,baseline_F1_S0,delta_F1,pct_delta_F1,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S2_noise,sigma=0.01,LOF,3.922599,50000,20000,0.003595,0.271388,7549.509537,...,0.159516,0.995781,0.274982,0.248482,0.826951,0.849938,9888,47252,38,8968
1,CICIDS2018,S2_noise,sigma=0.05,LOF,3.922599,50000,20000,0.003595,0.246255,6850.379722,...,0.142846,0.995781,0.249850,0.413531,0.941775,0.949128,3327,53813,38,8968
2,CICIDS2018,S2_missing,missing=0.01,LOF,3.922599,50000,20000,0.003595,0.185721,5166.415036,...,0.164735,0.222518,0.189316,0.146624,0.177826,0.183911,46979,10161,7002,2004
3,CICIDS2018,S2_missing,missing=0.05,LOF,3.922599,50000,20000,0.003595,0.285595,7944.733213,...,0.183001,0.688985,0.289190,0.175397,0.484809,0.512608,29438,27702,2801,6205
4,HDFS,S2_noise,sigma=0.01,LOF,1.698734,50000,20000,0.208211,-0.168159,-80.763647,...,0.020688,0.625629,0.040052,0.057717,0.896242,0.888293,11585,100069,1265,2114
5,HDFS,S2_noise,sigma=0.05,LOF,1.698734,50000,20000,0.208211,-0.155818,-74.836632,...,0.026969,0.915064,0.052393,0.050774,0.999158,0.996688,94,111560,287,3092
6,HDFS,S2_missing,missing=0.01,LOF,1.698734,50000,20000,0.208211,-0.075429,-36.226884,...,0.082344,0.342705,0.132783,0.101759,0.115580,0.122252,98749,12905,2221,1158
7,HDFS,S2_missing,missing=0.05,LOF,1.698734,50000,20000,0.208211,-0.125104,-60.084991,...,0.045080,0.531222,0.083108,0.077870,0.340543,0.346144,73631,38023,1584,1795
8,BGL,S2_noise,sigma=0.01,LOF,1.364833,27992,11997,0.039667,0.085140,214.636273,...,0.067894,0.771739,0.124808,0.052705,0.840755,0.835681,1477,7798,168,568
9,BGL,S2_noise,sigma=0.05,LOF,1.364833,27992,11997,0.039667,0.098441,248.165568,...,0.074244,0.987772,0.138108,0.057888,0.977358,0.978124,210,9065,9,727



✅ Saved: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S2_lof_results.csv


In [19]:
# ============================================================
# STEP 4B-3 — S2 DBSCAN INDUCTIVE
# clean train reference/core/threshold
# -> perturbed held-out test
# ============================================================

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

S2_DB_RESULTS = []

S2_NOISE_SIGMAS = [0.01, 0.05]
S2_MISSING_RATES = [0.01, 0.05]

DB_REFERENCE_MAX = 15_000
DB_CALIB_MAX = 10_000
DB_MIN_SAMPLES = 10
DB_EPS_QUANTILE = 0.95
SCORE_BATCH = 10_000


# ------------------------------------------------------------
# LOAD S0 DBSCAN BASELINE
# ------------------------------------------------------------
s0_db = pd.read_csv(
    OUT / 'S0_dbscan_inductive_results.csv'
)

baseline_db_f1 = {
    r['dataset']: r['F1']
    for _, r in s0_db.iterrows()
    if r['status'] == 'OK'
}


def nearest_core_score_s2(
    nn_core,
    X,
    batch_size=10000
):
    scores = []

    for start in range(
        0,
        len(X),
        batch_size
    ):

        end = min(
            start + batch_size,
            len(X)
        )

        d, _ = nn_core.kneighbors(
            X[start:end],
            n_neighbors=1
        )

        scores.append(
            d[:, 0]
        )

    return np.concatenate(
        scores
    )


def inject_missing_raw_db(
    Xraw,
    eligible_cols,
    rate,
    seed=42
):

    Xmiss = Xraw.copy()

    if len(eligible_cols) == 0:
        return Xmiss

    rng = np.random.RandomState(seed)

    n_rows = len(Xmiss)
    n_cols = len(eligible_cols)

    total_cells = (
        n_rows * n_cols
    )

    n_missing = int(
        round(
            total_cells * rate
        )
    )

    flat_idx = rng.choice(
        total_cells,
        size=n_missing,
        replace=False
    )

    row_idx, col_idx = np.unravel_index(
        flat_idx,
        (n_rows, n_cols)
    )

    for j, col in enumerate(
        eligible_cols
    ):

        rows_j = row_idx[
            col_idx == j
        ]

        if len(rows_j):

            Xmiss.iloc[
                rows_j,
                Xmiss.columns.get_loc(col)
            ] = np.nan

    return Xmiss


# ============================================================
# RUN
# ============================================================
for name, (X, y) in DATASETS.items():

    print(
        f"\n{'='*76}"
    )

    print(
        f"S2 DBSCAN — {name}"
    )

    print(
        '='*76
    )

    tr, te = SPLITS[name]

    Xtr_raw = X.iloc[tr].copy()
    Xte_raw = X.iloc[te].copy()

    ytr = y[tr]
    yte = y[te]

    # --------------------------------------------------------
    # numeric telemetry
    # --------------------------------------------------------
    numeric_cols = list(
        Xtr_raw
        .select_dtypes(
            include=np.number
        )
        .columns
    )

    noise_cols = [
        c
        for c in numeric_cols
        if c not in ['node_h']
    ]

    noise_positions = [
        numeric_cols.index(c)
        for c in noise_cols
    ]

    # --------------------------------------------------------
    # CLEAN TRAIN PREPROCESSING
    # --------------------------------------------------------
    pre_db = make_preprocessor(
        Xtr_raw
    )

    Xtr = pre_db.fit_transform(
        Xtr_raw
    )

    Xte_clean = pre_db.transform(
        Xte_raw
    )

    # --------------------------------------------------------
    # SAME TRAIN REFERENCE/CALIBRATION DESIGN AS S0
    # --------------------------------------------------------
    rng = np.random.RandomState(
        CFG['SEED']
    )

    perm = rng.permutation(
        len(Xtr)
    )

    ref_n = min(
        DB_REFERENCE_MAX,
        max(
            2000,
            int(
                0.60 * len(Xtr)
            )
        )
    )

    ref_idx = perm[:ref_n]

    remaining = perm[ref_n:]

    cal_n = min(
        DB_CALIB_MAX,
        len(remaining)
    )

    cal_idx = remaining[:cal_n]

    X_ref = Xtr[
        ref_idx
    ]

    X_cal = Xtr[
        cal_idx
    ]

    # --------------------------------------------------------
    # EPS FROM CLEAN TRAIN REFERENCE
    # --------------------------------------------------------
    k = min(
        DB_MIN_SAMPLES,
        len(X_ref)
    )

    nn_eps = NearestNeighbors(
        n_neighbors=k,
        metric='euclidean',
        n_jobs=-1
    )

    nn_eps.fit(
        X_ref
    )

    dist_ref, _ = (
        nn_eps.kneighbors(
            X_ref
        )
    )

    eps = float(
        np.quantile(
            dist_ref[:, -1],
            DB_EPS_QUANTILE
        )
    )

    # --------------------------------------------------------
    # FIT DBSCAN ON CLEAN TRAIN REFERENCE
    # --------------------------------------------------------
    t0 = time.perf_counter()

    db = DBSCAN(
        eps=eps,
        min_samples=DB_MIN_SAMPLES,
        metric='euclidean',
        n_jobs=-1
    )

    db.fit(
        X_ref
    )

    train_sec = (
        time.perf_counter()
        - t0
    )

    n_core = len(
        db.core_sample_indices_
    )

    n_clusters = len(
        set(db.labels_) - {-1}
    )

    n_noise_ref = int(
        np.sum(
            db.labels_ == -1
        )
    )

    if n_core == 0:

        print(
            "⚠️ Tidak ada DBSCAN core point."
        )

        continue

    core_points = X_ref[
        db.core_sample_indices_
    ]

    nn_core = NearestNeighbors(
        n_neighbors=1,
        metric='euclidean',
        n_jobs=-1
    )

    nn_core.fit(
        core_points
    )

    # --------------------------------------------------------
    # FIXED THRESHOLD FROM CLEAN TRAIN CALIBRATION
    # --------------------------------------------------------
    cal_score = nearest_core_score_s2(
        nn_core,
        X_cal,
        SCORE_BATCH
    )

    threshold = float(
        np.quantile(
            cal_score,
            0.95
        )
    )

    print(
        f"eps={eps:.6f} | "
        f"threshold={threshold:.6f} | "
        f"clusters={n_clusters} | "
        f"core={n_core:,}"
    )


    # ========================================================
    # A. NOISE
    # ========================================================
    for sigma in S2_NOISE_SIGMAS:

        rng_noise = np.random.RandomState(
            CFG['SEED']
            + int(
                sigma * 10000
            )
        )

        Xpert = Xte_clean.copy()

        noise = rng_noise.normal(
            0.0,
            sigma,
            size=(
                len(Xpert),
                len(noise_positions)
            )
        )

        Xpert[
            :,
            noise_positions
        ] += noise

        Xpert[
            :,
            noise_positions
        ] = np.clip(
            Xpert[
                :,
                noise_positions
            ],
            0.0,
            1.0
        )

        t1 = time.perf_counter()

        score = nearest_core_score_s2(
            nn_core,
            Xpert,
            SCORE_BATCH
        )

        y_pred = (
            score > threshold
        ).astype(int)

        infer_sec = (
            time.perf_counter()
            - t1
        )

        m = calc_metrics(
            yte,
            y_pred,
            score
        )

        base_f1 = baseline_db_f1[
            name
        ]

        delta = (
            m['F1']
            - base_f1
        )

        pct = (
            100 * delta / base_f1
            if base_f1 > 0
            else np.nan
        )

        S2_DB_RESULTS.append({

            'dataset':
                name,

            'scenario':
                'S2_noise',

            'condition':
                f'sigma={sigma}',

            'model':
                'DBSCAN_inductive',

            'eps':
                eps,

            'threshold':
                threshold,

            'n_reference':
                len(X_ref),

            'n_calibration':
                len(X_cal),

            'n_core':
                n_core,

            'n_clusters':
                n_clusters,

            'reference_noise':
                n_noise_ref,

            'baseline_F1_S0':
                base_f1,

            'delta_F1':
                delta,

            'pct_delta_F1':
                pct,

            'train_sec':
                train_sec,

            'inference_sec':
                infer_sec,

            **m
        })

        print(
            f"Noise {sigma}: "
            f"F1={m['F1']:.6f} | "
            f"Recall={m['Recall']:.6f} | "
            f"FAR={m['FAR']:.6f} | "
            f"Alert={m['Alert_Rate']:.6f}"
        )


    # ========================================================
    # B. MISSING
    # ========================================================
    for rate in S2_MISSING_RATES:

        Xmiss_raw = inject_missing_raw_db(

            Xte_raw,

            eligible_cols=noise_cols,

            rate=rate,

            seed=(
                CFG['SEED']
                + int(
                    rate * 10000
                )
            )
        )

        Xpert = pre_db.transform(
            Xmiss_raw
        )

        t1 = time.perf_counter()

        score = nearest_core_score_s2(
            nn_core,
            Xpert,
            SCORE_BATCH
        )

        y_pred = (
            score > threshold
        ).astype(int)

        infer_sec = (
            time.perf_counter()
            - t1
        )

        m = calc_metrics(
            yte,
            y_pred,
            score
        )

        base_f1 = baseline_db_f1[
            name
        ]

        delta = (
            m['F1']
            - base_f1
        )

        pct = (
            100 * delta / base_f1
            if base_f1 > 0
            else np.nan
        )

        S2_DB_RESULTS.append({

            'dataset':
                name,

            'scenario':
                'S2_missing',

            'condition':
                f'missing={rate}',

            'model':
                'DBSCAN_inductive',

            'eps':
                eps,

            'threshold':
                threshold,

            'n_reference':
                len(X_ref),

            'n_calibration':
                len(X_cal),

            'n_core':
                n_core,

            'n_clusters':
                n_clusters,

            'reference_noise':
                n_noise_ref,

            'baseline_F1_S0':
                base_f1,

            'delta_F1':
                delta,

            'pct_delta_F1':
                pct,

            'train_sec':
                train_sec,

            'inference_sec':
                infer_sec,

            **m
        })

        print(
            f"Missing {rate}: "
            f"F1={m['F1']:.6f} | "
            f"Recall={m['Recall']:.6f} | "
            f"FAR={m['FAR']:.6f} | "
            f"Alert={m['Alert_Rate']:.6f}"
        )


# ============================================================
# SAVE
# ============================================================
s2_db_df = pd.DataFrame(
    S2_DB_RESULTS
)

display(
    s2_db_df
)

s2_db_df.to_csv(
    OUT / 'S2_dbscan_inductive_results.csv',
    index=False
)

print(
    "\n✅ Saved:",
    OUT / 'S2_dbscan_inductive_results.csv'
)


S2 DBSCAN — CICIDS2018
eps=0.128265 | threshold=0.097505 | clusters=43 | core=14,250
Noise 0.01: F1=0.006495 | Recall=0.004553 | FAR=0.062636 | Alert=0.054727
Noise 0.05: F1=0.239674 | Recall=1.000000 | FAR=1.000000 | Alert=1.000000
Missing 0.01: F1=0.053148 | Recall=0.043304 | FAR=0.092405 | Alert=0.085719
Missing 0.05: F1=0.136815 | Recall=0.178881 | FAR=0.226339 | Alert=0.219877

S2 DBSCAN — HDFS
eps=0.057760 | threshold=0.033313 | clusters=42 | core=14,250
Noise 0.01: F1=0.060773 | Recall=0.846996 | FAR=0.787665 | Alert=0.789408
Noise 0.05: F1=0.057072 | Recall=1.000000 | FAR=1.000000 | Alert=1.000000
Missing 0.01: F1=0.220723 | Recall=0.546907 | FAR=0.103158 | Alert=0.116193
Missing 0.05: F1=0.121675 | Recall=0.718852 | FAR=0.305569 | Alert=0.317709

S2 DBSCAN — BGL
eps=0.038592 | threshold=0.016258 | clusters=12 | core=14,250
Noise 0.01: F1=0.133994 | Recall=0.735054 | FAR=0.732938 | Alert=0.733094
Noise 0.05: F1=0.136810 | Recall=0.994565 | FAR=0.995472 | Alert=0.995405
Missing

,dataset,scenario,condition,model,eps,threshold,n_reference,n_calibration,n_core,n_clusters,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S2_noise,sigma=0.01,DBSCAN_inductive,0.128265,0.097505,15000,10000,14250,43,...,0.011326,0.004553,0.006495,0.131776,0.062636,0.054727,53561,3579,8965,41
1,CICIDS2018,S2_noise,sigma=0.05,DBSCAN_inductive,0.128265,0.097505,15000,10000,14250,43,...,0.136153,1.000000,0.239674,0.140639,1.000000,1.000000,0,57140,0,9006
2,CICIDS2018,S2_missing,missing=0.01,DBSCAN_inductive,0.128265,0.097505,15000,10000,14250,43,...,0.068783,0.043304,0.053148,0.106098,0.092405,0.085719,51860,5280,8616,390
3,CICIDS2018,S2_missing,missing=0.05,DBSCAN_inductive,0.128265,0.097505,15000,10000,14250,43,...,0.110767,0.178881,0.136815,0.113228,0.226339,0.219877,44207,12933,7395,1611
4,HDFS,S2_noise,sigma=0.01,DBSCAN_inductive,0.057760,0.033313,15000,10000,14250,42,...,0.031517,0.846996,0.060773,0.371839,0.787665,0.789408,23708,87946,517,2862
5,HDFS,S2_noise,sigma=0.05,DBSCAN_inductive,0.057760,0.033313,15000,10000,14250,42,...,0.029374,1.000000,0.057072,0.293003,1.000000,1.000000,0,111654,0,3379
6,HDFS,S2_missing,missing=0.01,DBSCAN_inductive,0.057760,0.033313,15000,10000,14250,42,...,0.138261,0.546907,0.220723,0.216161,0.103158,0.116193,100136,11518,1531,1848
7,HDFS,S2_missing,missing=0.05,DBSCAN_inductive,0.057760,0.033313,15000,10000,14250,42,...,0.066462,0.718852,0.121675,0.118821,0.305569,0.317709,77536,34118,950,2429
8,BGL,S2_noise,sigma=0.01,DBSCAN_inductive,0.038592,0.016258,15000,10000,14250,12,...,0.073716,0.735054,0.133994,0.070286,0.732938,0.733094,2477,6798,195,541
9,BGL,S2_noise,sigma=0.05,DBSCAN_inductive,0.038592,0.016258,15000,10000,14250,12,...,0.073457,0.994565,0.136810,0.071691,0.995472,0.995405,42,9233,4,732



✅ Saved: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S2_dbscan_inductive_results.csv


In [20]:
# ============================================================
# STEP 5A — S3 TEMPORAL / GENERALIZATION AVAILABILITY AUDIT
# BELUM TRAIN MODEL
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


def audit_order_split(name, X, y, order=None, train_frac=0.70):

    y = np.asarray(y, dtype=int)

    if order is None:
        order = np.arange(len(y))
        order_type = 'current_row/log_order_proxy'
    else:
        order = np.asarray(order)
        order_type = 'explicit_temporal_order'

    idx = np.argsort(
        order,
        kind='stable'
    )

    cut = int(
        len(idx) * train_frac
    )

    tr = idx[:cut]
    te = idx[cut:]

    # feature-vector overlap audit
    train_hash = set(
        pd.util.hash_pandas_object(
            X.iloc[tr],
            index=False
        ).astype('uint64')
    )

    test_hash = (
        pd.util.hash_pandas_object(
            X.iloc[te],
            index=False
        ).astype('uint64')
    )

    shared_test_rows = int(
        np.sum([
            h in train_hash
            for h in test_hash
        ])
    )

    return {
        'dataset': name,

        'order_type':
            order_type,

        'n_total':
            len(y),

        'n_train':
            len(tr),

        'n_test':
            len(te),

        'train_positive':
            int(y[tr].sum()),

        'test_positive':
            int(y[te].sum()),

        'train_positive_pct':
            float(100*y[tr].mean()),

        'test_positive_pct':
            float(100*y[te].mean()),

        'shared_exact_feature_test_rows':
            shared_test_rows,

        'both_classes_train':
            bool(len(np.unique(y[tr])) == 2),

        'both_classes_test':
            bool(len(np.unique(y[te])) == 2)
    }


AUDIT_S3 = []


# ============================================================
# A. CICIDS2018 — CHECK CROSS-DAY FILE
# ============================================================

print("\n=== CICIDS2018 CROSS-DAY FILE CHECK ===")

base_path = Path(BASE)

candidates_15 = list(
    base_path.glob('*02-15*')
)

print(
    "02-14:",
    CFG['CICIDS']
)

print(
    "02-15 candidates:",
    [str(p) for p in candidates_15]
)

if len(candidates_15) == 0:

    print(
        "⚠️ File 02-15 belum ditemukan. "
        "Jangan jalankan S3 CICIDS sebelum file ini tersedia."
    )

else:

    cic15_path = str(
        candidates_15[0]
    )

    # label audit only
    header15 = pd.read_csv(
        cic15_path,
        nrows=0
    )

    label15 = next(
        c for c in header15.columns
        if 'label' in str(c).lower()
    )

    y15raw = pd.read_csv(
        cic15_path,
        usecols=[label15],
        low_memory=False
    )[label15]

    y15 = (
        y15raw
        .astype(str)
        .str.strip()
        .str.upper()
        != 'BENIGN'
    ).astype(int)

    print(
        f"02-15 rows={len(y15):,} | "
        f"attack={y15.sum():,} "
        f"({100*y15.mean():.3f}%)"
    )


# ============================================================
# B. HDFS
#
# HDFS richer cache row order mengikuti insertion order
# block saat pertama kali muncul pada log.
# Ini HANYA log-order proxy, bukan timestamp absolute.
# ============================================================

X_hdfs, y_hdfs = DATASETS['HDFS']

AUDIT_S3.append(
    audit_order_split(
        'HDFS',
        X_hdfs,
        y_hdfs
    )
)


# ============================================================
# C. BGL
#
# Loader BGL sebelumnya telah disortir kembali berdasarkan
# original log sequence sebelum __seq__ dibuang.
# ============================================================

X_bgl, y_bgl = DATASETS['BGL']

AUDIT_S3.append(
    audit_order_split(
        'BGL',
        X_bgl,
        y_bgl
    )
)


# ============================================================
# D. UNSW-NB15
# gunakan STIME eksplisit
# ============================================================

unsw_time = pd.read_csv(
    CFG['UNSW'],
    header=None,
    usecols=[28],      # STIME
    low_memory=False
).iloc[:,0]

unsw_time = pd.to_numeric(
    unsw_time,
    errors='coerce'
)

# fallback hanya bila ada invalid timestamp
unsw_time = unsw_time.fillna(
    pd.Series(
        np.arange(len(unsw_time)),
        index=unsw_time.index
    )
)

X_unsw, y_unsw = DATASETS[
    'UNSW-NB15'
]

AUDIT_S3.append(
    audit_order_split(
        'UNSW-NB15',
        X_unsw,
        y_unsw,
        order=unsw_time.values
    )
)


# ============================================================
# RESULT
# ============================================================

s3_audit_df = pd.DataFrame(
    AUDIT_S3
)

display(
    s3_audit_df
)

print(
    "\n=== STOP ==="
)

print(
    "Belum melakukan training S3."
)


=== CICIDS2018 CROSS-DAY FILE CHECK ===
02-14: /content/drive/MyDrive/ercyris_siklus2/02-14-2018.csv
02-15 candidates: []
⚠️ File 02-15 belum ditemukan. Jangan jalankan S3 CICIDS sebelum file ini tersedia.


,dataset,order_type,n_total,n_train,n_test,train_positive,test_positive,train_positive_pct,test_positive_pct,shared_exact_feature_test_rows,both_classes_train,both_classes_test
0,HDFS,current_row/log_order_proxy,575061,402542,172519,13115,3723,3.258045,2.158023,1876,True,True
1,BGL,current_row/log_order_proxy,50000,35000,15000,2788,882,7.965714,5.880000,2669,True,True
2,UNSW-NB15,explicit_temporal_order,700001,490000,210001,22215,0,4.533673,0.000000,536,True,False



=== STOP ===
Belum melakukan training S3.


In [21]:
# ============================================================
# STEP 5B — VALID GENERALIZATION SPLIT AUDIT
# BELUM TRAINING MODEL
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# A. PURGED TEMPORAL/ORDER SPLIT
#    HDFS + BGL
# ============================================================

def purged_order_split_audit(
    name,
    X,
    y,
    train_frac=0.70
):

    y = np.asarray(
        y,
        dtype=int
    )

    n = len(X)

    cut = int(
        n * train_frac
    )

    tr = np.arange(
        0,
        cut
    )

    te_initial = np.arange(
        cut,
        n
    )

    # --------------------------------------------------------
    # hash feature vectors
    # --------------------------------------------------------
    train_hash = set(
        pd.util.hash_pandas_object(
            X.iloc[tr],
            index=False
        )
        .astype('uint64')
        .values
    )

    test_hash = (
        pd.util.hash_pandas_object(
            X.iloc[te_initial],
            index=False
        )
        .astype('uint64')
        .values
    )

    # --------------------------------------------------------
    # remove test rows whose exact feature vector
    # already appeared in training
    # --------------------------------------------------------
    novel_mask = np.array([
        h not in train_hash
        for h in test_hash
    ])

    te = te_initial[
        novel_mask
    ]

    removed = len(
        te_initial
    ) - len(te)

    result = {

        'dataset':
            name,

        'n_total':
            n,

        'n_train':
            len(tr),

        'n_test_initial':
            len(te_initial),

        'n_test_after_purge':
            len(te),

        'purged_test_rows':
            removed,

        'purged_pct_of_test':
            100 * removed / len(te_initial),

        'train_positive':
            int(
                y[tr].sum()
            ),

        'test_positive_after_purge':
            int(
                y[te].sum()
            ),

        'train_positive_pct':
            100 * y[tr].mean(),

        'test_positive_pct_after_purge':
            100 * y[te].mean()
            if len(te)
            else np.nan,

        'both_classes_train':
            len(
                np.unique(
                    y[tr]
                )
            ) == 2,

        'both_classes_test_after_purge':
            len(
                np.unique(
                    y[te]
                )
            ) == 2
    }

    return (
        tr,
        te,
        result
    )


PURGED_SPLITS = {}
purge_rows = []


# ------------------------------------------------------------
# HDFS
# ------------------------------------------------------------
X_hdfs, y_hdfs = DATASETS[
    'HDFS'
]

tr_hdfs, te_hdfs, audit_hdfs = (
    purged_order_split_audit(
        'HDFS',
        X_hdfs,
        y_hdfs
    )
)

PURGED_SPLITS[
    'HDFS'
] = (
    tr_hdfs,
    te_hdfs
)

purge_rows.append(
    audit_hdfs
)


# ------------------------------------------------------------
# BGL
# ------------------------------------------------------------
X_bgl, y_bgl = DATASETS[
    'BGL'
]

tr_bgl, te_bgl, audit_bgl = (
    purged_order_split_audit(
        'BGL',
        X_bgl,
        y_bgl
    )
)

PURGED_SPLITS[
    'BGL'
] = (
    tr_bgl,
    te_bgl
)

purge_rows.append(
    audit_bgl
)


purge_df = pd.DataFrame(
    purge_rows
)

print(
    "\n=== HDFS / BGL PURGED SHIFT AUDIT ==="
)

display(
    purge_df
)


# ============================================================
# B. UNSW ATTACK-FAMILY AUDIT
# ============================================================

UNSW_META_COLS = [
    'stime',
    'attack_cat',
    'label'
]

# File UNSW tanpa header:
# stime      = column 28
# attack_cat = column 47
# label      = column 48

unsw_meta = pd.read_csv(
    CFG['UNSW'],
    header=None,
    usecols=[
        28,
        47,
        48
    ],
    low_memory=False
)

unsw_meta.columns = (
    UNSW_META_COLS
)


unsw_meta[
    'label'
] = (
    pd.to_numeric(
        unsw_meta['label'],
        errors='coerce'
    )
    .fillna(0)
    .astype(int)
)


unsw_meta[
    'stime'
] = pd.to_numeric(
    unsw_meta['stime'],
    errors='coerce'
)


unsw_meta[
    'attack_cat'
] = (
    unsw_meta[
        'attack_cat'
    ]
    .fillna('NORMAL')
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# Full family distribution
# ------------------------------------------------------------
family_summary = (
    unsw_meta
    .groupby(
        'attack_cat'
    )
    .agg(
        rows=(
            'label',
            'size'
        ),
        positive=(
            'label',
            'sum'
        ),
        first_time=(
            'stime',
            'min'
        ),
        last_time=(
            'stime',
            'max'
        )
    )
    .reset_index()
)


family_summary[
    'positive_pct'
] = (
    100
    * family_summary[
        'positive'
    ]
    / family_summary[
        'rows'
    ]
)


family_summary = (
    family_summary
    .sort_values(
        'positive',
        ascending=False
    )
)


print(
    "\n=== UNSW ATTACK FAMILY DISTRIBUTION ==="
)

display(
    family_summary
)


# ------------------------------------------------------------
# only real attack categories
# ------------------------------------------------------------
attack_families = (
    family_summary[
        family_summary[
            'positive'
        ] > 0
    ]
    .copy()
)


print(
    "\n=== CANDIDATE UNSW HOLDOUT FAMILIES ==="
)

display(
    attack_families[
        [
            'attack_cat',
            'rows',
            'positive',
            'positive_pct',
            'first_time',
            'last_time'
        ]
    ]
)


print(
    "\n=== STOP ==="
)

print(
    "Belum ada training S3."
)


=== HDFS / BGL PURGED SHIFT AUDIT ===


,dataset,n_total,n_train,n_test_initial,n_test_after_purge,purged_test_rows,purged_pct_of_test,train_positive,test_positive_after_purge,train_positive_pct,test_positive_pct_after_purge,both_classes_train,both_classes_test_after_purge
0,HDFS,575061,402542,172519,170643,1876,1.087416,13115,2336,3.258045,1.368940,True,True
1,BGL,50000,35000,15000,12331,2669,17.793333,2788,753,7.965714,6.106561,True,True



=== UNSW ATTACK FAMILY DISTRIBUTION ===


,attack_cat,rows,positive,first_time,last_time,positive_pct
5,Generic,7522,7522,1421927420,1421934366,100.0
3,Exploits,5409,5409,1421927415,1421934373,100.0
4,Fuzzers,5051,5051,1421927596,1421934448,100.0
7,Reconnaissance,1759,1759,1421927414,1421934369,100.0
2,DoS,1167,1167,1421927419,1421934362,100.0
1,Backdoors,534,534,1421928094,1421934317,100.0
0,Analysis,526,526,1421930602,1421933541,100.0
8,Shellcode,223,223,1421927465,1421934357,100.0
9,Worms,24,24,1421927695,1421934351,100.0
6,NORMAL,677786,0,1421927377,1421955842,0.0



=== CANDIDATE UNSW HOLDOUT FAMILIES ===


,attack_cat,rows,positive,positive_pct,first_time,last_time
5,Generic,7522,7522,100.0,1421927420,1421934366
3,Exploits,5409,5409,100.0,1421927415,1421934373
4,Fuzzers,5051,5051,100.0,1421927596,1421934448
7,Reconnaissance,1759,1759,100.0,1421927414,1421934369
2,DoS,1167,1167,100.0,1421927419,1421934362
1,Backdoors,534,534,100.0,1421928094,1421934317
0,Analysis,526,526,100.0,1421930602,1421933541
8,Shellcode,223,223,100.0,1421927465,1421934357
9,Worms,24,24,100.0,1421927695,1421934351



=== STOP ===
Belum ada training S3.


In [22]:
# ============================================================
# STEP 5C-1 — S3 SHIFT GENERALIZATION
# HDFS + BGL ONLY
# XGBoost + Random Forest
# ============================================================

import pandas as pd
import numpy as np
import time

S3_SHIFT_RESULTS = []

# baseline S0 untuk menghitung degradasi
s0_sup = pd.read_csv(
    OUT / 'S0_supervised_results.csv'
)

baseline_f1 = {
    (r['dataset'], r['model']): r['F1']
    for _, r in s0_sup.iterrows()
}


for name in ['HDFS', 'BGL']:

    print(f"\n{'='*76}")
    print(f"S3 SHIFT — {name}")
    print('='*76)

    X, y = DATASETS[name]

    # hasil STEP 5B:
    # 70% awal = train
    # 30% akhir = test
    # exact feature overlap sudah dipurge
    tr, te = PURGED_SPLITS[name]

    Xtr_raw = X.iloc[tr].copy()
    Xte_raw = X.iloc[te].copy()

    ytr = y[tr]
    yte = y[te]

    print(
        f"Train={len(tr):,} | "
        f"Test={len(te):,} | "
        f"Train positive={ytr.sum():,} "
        f"({100*ytr.mean():.3f}%) | "
        f"Test positive={yte.sum():,} "
        f"({100*yte.mean():.3f}%)"
    )

    # --------------------------------------------------------
    # preprocessing fit TRAIN ONLY
    # --------------------------------------------------------
    pre_s3 = make_preprocessor(
        Xtr_raw
    )

    t0 = time.perf_counter()

    Xtr = pre_s3.fit_transform(
        Xtr_raw
    )

    Xte = pre_s3.transform(
        Xte_raw
    )

    prep_sec = (
        time.perf_counter() - t0
    )


    models = {

        'XGBoost':
            XGBClassifier(
                random_state=CFG['SEED'],
                n_jobs=-1,
                eval_metric='logloss',
                tree_method='hist',
                verbosity=0
            ),

        'RandomForest':
            RandomForestClassifier(
                n_estimators=100,
                random_state=CFG['SEED'],
                n_jobs=-1
            )
    }


    for model_name, model in models.items():

        print(f"\n▶ {model_name}")

        # ----------------------------------------------------
        # train
        # ----------------------------------------------------
        t1 = time.perf_counter()

        model.fit(
            Xtr,
            ytr
        )

        train_sec = (
            time.perf_counter() - t1
        )

        # ----------------------------------------------------
        # held-out shifted test
        # ----------------------------------------------------
        t2 = time.perf_counter()

        y_pred = model.predict(
            Xte
        )

        y_score = model.predict_proba(
            Xte
        )[:,1]

        infer_sec = (
            time.perf_counter() - t2
        )

        m = calc_metrics(
            yte,
            y_pred,
            y_score
        )

        f1_s0 = baseline_f1[
            (name, model_name)
        ]

        delta = (
            m['F1'] - f1_s0
        )

        pct = (
            100 * delta / f1_s0
            if f1_s0 > 0
            else np.nan
        )

        S3_SHIFT_RESULTS.append({

            'dataset':
                name,

            'scenario':
                'S3_shift',

            'shift_type':
                (
                    'log_order_proxy_70_30_purged'
                    if name == 'HDFS'
                    else
                    'log_order_70_30_purged'
                ),

            'model':
                model_name,

            'n_train':
                len(tr),

            'n_test':
                len(te),

            'train_positive_rate':
                float(ytr.mean()),

            'test_positive_rate':
                float(yte.mean()),

            'baseline_F1_S0':
                f1_s0,

            'delta_F1':
                delta,

            'pct_delta_F1':
                pct,

            'preprocessing_sec':
                prep_sec,

            'train_sec':
                train_sec,

            'inference_sec':
                infer_sec,

            **m
        })

        print(
            f"F1={m['F1']:.6f} | "
            f"ΔF1={delta:+.6f} "
            f"({pct:+.2f}%) | "
            f"Precision={m['Precision']:.6f} | "
            f"Recall={m['Recall']:.6f} | "
            f"PR-AUC={m['PR_AUC']:.6f} | "
            f"FAR={m['FAR']:.6f}"
        )


# ============================================================
# SAVE
# ============================================================
s3_shift_df = pd.DataFrame(
    S3_SHIFT_RESULTS
)

display(
    s3_shift_df
)

s3_shift_df.to_csv(
    OUT / 'S3_HDFS_BGL_shift_supervised_results.csv',
    index=False
)

print(
    "\n✅ Saved:",
    OUT / 'S3_HDFS_BGL_shift_supervised_results.csv'
)


S3 SHIFT — HDFS
Train=402,542 | Test=170,643 | Train positive=13,115 (3.258%) | Test positive=2,336 (1.369%)

▶ XGBoost
F1=0.996568 | ΔF1=-0.000471 (-0.05%) | Precision=0.998710 | Recall=0.994435 | PR-AUC=0.999850 | FAR=0.000018

▶ RandomForest
F1=0.998072 | ΔF1=+0.000736 (+0.07%) | Precision=0.998714 | Recall=0.997432 | PR-AUC=0.999887 | FAR=0.000018

S3 SHIFT — BGL
Train=35,000 | Test=12,331 | Train positive=2,788 (7.966%) | Test positive=753 (6.107%)

▶ XGBoost
F1=0.712980 | ΔF1=-0.214017 (-23.09%) | Precision=0.658784 | Recall=0.776892 | PR-AUC=0.557047 | FAR=0.026170

▶ RandomForest
F1=0.702899 | ΔF1=-0.193134 (-21.55%) | Precision=0.644518 | Recall=0.772908 | PR-AUC=0.567537 | FAR=0.027725


,dataset,scenario,shift_type,model,n_train,n_test,train_positive_rate,test_positive_rate,baseline_F1_S0,delta_F1,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,HDFS,S3_shift,log_order_proxy_70_30_purged,XGBoost,402542,170643,0.032580,0.013689,0.997039,-0.000471,...,0.998710,0.994435,0.996568,0.999850,0.000018,0.013631,168304,3,13,2323
1,HDFS,S3_shift,log_order_proxy_70_30_purged,RandomForest,402542,170643,0.032580,0.013689,0.997336,0.000736,...,0.998714,0.997432,0.998072,0.999887,0.000018,0.013672,168304,3,6,2330
2,BGL,S3_shift,log_order_70_30_purged,XGBoost,35000,12331,0.079657,0.061066,0.926997,-0.214017,...,0.658784,0.776892,0.712980,0.557047,0.026170,0.072014,11275,303,168,585
3,BGL,S3_shift,log_order_70_30_purged,RandomForest,35000,12331,0.079657,0.061066,0.896033,-0.193134,...,0.644518,0.772908,0.702899,0.567537,0.027725,0.073230,11257,321,171,582



✅ Saved: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S3_HDFS_BGL_shift_supervised_results.csv


In [23]:
# ============================================================
# STEP 5C-2 — UNSW-NB15
# LEAVE-ONE-ATTACK-FAMILY-OUT GENERALIZATION
#
# XGBoost + Random Forest
# exact-feature-group leakage prohibited
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

S3_UNSW_RESULTS = []

HOLDOUT_FAMILIES = [
    'Generic',
    'Exploits',
    'Fuzzers',
    'Reconnaissance',
    'DoS'
]


# ============================================================
# 1. LOAD ATTACK FAMILY METADATA
# ============================================================

unsw_meta = pd.read_csv(
    CFG['UNSW'],
    header=None,
    usecols=[47, 48],
    low_memory=False
)

unsw_meta.columns = [
    'attack_cat',
    'label'
]

unsw_meta['label'] = (
    pd.to_numeric(
        unsw_meta['label'],
        errors='coerce'
    )
    .fillna(0)
    .astype(int)
)

unsw_meta['attack_cat'] = (
    unsw_meta['attack_cat']
    .fillna('NORMAL')
    .astype(str)
    .str.strip()
)


X_unsw, y_unsw = DATASETS[
    'UNSW-NB15'
]

assert len(X_unsw) == len(unsw_meta)
assert np.array_equal(
    y_unsw,
    unsw_meta['label'].values
)

print(
    "UNSW rows:",
    len(X_unsw)
)


# ============================================================
# 2. EXACT FEATURE GROUP IDS
# ============================================================

feature_hash = (
    pd.util.hash_pandas_object(
        X_unsw,
        index=False
    )
    .astype('uint64')
    .values
)


# ============================================================
# 3. FIXED NORMAL TRAIN / TEST PARTITION
#
# Normal test set dibuat sekali dan dipakai sama untuk
# seluruh holdout family.
#
# Group-aware -> feature-identical normal samples tidak
# tersebar antara train dan test.
# ============================================================

normal_idx = np.where(
    y_unsw == 0
)[0]

normal_groups = feature_hash[
    normal_idx
]


# cari group split yang dekat dengan 80/20 row ratio
best_normal_split = None

for rs in range(
    CFG['SEED'],
    CFG['SEED'] + 100
):

    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=rs
    )

    tr_rel, te_rel = next(
        gss.split(
            X_unsw.iloc[normal_idx],
            groups=normal_groups
        )
    )

    train_n = normal_idx[
        tr_rel
    ]

    test_n = normal_idx[
        te_rel
    ]

    frac = (
        len(test_n)
        / len(normal_idx)
    )

    score = abs(
        frac - 0.20
    )

    if (
        best_normal_split is None
        or score
        < best_normal_split[0]
    ):

        best_normal_split = (
            score,
            train_n,
            test_n,
            rs
        )


_, normal_train_idx, normal_test_idx, normal_seed = (
    best_normal_split
)

print(
    f"Normal group-aware split seed={normal_seed}"
)

print(
    f"Normal train={len(normal_train_idx):,} | "
    f"normal test={len(normal_test_idx):,}"
)


# ============================================================
# 4. RUN LEAVE-ONE-FAMILY-OUT
# ============================================================

for family in HOLDOUT_FAMILIES:

    print(
        f"\n{'='*78}"
    )

    print(
        f"UNSEEN ATTACK FAMILY = {family}"
    )

    print(
        '='*78
    )

    family_mask = (
        (y_unsw == 1)
        &
        (
            unsw_meta[
                'attack_cat'
            ]
            .str.lower()
            .values
            ==
            family.lower()
        )
    )

    family_idx = np.where(
        family_mask
    )[0]

    other_attack_idx = np.where(
        (y_unsw == 1)
        &
        (~family_mask)
    )[0]

    print(
        f"Held-out attack rows: "
        f"{len(family_idx):,}"
    )

    print(
        f"Other attack rows available for training: "
        f"{len(other_attack_idx):,}"
    )


    # --------------------------------------------------------
    # TEST =
    # fixed held-out normals
    # +
    # ALL records from unseen attack family
    # --------------------------------------------------------

    test_idx = np.concatenate([
        normal_test_idx,
        family_idx
    ])


    # --------------------------------------------------------
    # INITIAL TRAIN =
    # normal training groups
    # +
    # all other attack families
    # --------------------------------------------------------

    train_initial = np.concatenate([
        normal_train_idx,
        other_attack_idx
    ])


    # --------------------------------------------------------
    # STRICT GROUP PURGE
    #
    # If any training record has an exact feature vector
    # occurring in the test set, remove it from training.
    #
    # We preserve the complete unseen-family test set.
    # --------------------------------------------------------

    test_groups = np.unique(
        feature_hash[
            test_idx
        ]
    )

    train_keep = ~np.isin(
        feature_hash[
            train_initial
        ],
        test_groups
    )

    train_idx = train_initial[
        train_keep
    ]

    removed_from_train = (
        len(train_initial)
        - len(train_idx)
    )


    # --------------------------------------------------------
    # HARD LEAKAGE CHECK
    # --------------------------------------------------------

    shared_groups = np.intersect1d(

        np.unique(
            feature_hash[
                train_idx
            ]
        ),

        np.unique(
            feature_hash[
                test_idx
            ]
        )
    )

    assert len(
        shared_groups
    ) == 0


    ytr = y_unsw[
        train_idx
    ]

    yte = y_unsw[
        test_idx
    ]

    print(
        f"Final train={len(train_idx):,} | "
        f"test={len(test_idx):,}"
    )

    print(
        f"Removed train rows due exact test-group overlap: "
        f"{removed_from_train:,}"
    )

    print(
        f"Train positive={ytr.sum():,} "
        f"({100*ytr.mean():.3f}%)"
    )

    print(
        f"Test unseen positive={yte.sum():,} "
        f"({100*yte.mean():.3f}%)"
    )


    assert len(
        np.unique(ytr)
    ) == 2

    assert len(
        np.unique(yte)
    ) == 2


    # ========================================================
    # 5. PREPROCESSING FIT TRAIN ONLY
    # ========================================================

    Xtr_raw = X_unsw.iloc[
        train_idx
    ].copy()

    Xte_raw = X_unsw.iloc[
        test_idx
    ].copy()


    pre_s3_unsw = make_preprocessor(
        Xtr_raw
    )


    t0 = time.perf_counter()

    Xtr = pre_s3_unsw.fit_transform(
        Xtr_raw
    )

    Xte = pre_s3_unsw.transform(
        Xte_raw
    )

    prep_sec = (
        time.perf_counter()
        - t0
    )


    # ========================================================
    # 6. SAME SUPERVISED MODELS
    # ========================================================

    models = {

        'XGBoost':
            XGBClassifier(
                random_state=CFG['SEED'],
                n_jobs=-1,
                eval_metric='logloss',
                tree_method='hist',
                verbosity=0
            ),

        'RandomForest':
            RandomForestClassifier(
                n_estimators=100,
                random_state=CFG['SEED'],
                n_jobs=-1
            )
    }


    # ========================================================
    # 7. TRAIN + TEST
    # ========================================================

    for model_name, model in models.items():

        print(
            f"\n▶ {model_name}"
        )

        t1 = time.perf_counter()

        model.fit(
            Xtr,
            ytr
        )

        train_sec = (
            time.perf_counter()
            - t1
        )


        t2 = time.perf_counter()

        y_pred = model.predict(
            Xte
        )

        y_score = model.predict_proba(
            Xte
        )[:, 1]

        infer_sec = (
            time.perf_counter()
            - t2
        )


        m = calc_metrics(
            yte,
            y_pred,
            y_score
        )


        # Since all positive test records belong to
        # the unseen family:
        unseen_family_recall = (
            m['Recall']
        )


        S3_UNSW_RESULTS.append({

            'dataset':
                'UNSW-NB15',

            'scenario':
                'S3_unseen_family',

            'holdout_family':
                family,

            'model':
                model_name,

            'n_train':
                len(train_idx),

            'n_test':
                len(test_idx),

            'n_normal_train':
                int(
                    np.sum(
                        ytr == 0
                    )
                ),

            'n_seen_attack_train':
                int(
                    np.sum(
                        ytr == 1
                    )
                ),

            'n_normal_test':
                len(
                    normal_test_idx
                ),

            'n_unseen_attack_test':
                len(
                    family_idx
                ),

            'removed_train_exact_overlap':
                removed_from_train,

            'shared_feature_groups_after_purge':
                len(
                    shared_groups
                ),

            'train_positive_rate':
                float(
                    ytr.mean()
                ),

            'test_positive_rate':
                float(
                    yte.mean()
                ),

            'unseen_family_recall':
                unseen_family_recall,

            'preprocessing_sec':
                prep_sec,

            'train_sec':
                train_sec,

            'inference_sec':
                infer_sec,

            **m
        })


        print(
            f"Precision={m['Precision']:.6f} | "
            f"Unseen Recall={m['Recall']:.6f} | "
            f"F1={m['F1']:.6f} | "
            f"PR-AUC={m['PR_AUC']:.6f} | "
            f"FAR={m['FAR']:.6f}"
        )


# ============================================================
# 8. SAVE
# ========================================================

s3_unsw_df = pd.DataFrame(
    S3_UNSW_RESULTS
)

display(
    s3_unsw_df
)


s3_unsw_df.to_csv(
    OUT
    / 'S3_UNSW_leave_one_family_out_supervised_results.csv',
    index=False
)


# ============================================================
# 9. COMPACT SUMMARY
# ============================================================

s3_unsw_summary = (
    s3_unsw_df[
        [
            'holdout_family',
            'model',
            'n_unseen_attack_test',
            'test_positive_rate',
            'Precision',
            'unseen_family_recall',
            'F1',
            'PR_AUC',
            'FAR',
            'removed_train_exact_overlap'
        ]
    ]
    .sort_values([
        'holdout_family',
        'model'
    ])
)

display(
    s3_unsw_summary
)


s3_unsw_summary.to_csv(
    OUT
    / 'S3_UNSW_leave_one_family_out_summary.csv',
    index=False
)


print(
    "\n✅ STEP 5C-2 selesai"
)

print(
    "Saved:",
    OUT
    / 'S3_UNSW_leave_one_family_out_supervised_results.csv'
)

UNSW rows: 700001
Normal group-aware split seed=48
Normal train=542,224 | normal test=135,562

UNSEEN ATTACK FAMILY = Generic
Held-out attack rows: 7,522
Other attack rows available for training: 14,693
Final train=556,789 | test=143,084
Removed train rows due exact test-group overlap: 128
Train positive=14,565 (2.616%)
Test unseen positive=7,522 (5.257%)

▶ XGBoost
Precision=0.969141 | Unseen Recall=0.997873 | F1=0.983297 | PR-AUC=0.994393 | FAR=0.001763

▶ RandomForest
Precision=0.966714 | Unseen Recall=0.996145 | F1=0.981209 | PR-AUC=0.969660 | FAR=0.001903

UNSEEN ATTACK FAMILY = Exploits
Held-out attack rows: 5,409
Other attack rows available for training: 16,806
Final train=556,476 | test=140,971
Removed train rows due exact test-group overlap: 2,554
Train positive=14,252 (2.561%)
Test unseen positive=5,409 (3.837%)

▶ XGBoost
Precision=0.955868 | Unseen Recall=0.904973 | F1=0.929725 | PR-AUC=0.985575 | FAR=0.001667

▶ RandomForest
Precision=0.948351 | Unseen Recall=0.807913 | F1

,dataset,scenario,holdout_family,model,n_train,n_test,n_normal_train,n_seen_attack_train,n_normal_test,n_unseen_attack_test,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,UNSW-NB15,S3_unseen_family,Generic,XGBoost,556789,143084,542224,14565,135562,7522,...,0.969141,0.997873,0.983297,0.994393,0.001763,0.054129,135323,239,16,7506
1,UNSW-NB15,S3_unseen_family,Generic,RandomForest,556789,143084,542224,14565,135562,7522,...,0.966714,0.996145,0.981209,0.969660,0.001903,0.054171,135304,258,29,7493
2,UNSW-NB15,S3_unseen_family,Exploits,XGBoost,556476,140971,542224,14252,135562,5409,...,0.955868,0.904973,0.929725,0.985575,0.001667,0.036327,135336,226,514,4895
3,UNSW-NB15,S3_unseen_family,Exploits,RandomForest,556476,140971,542224,14252,135562,5409,...,0.948351,0.807913,0.872517,0.982756,0.001756,0.032688,135324,238,1039,4370
4,UNSW-NB15,S3_unseen_family,Fuzzers,XGBoost,556890,140613,542224,14666,135562,5051,...,0.961806,0.109681,0.196908,0.661601,0.000162,0.004096,135540,22,4497,554
5,UNSW-NB15,S3_unseen_family,Fuzzers,RandomForest,556890,140613,542224,14666,135562,5051,...,0.971204,0.146902,0.255202,0.810323,0.000162,0.005433,135540,22,4309,742
6,UNSW-NB15,S3_unseen_family,Reconnaissance,XGBoost,562559,137321,542224,20335,135562,1759,...,0.881969,0.998294,0.936533,0.991900,0.001734,0.014499,135327,235,3,1756
7,UNSW-NB15,S3_unseen_family,Reconnaissance,RandomForest,562559,137321,542224,20335,135562,1759,...,0.869608,0.997157,0.929025,0.977475,0.001940,0.014688,135299,263,5,1754
8,UNSW-NB15,S3_unseen_family,DoS,XGBoost,560277,136729,542224,18053,135562,1167,...,0.795756,0.771208,0.783290,0.897923,0.001704,0.008272,135331,231,267,900
9,UNSW-NB15,S3_unseen_family,DoS,RandomForest,560277,136729,542224,18053,135562,1167,...,0.720751,0.559554,0.630005,0.834220,0.001866,0.006626,135309,253,514,653


,holdout_family,model,n_unseen_attack_test,test_positive_rate,Precision,unseen_family_recall,F1,PR_AUC,FAR,removed_train_exact_overlap
9,DoS,RandomForest,1167,0.008535,0.720751,0.559554,0.630005,0.834220,0.001866,2995
8,DoS,XGBoost,1167,0.008535,0.795756,0.771208,0.783290,0.897923,0.001704,2995
3,Exploits,RandomForest,5409,0.038370,0.948351,0.807913,0.872517,0.982756,0.001756,2554
2,Exploits,XGBoost,5409,0.038370,0.955868,0.904973,0.929725,0.985575,0.001667,2554
5,Fuzzers,RandomForest,5051,0.035921,0.971204,0.146902,0.255202,0.810323,0.000162,2498
4,Fuzzers,XGBoost,5051,0.035921,0.961806,0.109681,0.196908,0.661601,0.000162,2498
1,Generic,RandomForest,7522,0.052571,0.966714,0.996145,0.981209,0.969660,0.001903,128
0,Generic,XGBoost,7522,0.052571,0.969141,0.997873,0.983297,0.994393,0.001763,128
7,Reconnaissance,RandomForest,1759,0.012809,0.869608,0.997157,0.929025,0.977475,0.001940,121
6,Reconnaissance,XGBoost,1759,0.012809,0.881969,0.998294,0.936533,0.991900,0.001734,121



✅ STEP 5C-2 selesai
Saved: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S3_UNSW_leave_one_family_out_supervised_results.csv


In [24]:
# ============================================================
# STEP 5C-3 — CICIDS2018 CROSS-DAY GENERALIZATION
# TRAIN: 02-14-2018
# TEST : 02-15-2018
# XGBoost + Random Forest
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import time


# ============================================================
# 1. LOCATE 02-15 FILE
# ============================================================

exact_day2 = Path(BASE) / '02-15-2018.csv'

if exact_day2.exists():

    DAY2 = str(exact_day2)

else:

    candidates = sorted(
        Path(BASE).glob('*02-15*.csv')
    )

    if len(candidates) == 0:

        raise FileNotFoundError(
            "02-15-2018.csv tidak ditemukan di folder BASE."
        )

    DAY2 = str(
        candidates[0]
    )


print(
    "02-14:",
    CFG['CICIDS']
)

print(
    "02-15:",
    DAY2
)


# ============================================================
# 2. TRAINING SOURCE
#    gunakan CICIDS2018 S0 yang sudah dedup
# ============================================================

X14, y14 = DATASETS[
    'CICIDS2018'
]

X14 = X14.reset_index(
    drop=True
)

y14 = np.asarray(
    y14,
    dtype=int
)


print(
    f"\n02-14 clean training:"
    f" {len(X14):,} rows | "
    f"attack={y14.sum():,} "
    f"({100*y14.mean():.3f}%)"
)


# ============================================================
# 3. LOAD 02-15
# ============================================================

d15 = pd.read_csv(
    DAY2,
    low_memory=False
)

d15.columns = clean_cols(
    d15.columns
)

d15.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)


label_col = next(
    c
    for c in d15.columns
    if 'label' in c
)


raw_label15 = (
    d15[label_col]
    .astype(str)
    .str.strip()
)


y15_all = (
    raw_label15
    .str.upper()
    != 'BENIGN'
).astype(int)


print(
    f"\n02-15 raw rows: "
    f"{len(d15):,}"
)

print(
    "\n02-15 original label distribution:"
)

print(
    raw_label15
    .value_counts()
    .to_string()
)


# ============================================================
# 4. BUILD EXACT SAME FEATURE SCHEMA
# ============================================================

drop15 = [
    label_col
] + [
    c
    for c in d15.columns
    if any(
        k in c
        for k in [
            'timestamp',
            'flow_id',
            'src_ip',
            'dst_ip',
            'source_ip',
            'destination_ip'
        ]
    )
]


X15_raw = (
    d15
    .drop(
        columns=list(
            set(drop15)
        ),
        errors='ignore'
    )
)


# ------------------------------------------------------------
# schema must be compatible
# ------------------------------------------------------------

missing_features = [
    c
    for c in X14.columns
    if c not in X15_raw.columns
]

extra_features = [
    c
    for c in X15_raw.columns
    if c not in X14.columns
]


print(
    "\nMissing 02-15 features:",
    missing_features
)

print(
    "Extra 02-15 features:",
    extra_features
)


if len(
    missing_features
) > 0:

    raise ValueError(
        "Cross-day dihentikan karena schema 02-15 "
        "tidak memiliki seluruh fitur training 02-14."
    )


X15 = (
    X15_raw[
        X14.columns
    ]
    .copy()
    .reset_index(drop=True)
)

y15 = (
    y15_all
    .reset_index(drop=True)
)

label15 = (
    raw_label15
    .reset_index(drop=True)
)


# ============================================================
# 5. AUDIT FEATURE-IDENTICAL LABEL CONFLICTS IN DAY 2
# ============================================================

hash15 = (
    pd.util.hash_pandas_object(
        X15,
        index=False
    )
    .astype('uint64')
    .values
)


conflict_audit = pd.DataFrame({

    'feature_hash':
        hash15,

    'label':
        y15.values
})


group_conflict = (
    conflict_audit
    .groupby(
        'feature_hash'
    )['label']
    .nunique()
)


conflict_hashes = set(
    group_conflict[
        group_conflict > 1
    ].index
)


conflict_mask = np.array([
    h in conflict_hashes
    for h in hash15
])


print(
    "\n02-15 conflicting feature-label groups:",
    len(
        conflict_hashes
    )
)

print(
    "Rows in conflicting groups:",
    int(
        conflict_mask.sum()
    )
)


# Remove ambiguous groups entirely
if conflict_mask.any():

    keep_nonconflict = (
        ~conflict_mask
    )

    X15 = (
        X15.loc[
            keep_nonconflict
        ]
        .reset_index(drop=True)
    )

    y15 = (
        y15.loc[
            keep_nonconflict
        ]
        .reset_index(drop=True)
    )

    label15 = (
        label15.loc[
            keep_nonconflict
        ]
        .reset_index(drop=True)
    )


# ============================================================
# 6. DEDUPLICATE DAY 2
#    same feature vector + same label -> one observation
# ============================================================

before_dedup = len(
    X15
)


tmp15 = X15.copy()

tmp15[
    '__label_binary__'
] = y15.values

tmp15[
    '__label_original__'
] = label15.values


feature_cols = list(
    X15.columns
)


dup_mask = tmp15.duplicated(
    subset=feature_cols,
    keep='first'
)


tmp15 = (
    tmp15.loc[
        ~dup_mask
    ]
    .reset_index(drop=True)
)


y15 = (
    tmp15.pop(
        '__label_binary__'
    )
    .astype(int)
    .reset_index(drop=True)
)


label15 = (
    tmp15.pop(
        '__label_original__'
    )
    .astype(str)
    .reset_index(drop=True)
)


X15 = tmp15


print(
    f"\n02-15 dedup:"
    f" {before_dedup:,}"
    f" -> {len(X15):,}"
    f" | removed="
    f"{before_dedup-len(X15):,}"
)


# ============================================================
# 7. PURGE EXACT CROSS-DAY FEATURE OVERLAP
# ============================================================

train_hash = set(
    pd.util.hash_pandas_object(
        X14,
        index=False
    )
    .astype('uint64')
    .values
)


test_hash = (
    pd.util.hash_pandas_object(
        X15,
        index=False
    )
    .astype('uint64')
    .values
)


overlap_mask = np.array([
    h in train_hash
    for h in test_hash
])


crossday_overlap = int(
    overlap_mask.sum()
)


print(
    "\nExact 02-14 → 02-15 "
    "feature overlap rows:",
    crossday_overlap
)


if crossday_overlap > 0:

    keep = (
        ~overlap_mask
    )

    X15 = (
        X15.loc[
            keep
        ]
        .reset_index(drop=True)
    )

    y15 = (
        y15.loc[
            keep
        ]
        .reset_index(drop=True)
    )

    label15 = (
        label15.loc[
            keep
        ]
        .reset_index(drop=True)
    )


# hard check
shared_final = np.intersect1d(

    np.unique(
        pd.util.hash_pandas_object(
            X14,
            index=False
        ).astype('uint64')
    ),

    np.unique(
        pd.util.hash_pandas_object(
            X15,
            index=False
        ).astype('uint64')
    )
)


assert (
    len(shared_final) == 0
)


print(
    f"\nFINAL 02-15 TEST:"
    f" {len(X15):,} rows | "
    f"attack={y15.sum():,} "
    f"({100*y15.mean():.3f}%)"
)

print(
    "Both classes:",
    len(
        np.unique(
            y15
        )
    ) == 2
)


# ============================================================
# 8. PREPROCESS — FIT 02-14 ONLY
# ============================================================

pre_crossday = make_preprocessor(
    X14
)


t0 = time.perf_counter()


X14_p = pre_crossday.fit_transform(
    X14
)


X15_p = pre_crossday.transform(
    X15
)


prep_sec = (
    time.perf_counter()
    - t0
)


print(
    "\nProcessed shapes:",
    X14_p.shape,
    X15_p.shape
)


# ============================================================
# 9. MODELS
# ============================================================

models = {

    'XGBoost':
        XGBClassifier(
            random_state=CFG['SEED'],
            n_jobs=-1,
            eval_metric='logloss',
            tree_method='hist',
            verbosity=0
        ),

    'RandomForest':
        RandomForestClassifier(
            n_estimators=100,
            random_state=CFG['SEED'],
            n_jobs=-1
        )
}


CROSSDAY_RESULTS = []


# ============================================================
# 10. TRAIN 02-14 -> TEST 02-15
# ============================================================

for model_name, model in models.items():

    print(
        f"\n{'='*70}"
    )

    print(
        model_name
    )

    print(
        '='*70
    )


    t1 = time.perf_counter()


    model.fit(
        X14_p,
        y14
    )


    train_sec = (
        time.perf_counter()
        - t1
    )


    t2 = time.perf_counter()


    pred = model.predict(
        X15_p
    )


    score = model.predict_proba(
        X15_p
    )[:,1]


    infer_sec = (
        time.perf_counter()
        - t2
    )


    m = calc_metrics(
        y15.values,
        pred,
        score
    )


    CROSSDAY_RESULTS.append({

        'dataset':
            'CICIDS2018',

        'scenario':
            'S3_cross_day',

        'train_day':
            '02-14-2018',

        'test_day':
            '02-15-2018',

        'model':
            model_name,

        'n_train':
            len(X14),

        'n_test':
            len(X15),

        'train_positive_rate':
            float(
                y14.mean()
            ),

        'test_positive_rate':
            float(
                y15.mean()
            ),

        'conflicting_groups_removed':
            len(
                conflict_hashes
            ),

        'crossday_overlap_removed':
            crossday_overlap,

        'shared_feature_groups_final':
            len(
                shared_final
            ),

        'preprocessing_sec':
            prep_sec,

        'train_sec':
            train_sec,

        'inference_sec':
            infer_sec,

        **m
    })


    print(
        f"Precision={m['Precision']:.6f} | "
        f"Recall={m['Recall']:.6f} | "
        f"F1={m['F1']:.6f} | "
        f"PR-AUC={m['PR_AUC']:.6f} | "
        f"FAR={m['FAR']:.6f} | "
        f"Alert={m['Alert_Rate']:.6f}"
    )


# ============================================================
# 11. SAVE RESULTS
# ============================================================

crossday_df = pd.DataFrame(
    CROSSDAY_RESULTS
)


display(
    crossday_df
)


crossday_df.to_csv(
    OUT
    / 'S3_CICIDS_crossday_supervised_results.csv',
    index=False
)


# ============================================================
# 12. SAVE AUDIT
# ============================================================

crossday_audit = pd.DataFrame([{

    'train_day':
        '02-14-2018',

    'test_day':
        '02-15-2018',

    'n_train_final':
        len(X14),

    'n_test_final':
        len(X15),

    'train_attack_n':
        int(
            y14.sum()
        ),

    'test_attack_n':
        int(
            y15.sum()
        ),

    'train_attack_pct':
        float(
            100*y14.mean()
        ),

    'test_attack_pct':
        float(
            100*y15.mean()
        ),

    'day2_conflicting_feature_label_groups':
        len(
            conflict_hashes
        ),

    'crossday_exact_overlap_removed':
        crossday_overlap,

    'shared_groups_final':
        len(
            shared_final
        )
}])


display(
    crossday_audit
)


crossday_audit.to_csv(
    OUT
    / 'S3_CICIDS_crossday_audit.csv',
    index=False
)


print(
    "\n✅ CICIDS2018 cross-day selesai."
)

print(
    "Kirim:"
)

print(
    OUT
    / 'S3_CICIDS_crossday_supervised_results.csv'
)

print(
    OUT
    / 'S3_CICIDS_crossday_audit.csv'
)

02-14: /content/drive/MyDrive/ercyris_siklus2/02-14-2018.csv
02-15: /content/drive/MyDrive/ercyris_siklus2/02-15-2018.csv

02-14 clean training: 330,726 rows | attack=45,029 (13.615%)

02-15 raw rows: 1,048,575

02-15 original label distribution:
label
Benign                   996077
DoS attacks-GoldenEye     41508
DoS attacks-Slowloris     10990

Missing 02-15 features: []
Extra 02-15 features: []

02-15 conflicting feature-label groups: 0
Rows in conflicting groups: 0

02-15 dedup: 1,048,575 -> 884,658 | removed=163,917

Exact 02-14 → 02-15 feature overlap rows: 28732

FINAL 02-15 TEST: 855,926 rows | attack=51,314 (5.995%)
Both classes: True

Processed shapes: (330726, 78) (855926, 78)

XGBoost
Precision=0.000000 | Recall=0.000000 | F1=0.000000 | PR-AUC=0.917105 | FAR=0.000005 | Alert=0.000005

RandomForest
Precision=0.000000 | Recall=0.000000 | F1=0.000000 | PR-AUC=0.945189 | FAR=0.000002 | Alert=0.000002


,dataset,scenario,train_day,test_day,model,n_train,n_test,train_positive_rate,test_positive_rate,conflicting_groups_removed,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S3_cross_day,02-14-2018,02-15-2018,XGBoost,330726,855926,0.136152,0.059951,0,...,0.0,0.0,0.0,0.917105,0.000005,0.000005,804608,4,51314,0
1,CICIDS2018,S3_cross_day,02-14-2018,02-15-2018,RandomForest,330726,855926,0.136152,0.059951,0,...,0.0,0.0,0.0,0.945189,0.000002,0.000002,804610,2,51314,0


,train_day,test_day,n_train_final,n_test_final,train_attack_n,test_attack_n,train_attack_pct,test_attack_pct,day2_conflicting_feature_label_groups,crossday_exact_overlap_removed,shared_groups_final
0,02-14-2018,02-15-2018,330726,855926,45029,51314,13.615198,5.995144,0,28732,0



✅ CICIDS2018 cross-day selesai.
Kirim:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S3_CICIDS_crossday_supervised_results.csv
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S3_CICIDS_crossday_audit.csv


In [25]:
# ============================================================
# STEP 5C-3B — CICIDS CROSS-DAY THRESHOLD / SCORE AUDIT
# Threshold selected ONLY from 02-14 calibration
# Final evaluation on untouched 02-15
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve
import numpy as np
import pandas as pd
import time


THRESHOLD_AUDIT = []


# ============================================================
# 1. SPLIT DAY-14 INTO TRAIN + CALIBRATION
# ============================================================

idx14 = np.arange(
    len(X14)
)

fit_idx, cal_idx = train_test_split(
    idx14,
    test_size=0.20,
    random_state=CFG['SEED'],
    stratify=y14
)

Xfit_raw = X14.iloc[
    fit_idx
].copy()

Xcal_raw = X14.iloc[
    cal_idx
].copy()

yfit = y14[
    fit_idx
]

ycal = y14[
    cal_idx
]


print(
    f"02-14 fit={len(fit_idx):,} | "
    f"calibration={len(cal_idx):,}"
)

print(
    f"fit attack={yfit.sum():,} "
    f"({100*yfit.mean():.3f}%)"
)

print(
    f"cal attack={ycal.sum():,} "
    f"({100*ycal.mean():.3f}%)"
)


# ============================================================
# 2. PREPROCESS FIT ONLY ON DAY-14 FIT
# ============================================================

pre_thr = make_preprocessor(
    Xfit_raw
)

Xfit = pre_thr.fit_transform(
    Xfit_raw
)

Xcal = pre_thr.transform(
    Xcal_raw
)

X15_thr = pre_thr.transform(
    X15
)


# ============================================================
# 3. MODELS
# ============================================================

models_thr = {

    'XGBoost':
        XGBClassifier(
            random_state=CFG['SEED'],
            n_jobs=-1,
            eval_metric='logloss',
            tree_method='hist',
            verbosity=0
        ),

    'RandomForest':
        RandomForestClassifier(
            n_estimators=100,
            random_state=CFG['SEED'],
            n_jobs=-1
        )
}


for model_name, model in models_thr.items():

    print(
        f"\n{'='*72}"
    )

    print(
        model_name
    )

    print(
        '='*72
    )

    # --------------------------------------------------------
    # FIT ONLY DAY-14 FIT PARTITION
    # --------------------------------------------------------

    model.fit(
        Xfit,
        yfit
    )


    # --------------------------------------------------------
    # CALIBRATION SCORES — DAY 14 ONLY
    # --------------------------------------------------------

    cal_score = model.predict_proba(
        Xcal
    )[:,1]


    precision_c, recall_c, thresholds_c = (
        precision_recall_curve(
            ycal,
            cal_score
        )
    )


    # F1 per threshold
    f1_c = (
        2 * precision_c[:-1] * recall_c[:-1]
        /
        np.maximum(
            precision_c[:-1] + recall_c[:-1],
            1e-12
        )
    )


    best_idx = int(
        np.nanargmax(
            f1_c
        )
    )


    threshold_cal = float(
        thresholds_c[
            best_idx
        ]
    )


    best_cal_f1 = float(
        f1_c[
            best_idx
        ]
    )


    print(
        f"Day-14 calibration threshold="
        f"{threshold_cal:.8f}"
    )

    print(
        f"Day-14 best calibration F1="
        f"{best_cal_f1:.6f}"
    )


    # --------------------------------------------------------
    # TEST SCORES — DAY 15
    # --------------------------------------------------------

    test_score = model.predict_proba(
        X15_thr
    )[:,1]


    # score distribution by true class
    neg_score = test_score[
        y15.values == 0
    ]

    pos_score = test_score[
        y15.values == 1
    ]


    print(
        "\n02-15 score quantiles NORMAL:"
    )

    print(
        np.quantile(
            neg_score,
            [0, .50, .90, .95, .99, .999, 1]
        )
    )


    print(
        "\n02-15 score quantiles ATTACK:"
    )

    print(
        np.quantile(
            pos_score,
            [0, .50, .90, .95, .99, .999, 1]
        )
    )


    print(
        "\nMax score normal:",
        float(
            np.max(
                neg_score
            )
        )
    )

    print(
        "Max score attack:",
        float(
            np.max(
                pos_score
            )
        )
    )


    # ========================================================
    # A. DEFAULT 0.5
    # ========================================================

    pred_default = (
        test_score >= 0.5
    ).astype(int)

    m_default = calc_metrics(
        y15.values,
        pred_default,
        test_score
    )


    # ========================================================
    # B. DAY-14 CALIBRATED THRESHOLD
    # ========================================================

    pred_cal = (
        test_score >= threshold_cal
    ).astype(int)

    m_cal = calc_metrics(
        y15.values,
        pred_cal,
        test_score
    )


    print(
        "\nDEFAULT threshold=0.5"
    )

    print(
        f"Precision={m_default['Precision']:.6f} | "
        f"Recall={m_default['Recall']:.6f} | "
        f"F1={m_default['F1']:.6f} | "
        f"FAR={m_default['FAR']:.6f} | "
        f"Alert={m_default['Alert_Rate']:.6f}"
    )


    print(
        "\nDAY-14 CALIBRATED threshold"
    )

    print(
        f"Precision={m_cal['Precision']:.6f} | "
        f"Recall={m_cal['Recall']:.6f} | "
        f"F1={m_cal['F1']:.6f} | "
        f"FAR={m_cal['FAR']:.6f} | "
        f"Alert={m_cal['Alert_Rate']:.6f}"
    )


    THRESHOLD_AUDIT.append({

        'dataset':
            'CICIDS2018',

        'model':
            model_name,

        'train_day':
            '02-14-2018',

        'test_day':
            '02-15-2018',

        'threshold_default':
            0.5,

        'threshold_day14_calibrated':
            threshold_cal,

        'day14_calibration_best_F1':
            best_cal_f1,

        'test_score_normal_median':
            float(
                np.median(
                    neg_score
                )
            ),

        'test_score_attack_median':
            float(
                np.median(
                    pos_score
                )
            ),

        'test_score_normal_max':
            float(
                np.max(
                    neg_score
                )
            ),

        'test_score_attack_max':
            float(
                np.max(
                    pos_score
                )
            ),

        'default_Precision':
            m_default[
                'Precision'
            ],

        'default_Recall':
            m_default[
                'Recall'
            ],

        'default_F1':
            m_default[
                'F1'
            ],

        'default_FAR':
            m_default[
                'FAR'
            ],

        'calibrated_Precision':
            m_cal[
                'Precision'
            ],

        'calibrated_Recall':
            m_cal[
                'Recall'
            ],

        'calibrated_F1':
            m_cal[
                'F1'
            ],

        'calibrated_FAR':
            m_cal[
                'FAR'
            ],

        'PR_AUC':
            m_cal[
                'PR_AUC'
            ]
    })


# ============================================================
# 4. SAVE
# ============================================================

threshold_audit_df = pd.DataFrame(
    THRESHOLD_AUDIT
)

display(
    threshold_audit_df
)


threshold_audit_df.to_csv(
    OUT
    / 'S3_CICIDS_crossday_threshold_audit.csv',
    index=False
)


print(
    "\n✅ Saved:",
    OUT
    / 'S3_CICIDS_crossday_threshold_audit.csv'
)

02-14 fit=264,580 | calibration=66,146
fit attack=36,023 (13.615%)
cal attack=9,006 (13.615%)

XGBoost
Day-14 calibration threshold=0.95205748
Day-14 best calibration F1=1.000000

02-15 score quantiles NORMAL:
[2.10247072e-06 2.56105750e-06 3.62386254e-06 3.75541640e-06
 1.58179409e-05 7.10672321e-05 9.96588469e-01]

02-15 score quantiles ATTACK:
[2.10247072e-06 5.69691074e-05 1.57380127e-04 1.57380127e-04
 2.71153316e-04 3.97555181e-04 5.22919698e-04]

Max score normal: 0.9965884685516357
Max score attack: 0.0005229196976870298

DEFAULT threshold=0.5
Precision=0.000000 | Recall=0.000000 | F1=0.000000 | FAR=0.000005 | Alert=0.000005

DAY-14 CALIBRATED threshold
Precision=0.000000 | Recall=0.000000 | F1=0.000000 | FAR=0.000005 | Alert=0.000005

RandomForest
Day-14 calibration threshold=0.85000000
Day-14 best calibration F1=0.999944

02-15 score quantiles NORMAL:
[0.   0.   0.   0.   0.   0.01 0.88]

02-15 score quantiles ATTACK:
[0.   0.03 0.06 0.09 0.19 0.21 0.22]

Max score normal: 0.

,dataset,model,train_day,test_day,threshold_default,threshold_day14_calibrated,day14_calibration_best_F1,test_score_normal_median,test_score_attack_median,test_score_normal_max,test_score_attack_max,default_Precision,default_Recall,default_F1,default_FAR,calibrated_Precision,calibrated_Recall,calibrated_F1,calibrated_FAR,PR_AUC
0,CICIDS2018,XGBoost,02-14-2018,02-15-2018,0.5,0.952057,1.000000,0.000003,0.000057,0.996588,0.000523,0.0,0.0,0.0,0.000005,0.0,0.0,0.0,0.000005,0.904523
1,CICIDS2018,RandomForest,02-14-2018,02-15-2018,0.5,0.850000,0.999944,0.000000,0.030000,0.880000,0.220000,0.0,0.0,0.0,0.000002,0.0,0.0,0.0,0.000002,0.775773



✅ Saved: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S3_CICIDS_crossday_threshold_audit.csv


In [26]:
# ============================================================
# S4A — ISOLATION FOREST PARAMETER SENSITIVITY
# Multi-dataset confirmatory analysis
# ============================================================

from sklearn.ensemble import IsolationForest
import pandas as pd
import numpy as np
import time
import gc


S4_IF_CONTAMINATIONS = [
    0.01,
    0.03,
    0.05,
    0.10
]

S4_IF_RESULTS = []


# ============================================================
# RUN
# ============================================================

for name, (X, y) in DATASETS.items():

    print(
        f"\n{'='*78}"
    )

    print(
        f"S4A ISOLATION FOREST SENSITIVITY — {name}"
    )

    print(
        '='*78
    )

    # --------------------------------------------------------
    # SAME S0 TRAIN / TEST
    # --------------------------------------------------------

    tr, te = SPLITS[
        name
    ]

    Xtr_raw = X.iloc[
        tr
    ].copy()

    Xte_raw = X.iloc[
        te
    ].copy()

    ytr = y[
        tr
    ]

    yte = y[
        te
    ]


    # --------------------------------------------------------
    # PREPROCESSING FIT TRAIN ONLY
    # --------------------------------------------------------

    pre_if_s4 = make_preprocessor(
        Xtr_raw
    )

    t0 = time.perf_counter()

    Xtr = pre_if_s4.fit_transform(
        Xtr_raw
    )

    Xte = pre_if_s4.transform(
        Xte_raw
    )

    prep_sec = (
        time.perf_counter()
        - t0
    )


    # ========================================================
    # PARAMETER SWEEP
    # ========================================================

    for contamination in S4_IF_CONTAMINATIONS:

        print(
            f"\n--- contamination={contamination} ---"
        )

        model = IsolationForest(

            n_estimators=200,

            max_samples=min(
                4096,
                len(Xtr)
            ),

            contamination=contamination,

            random_state=CFG['SEED'],

            n_jobs=-1
        )


        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        t1 = time.perf_counter()

        model.fit(
            Xtr
        )

        train_sec = (
            time.perf_counter()
            - t1
        )


        # ----------------------------------------------------
        # TEST
        #
        # predict() menggunakan offset yang ditetapkan oleh
        # contamination TRAINING, bukan label test.
        # ----------------------------------------------------

        t2 = time.perf_counter()

        pred_raw = model.predict(
            Xte
        )

        # sklearn:
        #  1 = normal
        # -1 = anomaly
        y_pred = (
            pred_raw == -1
        ).astype(int)

        # score besar = semakin anomalous
        anomaly_score = -model.score_samples(
            Xte
        )

        infer_sec = (
            time.perf_counter()
            - t2
        )


        # ----------------------------------------------------
        # METRICS
        # ----------------------------------------------------

        m = calc_metrics(
            yte,
            y_pred,
            anomaly_score
        )


        S4_IF_RESULTS.append({

            'dataset':
                name,

            'scenario':
                'S4_parameter_sensitivity',

            'model':
                'IsolationForest',

            'parameter':
                'contamination',

            'parameter_value':
                contamination,

            'n_estimators':
                200,

            'max_samples':
                min(
                    4096,
                    len(Xtr)
                ),

            'n_train':
                len(Xtr),

            'n_test':
                len(Xte),

            'train_positive_rate':
                float(
                    ytr.mean()
                ),

            'test_positive_rate':
                float(
                    yte.mean()
                ),

            'preprocessing_sec':
                prep_sec,

            'train_sec':
                train_sec,

            'inference_sec':
                infer_sec,

            **m
        })


        print(
            f"F1={m['F1']:.6f} | "
            f"Precision={m['Precision']:.6f} | "
            f"Recall={m['Recall']:.6f} | "
            f"FAR={m['FAR']:.6f} | "
            f"Alert={m['Alert_Rate']:.6f}"
        )

    gc.collect()


# ============================================================
# SAVE FULL RESULTS
# ============================================================

s4_if_df = pd.DataFrame(
    S4_IF_RESULTS
)

display(
    s4_if_df
)

s4_if_df.to_csv(
    OUT
    / 'S4A_isolationforest_parameter_sensitivity.csv',
    index=False
)


# ============================================================
# VOLATILITY SUMMARY
# ============================================================

s4_if_summary = (

    s4_if_df

    .groupby(
        'dataset'
    )

    .agg(

        F1_min=(
            'F1',
            'min'
        ),

        F1_max=(
            'F1',
            'max'
        ),

        FAR_min=(
            'FAR',
            'min'
        ),

        FAR_max=(
            'FAR',
            'max'
        ),

        Alert_min=(
            'Alert_Rate',
            'min'
        ),

        Alert_max=(
            'Alert_Rate',
            'max'
        )
    )

    .reset_index()
)


s4_if_summary[
    'F1_range'
] = (
    s4_if_summary[
        'F1_max'
    ]
    -
    s4_if_summary[
        'F1_min'
    ]
)


s4_if_summary[
    'FAR_range'
] = (
    s4_if_summary[
        'FAR_max'
    ]
    -
    s4_if_summary[
        'FAR_min'
    ]
)


s4_if_summary[
    'Alert_range'
] = (
    s4_if_summary[
        'Alert_max'
    ]
    -
    s4_if_summary[
        'Alert_min'
    ]
)


print(
    "\n=== S4A VOLATILITY SUMMARY ==="
)

display(
    s4_if_summary
)


s4_if_summary.to_csv(

    OUT
    / 'S4A_isolationforest_volatility_summary.csv',

    index=False
)


print(
    "\n✅ S4A selesai."
)

print(
    "Kirim dua file:"
)

print(
    OUT
    / 'S4A_isolationforest_parameter_sensitivity.csv'
)

print(
    OUT
    / 'S4A_isolationforest_volatility_summary.csv'
)


S4A ISOLATION FOREST SENSITIVITY — CICIDS2018

--- contamination=0.01 ---
F1=0.000000 | Precision=0.000000 | Recall=0.000000 | FAR=0.011866 | Alert=0.010250

--- contamination=0.03 ---
F1=0.000000 | Precision=0.000000 | Recall=0.000000 | FAR=0.034197 | Alert=0.029541

--- contamination=0.05 ---
F1=0.000000 | Precision=0.000000 | Recall=0.000000 | FAR=0.058855 | Alert=0.050842

--- contamination=0.1 ---
F1=0.000383 | Precision=0.000451 | Recall=0.000333 | FAR=0.116398 | Alert=0.100596

S4A ISOLATION FOREST SENSITIVITY — HDFS

--- contamination=0.01 ---
F1=0.308304 | Precision=0.607485 | Recall=0.206570 | FAR=0.004039 | Alert=0.009988

--- contamination=0.03 ---
F1=0.693490 | Precision=0.674022 | Recall=0.714117 | FAR=0.010452 | Alert=0.031122

--- contamination=0.05 ---
F1=0.634471 | Precision=0.502244 | Recall=0.861202 | FAR=0.025830 | Alert=0.050368

--- contamination=0.1 ---
F1=0.456159 | Precision=0.295470 | Recall=1.000000 | FAR=0.072160 | Alert=0.099415

S4A ISOLATION FOREST SENS

,dataset,scenario,model,parameter,parameter_value,n_estimators,max_samples,n_train,n_test,train_positive_rate,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S4_parameter_sensitivity,IsolationForest,contamination,0.01,200,4096,264580,66146,0.136152,...,0.000000,0.000000,0.000000,0.114855,0.011866,0.010250,56462,678,9006,0
1,CICIDS2018,S4_parameter_sensitivity,IsolationForest,contamination,0.03,200,4096,264580,66146,0.136152,...,0.000000,0.000000,0.000000,0.114855,0.034197,0.029541,55186,1954,9006,0
2,CICIDS2018,S4_parameter_sensitivity,IsolationForest,contamination,0.05,200,4096,264580,66146,0.136152,...,0.000000,0.000000,0.000000,0.114855,0.058855,0.050842,53777,3363,9006,0
3,CICIDS2018,S4_parameter_sensitivity,IsolationForest,contamination,0.10,200,4096,264580,66146,0.136152,...,0.000451,0.000333,0.000383,0.114855,0.116398,0.100596,50489,6651,9003,3
4,HDFS,S4_parameter_sensitivity,IsolationForest,contamination,0.01,200,4096,460028,115033,0.029257,...,0.607485,0.206570,0.308304,0.614789,0.004039,0.009988,111203,451,2681,698
5,HDFS,S4_parameter_sensitivity,IsolationForest,contamination,0.03,200,4096,460028,115033,0.029257,...,0.674022,0.714117,0.693490,0.614789,0.010452,0.031122,110487,1167,966,2413
6,HDFS,S4_parameter_sensitivity,IsolationForest,contamination,0.05,200,4096,460028,115033,0.029257,...,0.502244,0.861202,0.634471,0.614789,0.025830,0.050368,108770,2884,469,2910
7,HDFS,S4_parameter_sensitivity,IsolationForest,contamination,0.10,200,4096,460028,115033,0.029257,...,0.295470,1.000000,0.456159,0.614789,0.072160,0.099415,103597,8057,0,3379
8,BGL,S4_parameter_sensitivity,IsolationForest,contamination,0.01,200,4096,39989,10011,0.073370,...,0.000000,0.000000,0.000000,0.096167,0.011752,0.010888,9166,109,736,0
9,BGL,S4_parameter_sensitivity,IsolationForest,contamination,0.03,200,4096,39989,10011,0.073370,...,0.018987,0.008152,0.011407,0.096167,0.033423,0.031565,8965,310,730,6



=== S4A VOLATILITY SUMMARY ===


,dataset,F1_min,F1_max,FAR_min,FAR_max,Alert_min,Alert_max,F1_range,FAR_range,Alert_range
0,BGL,0.000000,0.085297,0.011752,0.104582,0.010888,0.104485,0.085297,0.092830,0.093597
1,CICIDS2018,0.000000,0.000383,0.011866,0.116398,0.010250,0.100596,0.000383,0.104533,0.090346
2,HDFS,0.308304,0.693490,0.004039,0.072160,0.009988,0.099415,0.385187,0.068121,0.089427
3,UNSW-NB15,0.081109,0.402899,0.008054,0.076946,0.009469,0.101305,0.321789,0.068892,0.091835



✅ S4A selesai.
Kirim dua file:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S4A_isolationforest_parameter_sensitivity.csv
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S4A_isolationforest_volatility_summary.csv


In [27]:
# ============================================================
# ER-CyRIS — SAVE RESUME CHECKPOINT
# Jalankan SEKALI sebelum menutup Colab
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import os

CHECKPOINT_DIR = Path(
    '/content/drive/MyDrive/ercyris_siklus1_confirmatory/checkpoint'
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Checkpoint folder:")
print(CHECKPOINT_DIR)


# ============================================================
# 1. SAVE CLEAN DATASETS
# ============================================================

dataset_manifest = {}

for name, (X, y) in DATASETS.items():

    print(f"\nSaving {name} ...")

    safe_name = (
        name
        .replace('-', '_')
        .replace(' ', '_')
    )

    df_save = X.copy()

    df_save['__label__'] = np.asarray(
        y,
        dtype=int
    )

    path = (
        CHECKPOINT_DIR
        / f'{safe_name}_clean.parquet'
    )

    df_save.to_parquet(
        path,
        index=False
    )

    dataset_manifest[name] = {
        'file': path.name,
        'rows': int(len(df_save)),
        'features': int(len(X.columns))
    }

    print(
        f"✅ {name}: "
        f"{len(df_save):,} rows "
        f"saved to {path.name}"
    )


# ============================================================
# 2. SAVE SPLITS
# ============================================================

split_data = {}

for name, (tr, te) in SPLITS.items():

    safe_name = (
        name
        .replace('-', '_')
        .replace(' ', '_')
    )

    split_data[
        f'{safe_name}__train'
    ] = np.asarray(
        tr,
        dtype=np.int64
    )

    split_data[
        f'{safe_name}__test'
    ] = np.asarray(
        te,
        dtype=np.int64
    )


np.savez_compressed(
    CHECKPOINT_DIR
    / 'S0_splits.npz',
    **split_data
)

print(
    "\n✅ SPLITS saved"
)


# ============================================================
# 3. SAVE MANIFEST
# ============================================================

with open(
    CHECKPOINT_DIR
    / 'checkpoint_manifest.json',
    'w'
) as f:

    json.dump(
        dataset_manifest,
        f,
        indent=2
    )


# ============================================================
# 4. SAVE CURRENT PROGRESS MARKER
# ============================================================

progress = {

    'completed': [
        'S0 supervised',
        'S0 IsolationForest',
        'S0 LOF',
        'S0 DBSCAN inductive',
        'S1 imbalance',
        'S2 supervised noise/missing',
        'S2 IsolationForest',
        'S2 LOF',
        'S2 DBSCAN inductive',
        'S3 HDFS/BGL shift',
        'S3 UNSW unseen-family',
        'S3 CICIDS cross-day',
        'S3 CICIDS threshold audit',
        'S4A IsolationForest sensitivity'
    ],

    'next_step':
        'S4B kNN-distance sensitivity'
}


with open(
    CHECKPOINT_DIR
    / 'research_progress.json',
    'w'
) as f:

    json.dump(
        progress,
        f,
        indent=2
    )


print(
    "\n=============================================="
)

print(
    "✅ CHECKPOINT COMPLETE"
)

print(
    "Besok tidak perlu load dataset dari raw lagi."
)

print(
    "Next step = S4B kNN-distance sensitivity"
)

print(
    "=============================================="
)

Checkpoint folder:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/checkpoint

Saving CICIDS2018 ...
✅ CICIDS2018: 330,726 rows saved to CICIDS2018_clean.parquet

Saving HDFS ...
✅ HDFS: 575,061 rows saved to HDFS_clean.parquet

Saving BGL ...
✅ BGL: 50,000 rows saved to BGL_clean.parquet

Saving UNSW-NB15 ...
✅ UNSW-NB15: 700,001 rows saved to UNSW_NB15_clean.parquet

✅ SPLITS saved

✅ CHECKPOINT COMPLETE
Besok tidak perlu load dataset dari raw lagi.
Next step = S4B kNN-distance sensitivity


In [2]:
# ============================================================
# ER-CyRIS — RESUME FROM CHECKPOINT
# Jalankan setelah runtime Colab baru
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import json


CHECKPOINT_DIR = Path(
    '/content/drive/MyDrive/ercyris_siklus1_confirmatory/checkpoint'
)


# ============================================================
# 1. LOAD MANIFEST
# ============================================================

with open(
    CHECKPOINT_DIR
    / 'checkpoint_manifest.json'
) as f:

    manifest = json.load(
        f
    )


# ============================================================
# 2. RESTORE DATASETS
# ============================================================

DATASETS = {}

for name, info in manifest.items():

    print(
        f"Loading {name} ..."
    )

    df = pd.read_parquet(
        CHECKPOINT_DIR
        / info['file']
    )

    y = (
        df.pop('__label__')
        .to_numpy(
            dtype=int
        )
    )

    X = df.reset_index(
        drop=True
    )

    DATASETS[name] = (
        X,
        y
    )

    print(
        f"✅ {name}: "
        f"{len(X):,} rows × "
        f"{X.shape[1]} features"
    )


# ============================================================
# 3. RESTORE S0 SPLITS
# ============================================================

saved_splits = np.load(
    CHECKPOINT_DIR
    / 'S0_splits.npz'
)

SPLITS = {}

for name in DATASETS.keys():

    safe_name = (
        name
        .replace('-', '_')
        .replace(' ', '_')
    )

    tr = saved_splits[
        f'{safe_name}__train'
    ]

    te = saved_splits[
        f'{safe_name}__test'
    ]

    SPLITS[name] = (
        tr,
        te
    )


# ============================================================
# 4. SHOW PROGRESS
# ============================================================

with open(
    CHECKPOINT_DIR
    / 'research_progress.json'
) as f:

    progress = json.load(
        f
    )


print(
    "\n=============================================="
)

print(
    "✅ CHECKPOINT RESTORED"
)

print(
    "Datasets:",
    list(
        DATASETS.keys()
    )
)

print(
    "Next step:",
    progress[
        'next_step'
    ]
)

print(
    "=============================================="
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading CICIDS2018 ...
✅ CICIDS2018: 330,726 rows × 78 features
Loading HDFS ...
✅ HDFS: 575,061 rows × 22 features
Loading BGL ...
✅ BGL: 50,000 rows × 11 features
Loading UNSW-NB15 ...
✅ UNSW-NB15: 700,001 rows × 43 features

✅ CHECKPOINT RESTORED
Datasets: ['CICIDS2018', 'HDFS', 'BGL', 'UNSW-NB15']
Next step: S4B kNN-distance sensitivity


In [8]:
# ============================================================
# ER-CyRIS — BOOTSTRAP SETELAH COLAB RUNTIME RESET
# Jalankan SEKALI setelah RESUME checkpoint
# Setelah ini langsung jalankan ulang S4B
# ============================================================

import os
import time
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OrdinalEncoder,
    MinMaxScaler
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIG — SAMA DENGAN NOTEBOOK S0 v4
# ============================================================

CFG = {
    'SEED': 42
}

np.random.seed(
    CFG['SEED']
)


# ============================================================
# 2. OUTPUT DIRECTORY
# ============================================================

OUT = Path(
    '/content/drive/MyDrive/'
    'ercyris_siklus1_confirmatory/'
    'step2_s0/'
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. PREPROCESSOR
# IDENTIK DENGAN NOTEBOOK STEP2A S0 v4
# ============================================================

def make_preprocessor(Xtr):

    numeric_cols = list(
        Xtr.select_dtypes(
            include=np.number
        ).columns
    )

    categorical_cols = [
        c
        for c in Xtr.columns
        if c not in numeric_cols
    ]


    num_pipe = Pipeline([
        (
            'imputer',
            SimpleImputer(
                strategy='median'
            )
        )
    ])


    cat_pipe = Pipeline([
        (
            'imputer',
            SimpleImputer(
                strategy='most_frequent'
            )
        ),

        (
            'ordinal',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            )
        )
    ])


    pre = ColumnTransformer(

        transformers=[

            (
                'num',
                num_pipe,
                numeric_cols
            ),

            (
                'cat',
                cat_pipe,
                categorical_cols
            )
        ],

        remainder='drop',

        sparse_threshold=0
    )


    pipe = Pipeline([

        (
            'columns',
            pre
        ),

        (
            'scale',
            MinMaxScaler()
        )
    ])


    return pipe


# ============================================================
# 4. METRICS
# IDENTIK DENGAN NOTEBOOK STEP2A S0 v4
# ============================================================

def calc_metrics(
    y_true,
    y_pred,
    y_score
):

    tn, fp, fn, tp = confusion_matrix(

        y_true,
        y_pred,

        labels=[
            0,
            1
        ]

    ).ravel()


    N = (
        tn
        + fp
        + fn
        + tp
    )


    return {

        'Precision':
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        'Recall':
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        'F1':
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        'PR_AUC':
            average_precision_score(
                y_true,
                y_score
            ),

        'FAR':
            fp
            / max(
                fp + tn,
                1
            ),

        'Alert_Rate':
            (
                tp + fp
            )
            / max(
                N,
                1
            ),

        'TN':
            int(tn),

        'FP':
            int(fp),

        'FN':
            int(fn),

        'TP':
            int(tp)
    }


# ============================================================
# 5. HARD CHECK
# ============================================================

required_objects = [
    'DATASETS',
    'SPLITS',
    'CFG',
    'OUT',
    'make_preprocessor',
    'calc_metrics',
    'GroupShuffleSplit',
    'NearestNeighbors'
]


missing = [
    x
    for x in required_objects
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Masih ada object yang belum dipulihkan: "
        + ", ".join(missing)
        + "\nJalankan RESUME CHECKPOINT terlebih dahulu."
    )


# ============================================================
# 6. DATASET / SPLIT SANITY CHECK
# ============================================================

print(
    "=============================================="
)

print(
    "✅ BOOTSTRAP S4B READY"
)

print(
    "SEED:",
    CFG['SEED']
)

print(
    "OUT:",
    OUT
)

print(
    "DATASETS:",
    list(DATASETS.keys())
)


for name in DATASETS:

    X, y = DATASETS[name]

    tr, te = SPLITS[name]

    print(
        f"{name:12s} | "
        f"rows={len(X):,} | "
        f"train={len(tr):,} | "
        f"test={len(te):,}"
    )


print(
    "=============================================="
)

print(
    "Sekarang langsung jalankan ulang cell S4B."
)

✅ BOOTSTRAP S4B READY
SEED: 42
OUT: /content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0
DATASETS: ['CICIDS2018', 'HDFS', 'BGL', 'UNSW-NB15']
CICIDS2018   | rows=330,726 | train=264,580 | test=66,146
HDFS         | rows=575,061 | train=460,028 | test=115,033
BGL          | rows=50,000 | train=39,989 | test=10,011
UNSW-NB15    | rows=700,001 | train=559,968 | test=140,033
Sekarang langsung jalankan ulang cell S4B.


In [9]:
# ============================================================
# S4B — kNN-DISTANCE PARAMETER SENSITIVITY
# Multi-dataset confirmatory analysis
# k = 5, 10, 20, 50
# ============================================================

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupShuffleSplit

import pandas as pd
import numpy as np
import time
import gc


S4_K_VALUES = [
    5,
    10,
    20,
    50
]

MAX_K = max(
    S4_K_VALUES
)

REF_MAX = 15000
CAL_MAX = 10000

THRESHOLD_Q = 0.95

S4_KNN_RESULTS = []


# ============================================================
# HELPER
# ============================================================

def capped_sample(
    idx,
    max_n,
    seed
):

    idx = np.asarray(
        idx,
        dtype=int
    )

    if len(idx) <= max_n:
        return idx

    rng = np.random.RandomState(
        seed
    )

    return np.sort(
        rng.choice(
            idx,
            size=max_n,
            replace=False
        )
    )


# ============================================================
# RUN DATASET BY DATASET
# ============================================================

for name, (X, y) in DATASETS.items():

    print(
        f"\n{'='*78}"
    )

    print(
        f"S4B kNN-DISTANCE SENSITIVITY — {name}"
    )

    print(
        '='*78
    )


    # --------------------------------------------------------
    # SAME S0 SPLIT
    # --------------------------------------------------------

    tr, te = SPLITS[
        name
    ]

    Xtr_raw = (
        X.iloc[
            tr
        ]
        .reset_index(
            drop=True
        )
    )

    Xte_raw = (
        X.iloc[
            te
        ]
        .reset_index(
            drop=True
        )
    )

    ytr = np.asarray(
        y[tr],
        dtype=int
    )

    yte = np.asarray(
        y[te],
        dtype=int
    )


    # ========================================================
    # 1. GROUP IDS INSIDE S0 TRAIN
    #
    # reference and calibration must not contain
    # identical feature vectors across partitions
    # ========================================================

    train_groups = (
        pd.util.hash_pandas_object(
            Xtr_raw,
            index=False
        )
        .astype(
            'uint64'
        )
        .values
    )


    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.25,
        random_state=CFG['SEED']
    )


    ref_rel, cal_rel = next(
        gss.split(
            Xtr_raw,
            groups=train_groups
        )
    )


    # --------------------------------------------------------
    # CAP COMPUTATION SIZE
    # --------------------------------------------------------

    ref_idx = capped_sample(
        ref_rel,
        REF_MAX,
        CFG['SEED']
    )

    cal_idx = capped_sample(
        cal_rel,
        CAL_MAX,
        CFG['SEED'] + 1
    )


    # --------------------------------------------------------
    # HARD GROUP-DISJOINT CHECK
    # --------------------------------------------------------

    ref_groups = np.unique(
        train_groups[
            ref_idx
        ]
    )

    cal_groups = np.unique(
        train_groups[
            cal_idx
        ]
    )

    shared_ref_cal = np.intersect1d(
        ref_groups,
        cal_groups
    )

    assert (
        len(
            shared_ref_cal
        ) == 0
    )


    print(
        f"Reference={len(ref_idx):,} | "
        f"Calibration={len(cal_idx):,} | "
        f"Test={len(te):,}"
    )

    print(
        "Shared exact feature groups "
        "reference/calibration:",
        len(
            shared_ref_cal
        )
    )


    # ========================================================
    # 2. PREPROCESS FIT ON S0 TRAIN ONLY
    # ========================================================

    pre_knn_s4 = make_preprocessor(
        Xtr_raw
    )

    t0 = time.perf_counter()


    Xtr_p = pre_knn_s4.fit_transform(
        Xtr_raw
    )

    Xte_p = pre_knn_s4.transform(
        Xte_raw
    )


    prep_sec = (
        time.perf_counter()
        - t0
    )


    Xref = Xtr_p[
        ref_idx
    ]

    Xcal = Xtr_p[
        cal_idx
    ]


    # ========================================================
    # 3. FIT REFERENCE NEIGHBOR INDEX
    # ========================================================

    print(
        "\nFitting nearest-neighbor reference..."
    )

    t1 = time.perf_counter()


    nn = NearestNeighbors(
        n_neighbors=MAX_K,
        metric='euclidean',
        algorithm='auto',
        n_jobs=-1
    )

    nn.fit(
        Xref
    )


    fit_sec = (
        time.perf_counter()
        - t1
    )


    # ========================================================
    # 4. COMPUTE DISTANCES ONCE
    #
    # IMPORTANT:
    # calibration and test are NOT reference records/groups,
    # so there is NO self-neighbor column to remove.
    # ========================================================

    print(
        "Computing calibration distances..."
    )

    t2 = time.perf_counter()


    d_cal, _ = nn.kneighbors(
        Xcal,
        n_neighbors=MAX_K,
        return_distance=True
    )


    cal_query_sec = (
        time.perf_counter()
        - t2
    )


    print(
        "Computing held-out test distances..."
    )

    t3 = time.perf_counter()


    d_test, _ = nn.kneighbors(
        Xte_p,
        n_neighbors=MAX_K,
        return_distance=True
    )


    test_query_sec = (
        time.perf_counter()
        - t3
    )


    # ========================================================
    # 5. k SWEEP
    # ========================================================

    for k in S4_K_VALUES:

        print(
            f"\n--- k={k} ---"
        )


        # kth nearest TRAINING reference
        # zero-based position = k-1
        cal_score = d_cal[
            :,
            k - 1
        ]

        test_score = d_test[
            :,
            k - 1
        ]


        # ----------------------------------------------------
        # TRAIN-ONLY CALIBRATION THRESHOLD
        # ----------------------------------------------------

        threshold = float(
            np.quantile(
                cal_score,
                THRESHOLD_Q
            )
        )


        y_pred = (
            test_score > threshold
        ).astype(int)


        m = calc_metrics(
            yte,
            y_pred,
            test_score
        )


        S4_KNN_RESULTS.append({

            'dataset':
                name,

            'scenario':
                'S4_parameter_sensitivity',

            'model':
                'kNN-distance',

            'parameter':
                'k',

            'parameter_value':
                k,

            'threshold_quantile':
                THRESHOLD_Q,

            'calibrated_threshold':
                threshold,

            'reference_n':
                len(
                    ref_idx
                ),

            'calibration_n':
                len(
                    cal_idx
                ),

            'test_n':
                len(
                    te
                ),

            'shared_ref_cal_groups':
                len(
                    shared_ref_cal
                ),

            'train_positive_rate':
                float(
                    ytr.mean()
                ),

            'test_positive_rate':
                float(
                    yte.mean()
                ),

            'preprocessing_sec':
                prep_sec,

            'nn_fit_sec':
                fit_sec,

            'calibration_query_sec':
                cal_query_sec,

            'test_query_sec':
                test_query_sec,

            **m
        })


        print(
            f"Threshold={threshold:.6f} | "
            f"F1={m['F1']:.6f} | "
            f"Precision={m['Precision']:.6f} | "
            f"Recall={m['Recall']:.6f} | "
            f"PR-AUC={m['PR_AUC']:.6f} | "
            f"FAR={m['FAR']:.6f} | "
            f"Alert={m['Alert_Rate']:.6f}"
        )


    # ========================================================
    # CLEAN MEMORY
    # ========================================================

    del (
        Xtr_p,
        Xte_p,
        Xref,
        Xcal,
        d_cal,
        d_test,
        nn
    )

    gc.collect()


# ============================================================
# 6. SAVE FULL RESULTS
# ============================================================

s4_knn_df = pd.DataFrame(
    S4_KNN_RESULTS
)


display(
    s4_knn_df
)


s4_knn_df.to_csv(
    OUT
    / 'S4B_knn_distance_parameter_sensitivity.csv',
    index=False
)


# ============================================================
# 7. VOLATILITY SUMMARY
# ============================================================

s4_knn_summary = (

    s4_knn_df

    .groupby(
        'dataset'
    )

    .agg(

        F1_min=(
            'F1',
            'min'
        ),

        F1_max=(
            'F1',
            'max'
        ),

        PR_AUC_min=(
            'PR_AUC',
            'min'
        ),

        PR_AUC_max=(
            'PR_AUC',
            'max'
        ),

        FAR_min=(
            'FAR',
            'min'
        ),

        FAR_max=(
            'FAR',
            'max'
        ),

        Alert_min=(
            'Alert_Rate',
            'min'
        ),

        Alert_max=(
            'Alert_Rate',
            'max'
        ),

        Threshold_min=(
            'calibrated_threshold',
            'min'
        ),

        Threshold_max=(
            'calibrated_threshold',
            'max'
        )
    )

    .reset_index()
)


s4_knn_summary[
    'F1_range'
] = (
    s4_knn_summary[
        'F1_max'
    ]
    -
    s4_knn_summary[
        'F1_min'
    ]
)


s4_knn_summary[
    'PR_AUC_range'
] = (
    s4_knn_summary[
        'PR_AUC_max'
    ]
    -
    s4_knn_summary[
        'PR_AUC_min'
    ]
)


s4_knn_summary[
    'FAR_range'
] = (
    s4_knn_summary[
        'FAR_max'
    ]
    -
    s4_knn_summary[
        'FAR_min'
    ]
)


s4_knn_summary[
    'Alert_range'
] = (
    s4_knn_summary[
        'Alert_max'
    ]
    -
    s4_knn_summary[
        'Alert_min'
    ]
)


print(
    "\n=== S4B VOLATILITY SUMMARY ==="
)


display(
    s4_knn_summary
)


s4_knn_summary.to_csv(
    OUT
    / 'S4B_knn_distance_volatility_summary.csv',
    index=False
)


print(
    "\n✅ S4B selesai."
)

print(
    "Kirim dua file:"
)

print(
    OUT
    / 'S4B_knn_distance_parameter_sensitivity.csv'
)

print(
    OUT
    / 'S4B_knn_distance_volatility_summary.csv'
)


S4B kNN-DISTANCE SENSITIVITY — CICIDS2018
Reference=15,000 | Calibration=10,000 | Test=66,146
Shared exact feature groups reference/calibration: 0

Fitting nearest-neighbor reference...
Computing calibration distances...
Computing held-out test distances...

--- k=5 ---
Threshold=0.088028 | F1=0.001278 | Precision=0.002277 | Recall=0.000888 | PR-AUC=0.090746 | FAR=0.061341 | Alert=0.053110

--- k=10 ---
Threshold=0.130359 | F1=0.006255 | Precision=0.011259 | Recall=0.004330 | PR-AUC=0.085869 | FAR=0.059940 | Alert=0.052369

--- k=20 ---
Threshold=0.198926 | F1=0.006254 | Precision=0.011249 | Recall=0.004330 | PR-AUC=0.082282 | FAR=0.059993 | Alert=0.052414

--- k=50 ---
Threshold=0.403117 | F1=0.006005 | Precision=0.010411 | Recall=0.004219 | PR-AUC=0.079546 | FAR=0.063213 | Alert=0.055181

S4B kNN-DISTANCE SENSITIVITY — HDFS
Reference=15,000 | Calibration=10,000 | Test=115,033
Shared exact feature groups reference/calibration: 0

Fitting nearest-neighbor reference...
Computing calibr

,dataset,scenario,model,parameter,parameter_value,threshold_quantile,calibrated_threshold,reference_n,calibration_n,test_n,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S4_parameter_sensitivity,kNN-distance,k,5,0.95,0.088028,15000,10000,66146,...,0.002277,0.000888,0.001278,0.090746,0.061341,0.053110,53635,3505,8998,8
1,CICIDS2018,S4_parameter_sensitivity,kNN-distance,k,10,0.95,0.130359,15000,10000,66146,...,0.011259,0.004330,0.006255,0.085869,0.059940,0.052369,53715,3425,8967,39
2,CICIDS2018,S4_parameter_sensitivity,kNN-distance,k,20,0.95,0.198926,15000,10000,66146,...,0.011249,0.004330,0.006254,0.082282,0.059993,0.052414,53712,3428,8967,39
3,CICIDS2018,S4_parameter_sensitivity,kNN-distance,k,50,0.95,0.403117,15000,10000,66146,...,0.010411,0.004219,0.006005,0.079546,0.063213,0.055181,53528,3612,8968,38
4,HDFS,S4_parameter_sensitivity,kNN-distance,k,5,0.95,0.046654,15000,10000,115033,...,0.269996,0.441551,0.335093,0.346024,0.036129,0.048038,107620,4034,1887,1492
5,HDFS,S4_parameter_sensitivity,kNN-distance,k,10,0.95,0.060015,15000,10000,115033,...,0.285389,0.482687,0.358698,0.360539,0.036577,0.049681,107570,4084,1748,1631
6,HDFS,S4_parameter_sensitivity,kNN-distance,k,20,0.95,0.081790,15000,10000,115033,...,0.294764,0.508139,0.373099,0.401187,0.036792,0.050638,107546,4108,1662,1717
7,HDFS,S4_parameter_sensitivity,kNN-distance,k,50,0.95,0.134278,15000,10000,115033,...,0.279579,0.487422,0.355340,0.347012,0.038010,0.051211,107410,4244,1732,1647
8,BGL,S4_parameter_sensitivity,kNN-distance,k,5,0.95,0.026968,15000,9889,10011,...,0.070611,0.050272,0.058730,0.106104,0.052507,0.052342,8788,487,699,37
9,BGL,S4_parameter_sensitivity,kNN-distance,k,10,0.95,0.039940,15000,9889,10011,...,0.062378,0.043478,0.051241,0.103359,0.051860,0.051244,8794,481,704,32



=== S4B VOLATILITY SUMMARY ===


,dataset,F1_min,F1_max,PR_AUC_min,PR_AUC_max,FAR_min,FAR_max,Alert_min,Alert_max,Threshold_min,Threshold_max,F1_range,PR_AUC_range,FAR_range,Alert_range
0,BGL,0.051241,0.069455,0.103359,0.133310,0.051429,0.052507,0.051244,0.053042,0.026968,0.093093,0.018214,0.029951,0.001078,0.001798
1,CICIDS2018,0.001278,0.006255,0.079546,0.090746,0.059940,0.063213,0.052369,0.055181,0.088028,0.403117,0.004977,0.011200,0.003273,0.002812
2,HDFS,0.335093,0.373099,0.346024,0.401187,0.036129,0.038010,0.048038,0.051211,0.046654,0.134278,0.038006,0.055163,0.001881,0.003173
3,UNSW-NB15,0.240261,0.409225,0.126917,0.227336,0.034885,0.040851,0.048082,0.050631,0.414912,0.631205,0.168964,0.100419,0.005967,0.002549



✅ S4B selesai.
Kirim dua file:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S4B_knn_distance_parameter_sensitivity.csv
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S4B_knn_distance_volatility_summary.csv


In [10]:
# ============================================================
# S4C-1 — DBSCAN EPSILON SENSITIVITY
#
# min_samples fixed = 10
# epsilon multiplier = 0.75, 1.00, 1.25
#
# Held-out inductive DBSCAN proxy:
# score = nearest distance to TRAINING DBSCAN core point
# threshold = 95th percentile TRAIN calibration score
#
# NO TEST-BASED CALIBRATION
# ============================================================

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupShuffleSplit

import pandas as pd
import numpy as np
import time
import gc


# ============================================================
# CONFIG
# ============================================================

DB_MIN_SAMPLES = 10

EPS_MULTIPLIERS = [
    0.75,
    1.00,
    1.25
]

REF_MAX = 15000
CAL_MAX = 10000

THRESHOLD_Q = 0.95
EPS_Q = 0.95

S4_DB_EPS_RESULTS = []


# ============================================================
# HELPER
# ============================================================

def capped_sample_s4c(
    idx,
    max_n,
    seed
):

    idx = np.asarray(
        idx,
        dtype=int
    )

    if len(idx) <= max_n:
        return idx

    rng = np.random.RandomState(
        seed
    )

    return np.sort(
        rng.choice(
            idx,
            size=max_n,
            replace=False
        )
    )


# ============================================================
# DATASET LOOP
# ============================================================

for name, (X, y) in DATASETS.items():

    print(
        f"\n{'='*78}"
    )

    print(
        f"S4C-1 DBSCAN EPS SENSITIVITY — {name}"
    )

    print(
        '='*78
    )


    # --------------------------------------------------------
    # SAME S0 TRAIN / TEST
    # --------------------------------------------------------

    tr, te = SPLITS[
        name
    ]

    Xtr_raw = (
        X.iloc[
            tr
        ]
        .reset_index(
            drop=True
        )
    )

    Xte_raw = (
        X.iloc[
            te
        ]
        .reset_index(
            drop=True
        )
    )

    ytr = np.asarray(
        y[tr],
        dtype=int
    )

    yte = np.asarray(
        y[te],
        dtype=int
    )


    # ========================================================
    # 1. GROUP-DISJOINT REFERENCE / CALIBRATION
    # ========================================================

    train_groups = (
        pd.util.hash_pandas_object(
            Xtr_raw,
            index=False
        )
        .astype(
            'uint64'
        )
        .values
    )


    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.25,
        random_state=CFG['SEED']
    )


    ref_rel, cal_rel = next(
        gss.split(
            Xtr_raw,
            groups=train_groups
        )
    )


    ref_idx = capped_sample_s4c(
        ref_rel,
        REF_MAX,
        CFG['SEED']
    )

    cal_idx = capped_sample_s4c(
        cal_rel,
        CAL_MAX,
        CFG['SEED'] + 1
    )


    ref_groups = np.unique(
        train_groups[
            ref_idx
        ]
    )

    cal_groups = np.unique(
        train_groups[
            cal_idx
        ]
    )


    shared_ref_cal = np.intersect1d(
        ref_groups,
        cal_groups
    )


    assert (
        len(
            shared_ref_cal
        ) == 0
    )


    print(
        f"Reference={len(ref_idx):,} | "
        f"Calibration={len(cal_idx):,} | "
        f"Test={len(te):,}"
    )

    print(
        "Shared reference/calibration groups:",
        len(
            shared_ref_cal
        )
    )


    # ========================================================
    # 2. PREPROCESS TRAIN ONLY
    # ========================================================

    pre_db_s4 = make_preprocessor(
        Xtr_raw
    )

    t0 = time.perf_counter()


    Xtr_p = pre_db_s4.fit_transform(
        Xtr_raw
    )

    Xte_p = pre_db_s4.transform(
        Xte_raw
    )


    prep_sec = (
        time.perf_counter()
        - t0
    )


    Xref = Xtr_p[
        ref_idx
    ]

    Xcal = Xtr_p[
        cal_idx
    ]


    # ========================================================
    # 3. DATA-DRIVEN BASE EPS
    #
    # Reference query contains SELF neighbor.
    #
    # For MinPts=10:
    # query 11 neighbors and take column index 10
    # = 10th NON-SELF neighbor.
    # ========================================================

    print(
        "\nEstimating base epsilon..."
    )


    nn_eps = NearestNeighbors(
        n_neighbors=DB_MIN_SAMPLES + 1,
        metric='euclidean',
        n_jobs=-1
    )


    nn_eps.fit(
        Xref
    )


    d_ref, _ = nn_eps.kneighbors(
        Xref,
        return_distance=True
    )


    kth_nonself = d_ref[
        :,
        DB_MIN_SAMPLES
    ]


    base_eps = float(
        np.quantile(
            kth_nonself,
            EPS_Q
        )
    )


    print(
        f"Base epsilon "
        f"(q={EPS_Q}, "
        f"MinPts={DB_MIN_SAMPLES}) = "
        f"{base_eps:.8f}"
    )


    del (
        nn_eps,
        d_ref,
        kth_nonself
    )

    gc.collect()


    # ========================================================
    # 4. EPSILON SWEEP
    # ========================================================

    for eps_mult in EPS_MULTIPLIERS:

        eps = (
            base_eps
            * eps_mult
        )


        print(
            f"\n--- eps multiplier="
            f"{eps_mult:.2f} "
            f"| eps={eps:.8f} ---"
        )


        # ----------------------------------------------------
        # FIT DBSCAN TRAIN REFERENCE ONLY
        # ----------------------------------------------------

        t1 = time.perf_counter()


        db = DBSCAN(
            eps=eps,
            min_samples=DB_MIN_SAMPLES,
            metric='euclidean',
            n_jobs=-1
        )


        db.fit(
            Xref
        )


        db_fit_sec = (
            time.perf_counter()
            - t1
        )


        # ----------------------------------------------------
        # CORE POINT AUDIT
        # ----------------------------------------------------

        core_idx = (
            db.core_sample_indices_
        )


        n_core = len(
            core_idx
        )


        n_noise_ref = int(
            np.sum(
                db.labels_ == -1
            )
        )


        n_clusters = len(
            set(
                db.labels_
            )
            - {-1}
        )


        print(
            f"clusters={n_clusters} | "
            f"core={n_core:,} | "
            f"reference noise={n_noise_ref:,}"
        )


        # ----------------------------------------------------
        # If no core points, result is invalid.
        # Do NOT silently fabricate scores.
        # ----------------------------------------------------

        if n_core == 0:

            S4_DB_EPS_RESULTS.append({

                'dataset':
                    name,

                'scenario':
                    'S4_parameter_sensitivity',

                'model':
                    'DBSCAN_inductive_proxy',

                'parameter':
                    'epsilon',

                'eps_multiplier':
                    eps_mult,

                'base_epsilon':
                    base_eps,

                'epsilon':
                    eps,

                'min_samples':
                    DB_MIN_SAMPLES,

                'reference_n':
                    len(
                        ref_idx
                    ),

                'calibration_n':
                    len(
                        cal_idx
                    ),

                'test_n':
                    len(
                        te
                    ),

                'shared_ref_cal_groups':
                    len(
                        shared_ref_cal
                    ),

                'n_clusters':
                    n_clusters,

                'n_core':
                    n_core,

                'reference_noise_n':
                    n_noise_ref,

                'valid_configuration':
                    False
            })

            print(
                "⚠️ No core points. "
                "Configuration marked INVALID."
            )

            continue


        Xcore = Xref[
            core_idx
        ]


        # ====================================================
        # 5. INDUCTIVE SCORE
        #
        # Held-out score =
        # nearest distance to TRAINING DBSCAN core
        # ====================================================

        core_nn = NearestNeighbors(
            n_neighbors=1,
            metric='euclidean',
            n_jobs=-1
        )


        core_nn.fit(
            Xcore
        )


        # ----------------------------------------------------
        # TRAINING CALIBRATION
        # ----------------------------------------------------

        t2 = time.perf_counter()


        cal_score = (
            core_nn
            .kneighbors(
                Xcal,
                return_distance=True
            )[0][:, 0]
        )


        cal_sec = (
            time.perf_counter()
            - t2
        )


        threshold = float(
            np.quantile(
                cal_score,
                THRESHOLD_Q
            )
        )


        # ----------------------------------------------------
        # HELD-OUT TEST
        # ----------------------------------------------------

        t3 = time.perf_counter()


        test_score = (
            core_nn
            .kneighbors(
                Xte_p,
                return_distance=True
            )[0][:, 0]
        )


        test_sec = (
            time.perf_counter()
            - t3
        )


        y_pred = (
            test_score
            > threshold
        ).astype(int)


        # ====================================================
        # 6. METRICS
        # ====================================================

        m = calc_metrics(
            yte,
            y_pred,
            test_score
        )


        S4_DB_EPS_RESULTS.append({

            'dataset':
                name,

            'scenario':
                'S4_parameter_sensitivity',

            'model':
                'DBSCAN_inductive_proxy',

            'parameter':
                'epsilon',

            'eps_multiplier':
                eps_mult,

            'base_epsilon':
                base_eps,

            'epsilon':
                eps,

            'min_samples':
                DB_MIN_SAMPLES,

            'reference_n':
                len(
                    ref_idx
                ),

            'calibration_n':
                len(
                    cal_idx
                ),

            'test_n':
                len(
                    te
                ),

            'shared_ref_cal_groups':
                len(
                    shared_ref_cal
                ),

            'n_clusters':
                n_clusters,

            'n_core':
                n_core,

            'reference_noise_n':
                n_noise_ref,

            'valid_configuration':
                True,

            'calibrated_threshold':
                threshold,

            'preprocessing_sec':
                prep_sec,

            'dbscan_fit_sec':
                db_fit_sec,

            'calibration_query_sec':
                cal_sec,

            'test_query_sec':
                test_sec,

            **m
        })


        print(
            f"Threshold={threshold:.6f} | "
            f"F1={m['F1']:.6f} | "
            f"P={m['Precision']:.6f} | "
            f"R={m['Recall']:.6f} | "
            f"PR-AUC={m['PR_AUC']:.6f} | "
            f"FAR={m['FAR']:.6f} | "
            f"Alert={m['Alert_Rate']:.6f}"
        )


        del (
            db,
            Xcore,
            core_nn,
            cal_score,
            test_score
        )

        gc.collect()


    # ========================================================
    # CLEAN DATASET MEMORY
    # ========================================================

    del (
        Xtr_p,
        Xte_p,
        Xref,
        Xcal
    )

    gc.collect()


# ============================================================
# 7. SAVE FULL RESULTS
# ============================================================

s4_db_eps_df = pd.DataFrame(
    S4_DB_EPS_RESULTS
)


display(
    s4_db_eps_df
)


s4_db_eps_df.to_csv(
    OUT
    / 'S4C1_dbscan_epsilon_sensitivity.csv',
    index=False
)


# ============================================================
# 8. VALID CONFIGURATION SUMMARY
# ============================================================

valid_db_eps = s4_db_eps_df[
    s4_db_eps_df[
        'valid_configuration'
    ] == True
].copy()


s4_db_eps_summary = (

    valid_db_eps

    .groupby(
        'dataset'
    )

    .agg(

        F1_min=(
            'F1',
            'min'
        ),

        F1_max=(
            'F1',
            'max'
        ),

        PR_AUC_min=(
            'PR_AUC',
            'min'
        ),

        PR_AUC_max=(
            'PR_AUC',
            'max'
        ),

        FAR_min=(
            'FAR',
            'min'
        ),

        FAR_max=(
            'FAR',
            'max'
        ),

        Alert_min=(
            'Alert_Rate',
            'min'
        ),

        Alert_max=(
            'Alert_Rate',
            'max'
        ),

        Core_min=(
            'n_core',
            'min'
        ),

        Core_max=(
            'n_core',
            'max'
        )
    )

    .reset_index()
)


s4_db_eps_summary[
    'F1_range'
] = (
    s4_db_eps_summary[
        'F1_max'
    ]
    -
    s4_db_eps_summary[
        'F1_min'
    ]
)


s4_db_eps_summary[
    'PR_AUC_range'
] = (
    s4_db_eps_summary[
        'PR_AUC_max'
    ]
    -
    s4_db_eps_summary[
        'PR_AUC_min'
    ]
)


s4_db_eps_summary[
    'FAR_range'
] = (
    s4_db_eps_summary[
        'FAR_max'
    ]
    -
    s4_db_eps_summary[
        'FAR_min'
    ]
)


s4_db_eps_summary[
    'Alert_range'
] = (
    s4_db_eps_summary[
        'Alert_max'
    ]
    -
    s4_db_eps_summary[
        'Alert_min'
    ]
)


print(
    "\n=== S4C-1 DBSCAN EPSILON VOLATILITY ==="
)


display(
    s4_db_eps_summary
)


s4_db_eps_summary.to_csv(
    OUT
    / 'S4C1_dbscan_epsilon_volatility_summary.csv',
    index=False
)


print(
    "\n✅ S4C-1 selesai."
)

print(
    "Kirim dua file:"
)

print(
    OUT
    / 'S4C1_dbscan_epsilon_sensitivity.csv'
)

print(
    OUT
    / 'S4C1_dbscan_epsilon_volatility_summary.csv'
)


S4C-1 DBSCAN EPS SENSITIVITY — CICIDS2018
Reference=15,000 | Calibration=10,000 | Test=66,146
Shared reference/calibration groups: 0

Estimating base epsilon...
Base epsilon (q=0.95, MinPts=10) = 0.14451730

--- eps multiplier=0.75 | eps=0.10838797 ---
clusters=46 | core=14,095 | reference noise=751
Threshold=0.105602 | F1=0.006286 | P=0.011464 | R=0.004330 | PR-AUC=0.105964 | FAR=0.058855 | Alert=0.051432

--- eps multiplier=1.00 | eps=0.14451730 ---
clusters=42 | core=14,293 | reference noise=595
Threshold=0.064304 | F1=0.006552 | P=0.011681 | R=0.004553 | PR-AUC=0.106040 | FAR=0.060711 | Alert=0.053064

--- eps multiplier=1.25 | eps=0.18064662 ---
clusters=43 | core=14,430 | reference noise=451
Threshold=0.051765 | F1=0.006559 | P=0.011731 | R=0.004553 | PR-AUC=0.106133 | FAR=0.060448 | Alert=0.052838

S4C-1 DBSCAN EPS SENSITIVITY — HDFS
Reference=15,000 | Calibration=10,000 | Test=115,033
Shared reference/calibration groups: 0

Estimating base epsilon...
Base epsilon (q=0.95, MinP

,dataset,scenario,model,parameter,eps_multiplier,base_epsilon,epsilon,min_samples,reference_n,calibration_n,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S4_parameter_sensitivity,DBSCAN_inductive_proxy,epsilon,0.75,0.144517,0.108388,10,15000,10000,...,0.011464,0.004330,0.006286,0.105964,0.058855,0.051432,53777,3363,8967,39
1,CICIDS2018,S4_parameter_sensitivity,DBSCAN_inductive_proxy,epsilon,1.00,0.144517,0.144517,10,15000,10000,...,0.011681,0.004553,0.006552,0.106040,0.060711,0.053064,53671,3469,8965,41
2,CICIDS2018,S4_parameter_sensitivity,DBSCAN_inductive_proxy,epsilon,1.25,0.144517,0.180647,10,15000,10000,...,0.011731,0.004553,0.006559,0.106133,0.060448,0.052838,53686,3454,8965,41
3,HDFS,S4_parameter_sensitivity,DBSCAN_inductive_proxy,epsilon,0.75,0.059973,0.044980,10,15000,10000,...,0.280291,0.478544,0.353520,0.363705,0.037186,0.050151,107502,4152,1762,1617
4,HDFS,S4_parameter_sensitivity,DBSCAN_inductive_proxy,epsilon,1.00,0.059973,0.059973,10,15000,10000,...,0.288612,0.480024,0.360484,0.357383,0.035807,0.048856,107656,3998,1757,1622
5,HDFS,S4_parameter_sensitivity,DBSCAN_inductive_proxy,epsilon,1.25,0.059973,0.074967,10,15000,10000,...,0.265547,0.453685,0.335009,0.328131,0.037974,0.050186,107414,4240,1846,1533
6,BGL,S4_parameter_sensitivity,DBSCAN_inductive_proxy,epsilon,0.75,0.039039,0.029279,10,15000,9889,...,0.058824,0.042120,0.049089,0.091445,0.053477,0.052642,8779,496,705,31
7,BGL,S4_parameter_sensitivity,DBSCAN_inductive_proxy,epsilon,1.00,0.039039,0.039039,10,15000,9889,...,0.057514,0.042120,0.048627,0.090364,0.054771,0.053841,8767,508,705,31
8,BGL,S4_parameter_sensitivity,DBSCAN_inductive_proxy,epsilon,1.25,0.039039,0.048799,10,15000,9889,...,0.044355,0.029891,0.035714,0.089223,0.051105,0.049545,8801,474,714,22
9,UNSW-NB15,S4_parameter_sensitivity,DBSCAN_inductive_proxy,epsilon,0.75,0.466372,0.349779,10,15000,10000,...,0.236597,0.360486,0.285689,0.160364,0.038123,0.048353,130420,5169,2842,1602



=== S4C-1 DBSCAN EPSILON VOLATILITY ===


,dataset,F1_min,F1_max,PR_AUC_min,PR_AUC_max,FAR_min,FAR_max,Alert_min,Alert_max,Core_min,Core_max,F1_range,PR_AUC_range,FAR_range,Alert_range
0,BGL,0.035714,0.049089,0.089223,0.091445,0.051105,0.054771,0.049545,0.053841,13826,14521,0.013375,0.002222,0.003666,0.004295
1,CICIDS2018,0.006286,0.006559,0.105964,0.106133,0.058855,0.060711,0.051432,0.053064,14095,14430,0.000273,0.000170,0.001855,0.001633
2,HDFS,0.335009,0.360484,0.328131,0.363705,0.035807,0.037974,0.048856,0.050186,13836,14532,0.025476,0.035574,0.002167,0.001330
3,UNSW-NB15,0.212280,0.285689,0.117789,0.160364,0.038123,0.041633,0.048353,0.049188,13093,14683,0.073409,0.042575,0.003511,0.000836



✅ S4C-1 selesai.
Kirim dua file:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S4C1_dbscan_epsilon_sensitivity.csv
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S4C1_dbscan_epsilon_volatility_summary.csv


In [11]:
# ============================================================
# S4C-2 — DBSCAN MinPts SENSITIVITY
#
# epsilon FIXED
# min_samples / MinPts = 5, 10, 20, 50
#
# Held-out inductive DBSCAN proxy:
# score = nearest distance to TRAINING DBSCAN core point
# threshold = 95th percentile TRAIN calibration score
#
# ============================================================

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupShuffleSplit

import pandas as pd
import numpy as np
import time
import gc


# ============================================================
# HARD ENVIRONMENT CHECK
# ============================================================

required = [
    'DATASETS',
    'SPLITS',
    'CFG',
    'OUT',
    'make_preprocessor',
    'calc_metrics'
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Object belum tersedia: "
        + ", ".join(missing)
        + "\nJalankan bootstrap/resume terlebih dahulu."
    )


# ============================================================
# CONFIG
# ============================================================

MINPTS_VALUES = [
    5,
    10,
    20,
    50
]

BASE_K_FOR_EPS = 10

REF_MAX = 15000
CAL_MAX = 10000

EPS_Q = 0.95
THRESHOLD_Q = 0.95

S4_DB_MINPTS_RESULTS = []


# ============================================================
# LOCAL HELPER
# ============================================================

def capped_sample_s4c2(
    idx,
    max_n,
    seed
):
    idx = np.asarray(
        idx,
        dtype=int
    )

    if len(idx) <= max_n:
        return idx

    rng = np.random.RandomState(
        seed
    )

    return np.sort(
        rng.choice(
            idx,
            size=max_n,
            replace=False
        )
    )


# ============================================================
# DATASET LOOP
# ============================================================

for name, (X, y) in DATASETS.items():

    print(
        f"\n{'='*78}"
    )

    print(
        f"S4C-2 DBSCAN MinPts SENSITIVITY — {name}"
    )

    print(
        '='*78
    )


    # --------------------------------------------------------
    # SAME S0 SPLIT
    # --------------------------------------------------------

    tr, te = SPLITS[name]

    Xtr_raw = (
        X.iloc[tr]
        .reset_index(drop=True)
    )

    Xte_raw = (
        X.iloc[te]
        .reset_index(drop=True)
    )

    ytr = np.asarray(
        y[tr],
        dtype=int
    )

    yte = np.asarray(
        y[te],
        dtype=int
    )


    # ========================================================
    # 1. GROUP-DISJOINT REFERENCE / CALIBRATION
    # ========================================================

    train_groups = (
        pd.util.hash_pandas_object(
            Xtr_raw,
            index=False
        )
        .astype('uint64')
        .values
    )


    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.25,
        random_state=CFG['SEED']
    )


    ref_rel, cal_rel = next(
        gss.split(
            Xtr_raw,
            groups=train_groups
        )
    )


    ref_idx = capped_sample_s4c2(
        ref_rel,
        REF_MAX,
        CFG['SEED']
    )

    cal_idx = capped_sample_s4c2(
        cal_rel,
        CAL_MAX,
        CFG['SEED'] + 1
    )


    ref_groups = np.unique(
        train_groups[ref_idx]
    )

    cal_groups = np.unique(
        train_groups[cal_idx]
    )


    shared_ref_cal = np.intersect1d(
        ref_groups,
        cal_groups
    )


    assert len(shared_ref_cal) == 0


    print(
        f"Reference={len(ref_idx):,} | "
        f"Calibration={len(cal_idx):,} | "
        f"Test={len(te):,}"
    )

    print(
        "Shared reference/calibration groups:",
        len(shared_ref_cal)
    )


    # ========================================================
    # 2. PREPROCESSING FIT TRAIN ONLY
    # ========================================================

    pre_db_s4c2 = make_preprocessor(
        Xtr_raw
    )

    t0 = time.perf_counter()

    Xtr_p = pre_db_s4c2.fit_transform(
        Xtr_raw
    )

    Xte_p = pre_db_s4c2.transform(
        Xte_raw
    )

    prep_sec = (
        time.perf_counter()
        - t0
    )


    Xref = Xtr_p[
        ref_idx
    ]

    Xcal = Xtr_p[
        cal_idx
    ]


    # ========================================================
    # 3. COMPUTE FIXED BASE EPSILON
    #
    # SAME DEFINITION AS S4C-1:
    # 10th NON-SELF neighbor,
    # 95th percentile.
    # ========================================================

    print(
        "\nEstimating FIXED epsilon..."
    )


    nn_eps = NearestNeighbors(
        n_neighbors=BASE_K_FOR_EPS + 1,
        metric='euclidean',
        n_jobs=-1
    )

    nn_eps.fit(
        Xref
    )


    d_ref, _ = nn_eps.kneighbors(
        Xref,
        return_distance=True
    )


    kth_nonself = d_ref[
        :,
        BASE_K_FOR_EPS
    ]


    fixed_eps = float(
        np.quantile(
            kth_nonself,
            EPS_Q
        )
    )


    print(
        f"Fixed epsilon = "
        f"{fixed_eps:.8f}"
    )


    del (
        nn_eps,
        d_ref,
        kth_nonself
    )

    gc.collect()


    # ========================================================
    # 4. MinPts SWEEP
    # ========================================================

    for minpts in MINPTS_VALUES:

        print(
            f"\n--- MinPts={minpts} | "
            f"eps={fixed_eps:.8f} ---"
        )


        # ----------------------------------------------------
        # DBSCAN FIT REFERENCE ONLY
        # ----------------------------------------------------

        t1 = time.perf_counter()


        db = DBSCAN(
            eps=fixed_eps,
            min_samples=minpts,
            metric='euclidean',
            n_jobs=-1
        )


        db.fit(
            Xref
        )


        db_fit_sec = (
            time.perf_counter()
            - t1
        )


        # ----------------------------------------------------
        # STRUCTURAL AUDIT
        # ----------------------------------------------------

        core_idx = (
            db.core_sample_indices_
        )

        n_core = len(
            core_idx
        )

        n_noise_ref = int(
            np.sum(
                db.labels_ == -1
            )
        )

        n_clusters = len(
            set(
                db.labels_
            ) - {-1}
        )


        print(
            f"clusters={n_clusters} | "
            f"core={n_core:,} | "
            f"reference noise={n_noise_ref:,}"
        )


        # ----------------------------------------------------
        # INVALID CONFIGURATION SAFETY
        # ----------------------------------------------------

        if n_core == 0:

            S4_DB_MINPTS_RESULTS.append({

                'dataset':
                    name,

                'scenario':
                    'S4_parameter_sensitivity',

                'model':
                    'DBSCAN_inductive_proxy',

                'parameter':
                    'min_samples',

                'min_samples':
                    minpts,

                'fixed_epsilon':
                    fixed_eps,

                'reference_n':
                    len(ref_idx),

                'calibration_n':
                    len(cal_idx),

                'test_n':
                    len(te),

                'shared_ref_cal_groups':
                    len(shared_ref_cal),

                'n_clusters':
                    n_clusters,

                'n_core':
                    n_core,

                'reference_noise_n':
                    n_noise_ref,

                'valid_configuration':
                    False
            })


            print(
                "⚠️ No core points — INVALID configuration."
            )

            continue


        # ====================================================
        # 5. INDUCTIVE NEAREST-CORE SCORE
        # ====================================================

        Xcore = Xref[
            core_idx
        ]


        core_nn = NearestNeighbors(
            n_neighbors=1,
            metric='euclidean',
            n_jobs=-1
        )


        core_nn.fit(
            Xcore
        )


        # ----------------------------------------------------
        # TRAIN CALIBRATION
        # ----------------------------------------------------

        t2 = time.perf_counter()


        cal_score = (
            core_nn
            .kneighbors(
                Xcal,
                return_distance=True
            )[0][:, 0]
        )


        cal_sec = (
            time.perf_counter()
            - t2
        )


        threshold = float(
            np.quantile(
                cal_score,
                THRESHOLD_Q
            )
        )


        # ----------------------------------------------------
        # TEST
        # ----------------------------------------------------

        t3 = time.perf_counter()


        test_score = (
            core_nn
            .kneighbors(
                Xte_p,
                return_distance=True
            )[0][:, 0]
        )


        test_sec = (
            time.perf_counter()
            - t3
        )


        y_pred = (
            test_score
            > threshold
        ).astype(int)


        # ====================================================
        # 6. METRICS
        # ====================================================

        m = calc_metrics(
            yte,
            y_pred,
            test_score
        )


        S4_DB_MINPTS_RESULTS.append({

            'dataset':
                name,

            'scenario':
                'S4_parameter_sensitivity',

            'model':
                'DBSCAN_inductive_proxy',

            'parameter':
                'min_samples',

            'min_samples':
                minpts,

            'fixed_epsilon':
                fixed_eps,

            'reference_n':
                len(ref_idx),

            'calibration_n':
                len(cal_idx),

            'test_n':
                len(te),

            'shared_ref_cal_groups':
                len(shared_ref_cal),

            'n_clusters':
                n_clusters,

            'n_core':
                n_core,

            'reference_noise_n':
                n_noise_ref,

            'valid_configuration':
                True,

            'calibrated_threshold':
                threshold,

            'preprocessing_sec':
                prep_sec,

            'dbscan_fit_sec':
                db_fit_sec,

            'calibration_query_sec':
                cal_sec,

            'test_query_sec':
                test_sec,

            **m
        })


        print(
            f"Threshold={threshold:.6f} | "
            f"F1={m['F1']:.6f} | "
            f"P={m['Precision']:.6f} | "
            f"R={m['Recall']:.6f} | "
            f"PR-AUC={m['PR_AUC']:.6f} | "
            f"FAR={m['FAR']:.6f} | "
            f"Alert={m['Alert_Rate']:.6f}"
        )


        del (
            db,
            Xcore,
            core_nn,
            cal_score,
            test_score
        )

        gc.collect()


    # ========================================================
    # CLEAN DATASET MEMORY
    # ========================================================

    del (
        Xtr_p,
        Xte_p,
        Xref,
        Xcal
    )

    gc.collect()


# ============================================================
# 7. SAVE FULL RESULTS
# ============================================================

s4_db_minpts_df = pd.DataFrame(
    S4_DB_MINPTS_RESULTS
)


display(
    s4_db_minpts_df
)


s4_db_minpts_df.to_csv(
    OUT
    / 'S4C2_dbscan_minpts_sensitivity.csv',
    index=False
)


# ============================================================
# 8. VOLATILITY SUMMARY
# ============================================================

valid_minpts = s4_db_minpts_df[
    s4_db_minpts_df[
        'valid_configuration'
    ] == True
].copy()


s4_db_minpts_summary = (

    valid_minpts

    .groupby(
        'dataset'
    )

    .agg(

        F1_min=(
            'F1',
            'min'
        ),

        F1_max=(
            'F1',
            'max'
        ),

        PR_AUC_min=(
            'PR_AUC',
            'min'
        ),

        PR_AUC_max=(
            'PR_AUC',
            'max'
        ),

        FAR_min=(
            'FAR',
            'min'
        ),

        FAR_max=(
            'FAR',
            'max'
        ),

        Alert_min=(
            'Alert_Rate',
            'min'
        ),

        Alert_max=(
            'Alert_Rate',
            'max'
        ),

        Core_min=(
            'n_core',
            'min'
        ),

        Core_max=(
            'n_core',
            'max'
        ),

        Cluster_min=(
            'n_clusters',
            'min'
        ),

        Cluster_max=(
            'n_clusters',
            'max'
        )
    )

    .reset_index()
)


s4_db_minpts_summary[
    'F1_range'
] = (
    s4_db_minpts_summary[
        'F1_max'
    ]
    -
    s4_db_minpts_summary[
        'F1_min'
    ]
)


s4_db_minpts_summary[
    'PR_AUC_range'
] = (
    s4_db_minpts_summary[
        'PR_AUC_max'
    ]
    -
    s4_db_minpts_summary[
        'PR_AUC_min'
    ]
)


s4_db_minpts_summary[
    'FAR_range'
] = (
    s4_db_minpts_summary[
        'FAR_max'
    ]
    -
    s4_db_minpts_summary[
        'FAR_min'
    ]
)


s4_db_minpts_summary[
    'Alert_range'
] = (
    s4_db_minpts_summary[
        'Alert_max'
    ]
    -
    s4_db_minpts_summary[
        'Alert_min'
    ]
)


print(
    "\n=== S4C-2 DBSCAN MinPts VOLATILITY ==="
)


display(
    s4_db_minpts_summary
)


s4_db_minpts_summary.to_csv(
    OUT
    / 'S4C2_dbscan_minpts_volatility_summary.csv',
    index=False
)


print(
    "\n✅ S4C-2 selesai."
)

print(
    "Kirim dua file:"
)

print(
    OUT
    / 'S4C2_dbscan_minpts_sensitivity.csv'
)

print(
    OUT
    / 'S4C2_dbscan_minpts_volatility_summary.csv'
)


S4C-2 DBSCAN MinPts SENSITIVITY — CICIDS2018
Reference=15,000 | Calibration=10,000 | Test=66,146
Shared reference/calibration groups: 0

Estimating FIXED epsilon...
Fixed epsilon = 0.14451730

--- MinPts=5 | eps=0.14451730 ---
clusters=64 | core=14,551 | reference noise=363
Threshold=0.044114 | F1=0.001281 | P=0.002296 | R=0.000888 | PR-AUC=0.106218 | FAR=0.060851 | Alert=0.052686

--- MinPts=10 | eps=0.14451730 ---
clusters=42 | core=14,293 | reference noise=595
Threshold=0.064304 | F1=0.006552 | P=0.011681 | R=0.004553 | PR-AUC=0.106040 | FAR=0.060711 | Alert=0.053064

--- MinPts=20 | eps=0.14451730 ---
clusters=32 | core=13,971 | reference noise=849
Threshold=0.173685 | F1=0.006222 | P=0.011048 | R=0.004330 | PR-AUC=0.105625 | FAR=0.061096 | Alert=0.053367

--- MinPts=50 | eps=0.14451730 ---
clusters=23 | core=13,187 | reference noise=1,391
Threshold=0.473258 | F1=0.006035 | P=0.010591 | R=0.004219 | PR-AUC=0.105134 | FAR=0.062128 | Alert=0.054244

S4C-2 DBSCAN MinPts SENSITIVITY —

,dataset,scenario,model,parameter,min_samples,fixed_epsilon,reference_n,calibration_n,test_n,shared_ref_cal_groups,...,Precision,Recall,F1,PR_AUC,FAR,Alert_Rate,TN,FP,FN,TP
0,CICIDS2018,S4_parameter_sensitivity,DBSCAN_inductive_proxy,min_samples,5,0.144517,15000,10000,66146,0,...,0.002296,0.000888,0.001281,0.106218,0.060851,0.052686,53663,3477,8998,8
1,CICIDS2018,S4_parameter_sensitivity,DBSCAN_inductive_proxy,min_samples,10,0.144517,15000,10000,66146,0,...,0.011681,0.004553,0.006552,0.106040,0.060711,0.053064,53671,3469,8965,41
2,CICIDS2018,S4_parameter_sensitivity,DBSCAN_inductive_proxy,min_samples,20,0.144517,15000,10000,66146,0,...,0.011048,0.004330,0.006222,0.105625,0.061096,0.053367,53649,3491,8967,39
3,CICIDS2018,S4_parameter_sensitivity,DBSCAN_inductive_proxy,min_samples,50,0.144517,15000,10000,66146,0,...,0.010591,0.004219,0.006035,0.105134,0.062128,0.054244,53590,3550,8968,38
4,HDFS,S4_parameter_sensitivity,DBSCAN_inductive_proxy,min_samples,5,0.059973,15000,10000,115033,0,...,0.253135,0.418171,0.315367,0.333787,0.037339,0.048525,107485,4169,1966,1413
5,HDFS,S4_parameter_sensitivity,DBSCAN_inductive_proxy,min_samples,10,0.059973,15000,10000,115033,0,...,0.288612,0.480024,0.360484,0.357383,0.035807,0.048856,107656,3998,1757,1622
6,HDFS,S4_parameter_sensitivity,DBSCAN_inductive_proxy,min_samples,20,0.059973,15000,10000,115033,0,...,0.279652,0.475880,0.352284,0.289110,0.037097,0.049986,107512,4142,1771,1608
7,HDFS,S4_parameter_sensitivity,DBSCAN_inductive_proxy,min_samples,50,0.059973,15000,10000,115033,0,...,0.318737,0.555786,0.405134,0.271514,0.035950,0.051220,107640,4014,1501,1878
8,BGL,S4_parameter_sensitivity,DBSCAN_inductive_proxy,min_samples,5,0.039039,15000,9889,10011,0,...,0.072549,0.050272,0.059390,0.090357,0.050997,0.050944,8802,473,699,37
9,BGL,S4_parameter_sensitivity,DBSCAN_inductive_proxy,min_samples,10,0.039039,15000,9889,10011,0,...,0.057514,0.042120,0.048627,0.090364,0.054771,0.053841,8767,508,705,31



=== S4C-2 DBSCAN MinPts VOLATILITY ===


,dataset,F1_min,F1_max,PR_AUC_min,PR_AUC_max,FAR_min,FAR_max,Alert_min,Alert_max,Core_min,Core_max,Cluster_min,Cluster_max,F1_range,PR_AUC_range,FAR_range,Alert_range
0,BGL,0.000000,0.079536,0.077604,0.095936,0.045606,0.054771,0.045550,0.053841,11826,14667,5,27,0.079536,0.018332,0.009164,0.008291
1,CICIDS2018,0.001281,0.006552,0.105134,0.106218,0.060711,0.062128,0.052686,0.054244,13187,14551,23,64,0.005271,0.001084,0.001418,0.001557
2,HDFS,0.315367,0.405134,0.271514,0.357383,0.035807,0.037339,0.048525,0.051220,12462,14609,21,62,0.089768,0.085869,0.001532,0.002695
3,UNSW-NB15,0.214203,0.416560,0.115957,0.261051,0.035490,0.041316,0.047682,0.051752,12812,14552,8,11,0.202357,0.145094,0.005826,0.004070



✅ S4C-2 selesai.
Kirim dua file:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S4C2_dbscan_minpts_sensitivity.csv
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S4C2_dbscan_minpts_volatility_summary.csv


In [12]:
# ============================================================
# W2A — XGBOOST SHAP EXPLANATION STABILITY
# Clean vs Noise/Missing
# Multi-dataset confirmatory evaluation
#
# Metrics:
# - FSS Top-10 overlap
# - Spearman ranking stability
# - Magnitude similarity (1 - total variation distance)
#
# TreeSHAP via XGBoost pred_contribs
# ============================================================

import numpy as np
import pandas as pd
import time
import gc

import xgboost as xgb

from xgboost import XGBClassifier
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split


# ============================================================
# HARD CHECK
# ============================================================

required = [
    'DATASETS',
    'SPLITS',
    'CFG',
    'OUT',
    'make_preprocessor'
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Object belum tersedia: "
        + ", ".join(missing)
        + "\nJalankan bootstrap/resume terlebih dahulu."
    )


# ============================================================
# CONFIG
# ============================================================

W2_SAMPLE_MAX = 3000

W2_SEEDS = [
    CFG['SEED'],
    CFG['SEED'] + 1,
    CFG['SEED'] + 2
]

W2_CONDITIONS = [
    ('noise', 0.01),
    ('noise', 0.05),
    ('missing', 0.01),
    ('missing', 0.05)
]

TOP_K = 10

W2_RESULTS = []
W2_CLEAN_IMPORTANCE = []


# ============================================================
# HELPERS
# ============================================================

def get_fixed_stratified_sample(
    y,
    max_n,
    seed
):
    y = np.asarray(
        y,
        dtype=int
    )

    idx = np.arange(
        len(y)
    )

    if len(idx) <= max_n:
        return idx

    keep, _ = train_test_split(
        idx,
        train_size=max_n,
        random_state=seed,
        stratify=y
    )

    return np.sort(
        keep
    )


def tree_shap_values(
    model,
    X
):
    dm = xgb.DMatrix(
        X
    )

    contrib = (
        model
        .get_booster()
        .predict(
            dm,
            pred_contribs=True
        )
    )

    # last column = expected value / bias
    return contrib[:, :-1]


def global_shap_importance(
    shap_values
):
    return np.mean(
        np.abs(
            shap_values
        ),
        axis=0
    )


def stability_metrics(
    base_imp,
    pert_imp,
    k=10
):

    base_imp = np.asarray(
        base_imp,
        dtype=float
    )

    pert_imp = np.asarray(
        pert_imp,
        dtype=float
    )

    k = min(
        k,
        len(base_imp)
    )

    # --------------------------------------------------------
    # Top-K overlap / FSS
    # --------------------------------------------------------

    top_base = np.argsort(
        base_imp
    )[-k:]

    top_pert = np.argsort(
        pert_imp
    )[-k:]

    overlap = len(
        set(top_base)
        .intersection(
            set(top_pert)
        )
    )

    fss = (
        overlap / k
    )


    # --------------------------------------------------------
    # Rank stability
    # --------------------------------------------------------

    rho = spearmanr(
        base_imp,
        pert_imp
    ).statistic

    if np.isnan(rho):
        rho = 0.0


    # --------------------------------------------------------
    # Magnitude similarity
    #
    # normalize importance to distributions
    # similarity = 1 - total variation distance
    # range 0..1
    # --------------------------------------------------------

    p = base_imp / max(
        base_imp.sum(),
        1e-12
    )

    q = pert_imp / max(
        pert_imp.sum(),
        1e-12
    )

    tvd = (
        0.5
        * np.sum(
            np.abs(
                p - q
            )
        )
    )

    magnitude_similarity = (
        1.0 - tvd
    )


    return {
        'FSS_Top10':
            float(fss),

        'Top10_overlap_n':
            int(overlap),

        'Spearman':
            float(rho),

        'Magnitude_similarity':
            float(
                magnitude_similarity
            )
    }


# ============================================================
# DATASET LOOP
# ============================================================

for name, (X, y) in DATASETS.items():

    print(
        f"\n{'='*78}"
    )

    print(
        f"W2A SHAP STABILITY — {name}"
    )

    print(
        '='*78
    )


    # --------------------------------------------------------
    # SAME S0 SPLIT
    # --------------------------------------------------------

    tr, te = SPLITS[
        name
    ]

    Xtr_raw = (
        X.iloc[
            tr
        ]
        .reset_index(
            drop=True
        )
    )

    Xte_raw = (
        X.iloc[
            te
        ]
        .reset_index(
            drop=True
        )
    )

    ytr = np.asarray(
        y[tr],
        dtype=int
    )

    yte = np.asarray(
        y[te],
        dtype=int
    )


    # ========================================================
    # 1. PREPROCESS FIT TRAIN ONLY
    # ========================================================

    pre = make_preprocessor(
        Xtr_raw
    )

    Xtr_p = pre.fit_transform(
        Xtr_raw
    )

    Xte_clean = pre.transform(
        Xte_raw
    )


    # ========================================================
    # 2. FIXED TEST SAMPLE
    # ========================================================

    sample_idx = get_fixed_stratified_sample(
        yte,
        W2_SAMPLE_MAX,
        CFG['SEED']
    )

    Xsample_raw = (
        Xte_raw.iloc[
            sample_idx
        ]
        .reset_index(
            drop=True
        )
    )

    Xsample_clean = (
        Xte_clean[
            sample_idx
        ].copy()
    )


    print(
        f"Train={len(Xtr_raw):,} | "
        f"Test={len(Xte_raw):,} | "
        f"SHAP sample={len(sample_idx):,}"
    )


    # ========================================================
    # 3. TRAIN XGBOOST CLEAN
    # ========================================================

    model = XGBClassifier(
        random_state=CFG['SEED'],
        n_jobs=-1,
        eval_metric='logloss',
        tree_method='hist',
        verbosity=0
    )

    model.fit(
        Xtr_p,
        ytr
    )


    # ========================================================
    # 4. CLEAN SHAP BASELINE
    # ========================================================

    t0 = time.perf_counter()

    shap_clean = tree_shap_values(
        model,
        Xsample_clean
    )

    clean_sec = (
        time.perf_counter()
        - t0
    )

    imp_clean = global_shap_importance(
        shap_clean
    )


    # --------------------------------------------------------
    # feature names
    # --------------------------------------------------------

    try:

        feature_names = (
            pre
            .named_steps[
                'columns'
            ]
            .get_feature_names_out()
        )

    except Exception:

        feature_names = np.array([
            f'f{i}'
            for i in range(
                Xsample_clean.shape[1]
            )
        ])


    clean_rank = np.argsort(
        imp_clean
    )[::-1]


    for rank_pos, j in enumerate(
        clean_rank,
        start=1
    ):

        W2_CLEAN_IMPORTANCE.append({

            'dataset':
                name,

            'rank':
                rank_pos,

            'feature':
                str(
                    feature_names[j]
                ),

            'mean_abs_SHAP':
                float(
                    imp_clean[j]
                )
        })


    print(
        "\nClean Top-10:"
    )

    for j in clean_rank[
        :min(
            TOP_K,
            len(clean_rank)
        )
    ]:

        print(
            f"  {feature_names[j]} "
            f"= {imp_clean[j]:.6f}"
        )


    # ========================================================
    # PREPARE NUMERIC COLUMN POSITIONS
    #
    # ColumnTransformer outputs numeric columns first.
    # ========================================================

    numeric_cols = list(
        Xtr_raw
        .select_dtypes(
            include=np.number
        )
        .columns
    )

    numeric_positions = []

    for pos, col in enumerate(
        numeric_cols
    ):

        # Preserve prior S2 treatment:
        # do not perturb node hash directly
        if str(col).lower() == 'node_h':
            continue

        numeric_positions.append(
            pos
        )

    numeric_positions = np.asarray(
        numeric_positions,
        dtype=int
    )


    # ========================================================
    # 5. PERTURBATION LOOP
    # ========================================================

    for condition, level in W2_CONDITIONS:

        for seed in W2_SEEDS:

            rng = np.random.RandomState(
                seed
            )


            # =================================================
            # A. GAUSSIAN NOISE
            # normalized numeric representation
            # =================================================

            if condition == 'noise':

                Xpert = (
                    Xsample_clean.copy()
                )

                noise = rng.normal(
                    loc=0.0,
                    scale=level,
                    size=(
                        len(Xpert),
                        len(
                            numeric_positions
                        )
                    )
                )

                Xpert[
                    :,
                    numeric_positions
                ] += noise


                # numeric MinMax representation
                Xpert[
                    :,
                    numeric_positions
                ] = np.clip(
                    Xpert[
                        :,
                        numeric_positions
                    ],
                    0.0,
                    1.0
                )


            # =================================================
            # B. MISSING TELEMETRY
            # raw numeric entries -> NaN -> train-fitted imputer
            # =================================================

            elif condition == 'missing':

                Xpert_raw = (
                    Xsample_raw.copy()
                )

                miss_cols = [
                    c
                    for c in numeric_cols
                    if str(c).lower()
                    != 'node_h'
                ]


                if len(
                    miss_cols
                ) > 0:

                    mask = (
                        rng.random(
                            (
                                len(
                                    Xpert_raw
                                ),
                                len(
                                    miss_cols
                                )
                            )
                        )
                        < level
                    )

                    arr = (
                        Xpert_raw[
                            miss_cols
                        ]
                        .to_numpy(
                            dtype=float,
                            copy=True
                        )
                    )

                    arr[
                        mask
                    ] = np.nan

                    Xpert_raw.loc[
                        :,
                        miss_cols
                    ] = arr


                Xpert = pre.transform(
                    Xpert_raw
                )


            # =================================================
            # SHAP
            # =================================================

            t1 = time.perf_counter()

            shap_pert = tree_shap_values(
                model,
                Xpert
            )

            shap_sec = (
                time.perf_counter()
                - t1
            )

            imp_pert = (
                global_shap_importance(
                    shap_pert
                )
            )


            st = stability_metrics(
                imp_clean,
                imp_pert,
                TOP_K
            )


            W2_RESULTS.append({

                'dataset':
                    name,

                'model':
                    'XGBoost',

                'condition':
                    condition,

                'level':
                    level,

                'seed':
                    seed,

                'sample_n':
                    len(
                        sample_idx
                    ),

                'clean_SHAP_sec':
                    clean_sec,

                'perturbed_SHAP_sec':
                    shap_sec,

                **st
            })


            print(
                f"{condition:7s} "
                f"{level:.2f} | "
                f"seed={seed} | "
                f"FSS={st['FSS_Top10']:.3f} | "
                f"rho={st['Spearman']:.3f} | "
                f"mag={st['Magnitude_similarity']:.3f}"
            )


            del (
                Xpert,
                shap_pert,
                imp_pert
            )

            gc.collect()


    del (
        Xtr_p,
        Xte_clean,
        Xsample_clean,
        shap_clean,
        model
    )

    gc.collect()


# ============================================================
# 6. SAVE DETAILED RESULTS
# ============================================================

w2_df = pd.DataFrame(
    W2_RESULTS
)

display(
    w2_df
)

w2_df.to_csv(
    OUT
    / 'W2A_XGB_SHAP_stability_results.csv',
    index=False
)


# ============================================================
# 7. SUMMARY ACROSS 3 SEEDS
# ============================================================

w2_summary = (

    w2_df

    .groupby(
        [
            'dataset',
            'condition',
            'level'
        ]
    )

    .agg(

        FSS_mean=(
            'FSS_Top10',
            'mean'
        ),

        FSS_std=(
            'FSS_Top10',
            'std'
        ),

        Spearman_mean=(
            'Spearman',
            'mean'
        ),

        Spearman_std=(
            'Spearman',
            'std'
        ),

        Magnitude_mean=(
            'Magnitude_similarity',
            'mean'
        ),

        Magnitude_std=(
            'Magnitude_similarity',
            'std'
        )
    )

    .reset_index()
)


display(
    w2_summary
)


w2_summary.to_csv(
    OUT
    / 'W2A_XGB_SHAP_stability_summary.csv',
    index=False
)


# ============================================================
# 8. CLEAN GLOBAL IMPORTANCE
# ============================================================

clean_importance_df = pd.DataFrame(
    W2_CLEAN_IMPORTANCE
)

clean_importance_df.to_csv(
    OUT
    / 'W2A_XGB_clean_global_importance.csv',
    index=False
)


print(
    "\n✅ W2A selesai."
)

print(
    "Kirim tiga file:"
)

print(
    OUT
    / 'W2A_XGB_SHAP_stability_results.csv'
)

print(
    OUT
    / 'W2A_XGB_SHAP_stability_summary.csv'
)

print(
    OUT
    / 'W2A_XGB_clean_global_importance.csv'
)


W2A SHAP STABILITY — CICIDS2018
Train=264,580 | Test=66,146 | SHAP sample=3,000

Clean Top-10:
  num__fwd_seg_size_min = 4.921649
  num__dst_port = 2.920434
  num__flow_duration = 0.362614
  num__totlen_bwd_pkts = 0.308526
  num__bwd_pkts_per_s = 0.267301
  num__fwd_act_data_pkts = 0.202187
  num__init_bwd_win_byts = 0.150629
  num__flow_byts_per_s = 0.129854
  num__fwd_header_len = 0.100368
  num__bwd_pkt_len_max = 0.099144
noise   0.01 | seed=42 | FSS=1.000 | rho=1.000 | mag=0.956
noise   0.01 | seed=43 | FSS=1.000 | rho=1.000 | mag=0.952
noise   0.01 | seed=44 | FSS=1.000 | rho=1.000 | mag=0.955
noise   0.05 | seed=42 | FSS=1.000 | rho=1.000 | mag=0.949
noise   0.05 | seed=43 | FSS=1.000 | rho=1.000 | mag=0.945
noise   0.05 | seed=44 | FSS=1.000 | rho=1.000 | mag=0.947
missing 0.01 | seed=42 | FSS=1.000 | rho=1.000 | mag=0.999
missing 0.01 | seed=43 | FSS=1.000 | rho=1.000 | mag=0.998
missing 0.01 | seed=44 | FSS=1.000 | rho=1.000 | mag=0.998
missing 0.05 | seed=42 | FSS=1.000 | rh

,dataset,model,condition,level,seed,sample_n,clean_SHAP_sec,perturbed_SHAP_sec,FSS_Top10,Top10_overlap_n,Spearman,Magnitude_similarity
0,CICIDS2018,XGBoost,noise,0.01,42,3000,0.008700,0.007292,1.0,10,0.999903,0.956244
1,CICIDS2018,XGBoost,noise,0.01,43,3000,0.008700,0.006462,1.0,10,0.999903,0.952338
2,CICIDS2018,XGBoost,noise,0.01,44,3000,0.008700,0.006534,1.0,10,0.999903,0.954627
3,CICIDS2018,XGBoost,noise,0.05,42,3000,0.008700,0.006622,1.0,10,0.999661,0.948610
4,CICIDS2018,XGBoost,noise,0.05,43,3000,0.008700,0.006718,1.0,10,0.999661,0.945192
5,CICIDS2018,XGBoost,noise,0.05,44,3000,0.008700,0.006597,1.0,10,0.999661,0.947371
6,CICIDS2018,XGBoost,missing,0.01,42,3000,0.008700,0.006547,1.0,10,1.000000,0.998882
7,CICIDS2018,XGBoost,missing,0.01,43,3000,0.008700,0.006456,1.0,10,1.000000,0.998018
8,CICIDS2018,XGBoost,missing,0.01,44,3000,0.008700,0.007280,1.0,10,0.999952,0.998314
9,CICIDS2018,XGBoost,missing,0.05,42,3000,0.008700,0.006707,1.0,10,0.999952,0.991413


,dataset,condition,level,FSS_mean,FSS_std,Spearman_mean,Spearman_std,Magnitude_mean,Magnitude_std
0,BGL,missing,0.01,1.000000,0.000000,1.000000,0.000000,0.996828,0.000438
1,BGL,missing,0.05,1.000000,0.000000,1.000000,0.000000,0.986189,0.000860
2,BGL,noise,0.01,1.000000,0.000000,0.989189,0.000000,0.949393,0.002323
3,BGL,noise,0.05,1.000000,0.000000,0.989189,0.000000,0.976679,0.001484
4,CICIDS2018,missing,0.01,1.000000,0.000000,0.999984,0.000028,0.998405,0.000439
5,CICIDS2018,missing,0.05,1.000000,0.000000,0.999952,0.000000,0.990082,0.001159
6,CICIDS2018,noise,0.01,1.000000,0.000000,0.999903,0.000000,0.954403,0.001963
7,CICIDS2018,noise,0.05,1.000000,0.000000,0.999661,0.000000,0.947058,0.001730
8,HDFS,missing,0.01,1.000000,0.000000,0.999616,0.000665,0.997595,0.000280
9,HDFS,missing,0.05,0.966667,0.057735,0.998464,0.000665,0.989024,0.001428



✅ W2A selesai.
Kirim tiga file:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/W2A_XGB_SHAP_stability_results.csv
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/W2A_XGB_SHAP_stability_summary.csv
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/W2A_XGB_clean_global_importance.csv


In [13]:
# ============================================================
# W2B — XGBOOST SHAP STABILITY UNDER ORDER/DISTRIBUTION SHIFT
# HDFS + BGL ONLY
#
# Source = first 70%
# Target = last 30%, exact-source overlap purged
#
# Source evaluation is held-out from model fit.
# SHAP samples are CLASS-BALANCED to control prevalence.
# ============================================================

import numpy as np
import pandas as pd
import gc
import xgboost as xgb

from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import spearmanr


# ============================================================
# HARD CHECK
# ============================================================

required = [
    'DATASETS',
    'CFG',
    'OUT',
    'make_preprocessor'
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Object belum tersedia: "
        + ", ".join(missing)
        + "\nJalankan bootstrap/resume terlebih dahulu."
    )


# ============================================================
# CONFIG
# ============================================================

W2B_DATASETS = [
    'HDFS',
    'BGL'
]

W2B_SEEDS = [
    CFG['SEED'],
    CFG['SEED'] + 1,
    CFG['SEED'] + 2
]

SOURCE_FRAC = 0.70

SOURCE_EVAL_FRAC = 0.20

MAX_PER_CLASS = 1500

TOP_K = 10

W2B_RESULTS = []


# ============================================================
# HELPERS
# ============================================================

def tree_shap_values_w2b(
    model,
    X
):
    dm = xgb.DMatrix(X)

    v = (
        model
        .get_booster()
        .predict(
            dm,
            pred_contribs=True
        )
    )

    return v[:, :-1]


def global_importance_w2b(
    shap_values
):
    return np.mean(
        np.abs(shap_values),
        axis=0
    )


def stability_w2b(
    base_imp,
    target_imp,
    k=10
):

    base_imp = np.asarray(
        base_imp,
        dtype=float
    )

    target_imp = np.asarray(
        target_imp,
        dtype=float
    )

    k = min(
        k,
        len(base_imp)
    )


    # --------------------------------------------------------
    # Top-k membership overlap
    # --------------------------------------------------------

    top_a = np.argsort(
        base_imp
    )[-k:]

    top_b = np.argsort(
        target_imp
    )[-k:]


    overlap = len(
        set(top_a)
        .intersection(
            set(top_b)
        )
    )


    fss = overlap / k


    # --------------------------------------------------------
    # Whole-ranking stability
    # --------------------------------------------------------

    rho = spearmanr(
        base_imp,
        target_imp
    ).statistic

    if np.isnan(rho):
        rho = 0.0


    # --------------------------------------------------------
    # Magnitude distribution similarity
    # --------------------------------------------------------

    p = base_imp / max(
        base_imp.sum(),
        1e-12
    )

    q = target_imp / max(
        target_imp.sum(),
        1e-12
    )


    tvd = (
        0.5
        * np.sum(
            np.abs(
                p - q
            )
        )
    )


    return {

        'FSS_Top10':
            float(fss),

        'Top10_overlap_n':
            int(overlap),

        'Spearman':
            float(rho),

        'Magnitude_similarity':
            float(
                1.0 - tvd
            )
    }


def balanced_indices(
    y,
    max_per_class,
    seed
):

    y = np.asarray(
        y,
        dtype=int
    )

    rng = np.random.RandomState(
        seed
    )

    neg = np.where(
        y == 0
    )[0]

    pos = np.where(
        y == 1
    )[0]


    n = min(
        max_per_class,
        len(neg),
        len(pos)
    )


    if n < 1:
        raise ValueError(
            "Balanced SHAP sample tidak dapat dibuat."
        )


    neg_sel = rng.choice(
        neg,
        size=n,
        replace=False
    )

    pos_sel = rng.choice(
        pos,
        size=n,
        replace=False
    )


    idx = np.concatenate([
        neg_sel,
        pos_sel
    ])


    rng.shuffle(
        idx
    )


    return (
        idx,
        n
    )


# ============================================================
# DATASET LOOP
# ============================================================

for name in W2B_DATASETS:

    print(
        f"\n{'='*78}"
    )

    print(
        f"W2B SHIFT SHAP STABILITY — {name}"
    )

    print(
        '='*78
    )


    X, y = DATASETS[
        name
    ]

    y = np.asarray(
        y,
        dtype=int
    )


    # ========================================================
    # 1. INITIAL 70/30 ORDER SPLIT
    # ========================================================

    cut = int(
        len(X)
        * SOURCE_FRAC
    )


    source_idx = np.arange(
        0,
        cut
    )

    target_initial_idx = np.arange(
        cut,
        len(X)
    )


    # ========================================================
    # 2. PURGE TARGET EXACT-FEATURE OVERLAP
    # ========================================================

    source_hash = set(

        pd.util.hash_pandas_object(
            X.iloc[
                source_idx
            ],
            index=False
        )

        .astype('uint64')
        .values
    )


    target_hash = (

        pd.util.hash_pandas_object(
            X.iloc[
                target_initial_idx
            ],
            index=False
        )

        .astype('uint64')
        .values
    )


    novel_mask = np.array([
        h not in source_hash
        for h in target_hash
    ])


    target_idx = (
        target_initial_idx[
            novel_mask
        ]
    )


    print(
        f"Source initial={len(source_idx):,} | "
        f"Target initial={len(target_initial_idx):,} | "
        f"Target after purge={len(target_idx):,}"
    )


    # ========================================================
    # 3. SOURCE FIT / SOURCE-EVAL GROUP SPLIT
    # ========================================================

    X_source = (
        X.iloc[
            source_idx
        ]
        .reset_index(
            drop=True
        )
    )

    y_source = y[
        source_idx
    ]


    source_groups = (

        pd.util.hash_pandas_object(
            X_source,
            index=False
        )

        .astype('uint64')
        .values
    )


    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=SOURCE_EVAL_FRAC,
        random_state=CFG['SEED']
    )


    fit_rel, eval_rel = next(

        gss.split(
            X_source,
            y_source,
            groups=source_groups
        )
    )


    Xfit_raw = (
        X_source.iloc[
            fit_rel
        ]
        .reset_index(
            drop=True
        )
    )

    yfit = y_source[
        fit_rel
    ]


    Xeval_raw = (
        X_source.iloc[
            eval_rel
        ]
        .reset_index(
            drop=True
        )
    )

    yeval = y_source[
        eval_rel
    ]


    Xtarget_raw = (
        X.iloc[
            target_idx
        ]
        .reset_index(
            drop=True
        )
    )

    ytarget = y[
        target_idx
    ]


    # hard check
    fit_hash = set(

        pd.util.hash_pandas_object(
            Xfit_raw,
            index=False
        )

        .astype('uint64')
        .values
    )


    eval_hash = (

        pd.util.hash_pandas_object(
            Xeval_raw,
            index=False
        )

        .astype('uint64')
        .values
    )


    assert not any(
        h in fit_hash
        for h in eval_hash
    )


    print(
        f"Model fit={len(Xfit_raw):,} | "
        f"Source eval={len(Xeval_raw):,} | "
        f"Target={len(Xtarget_raw):,}"
    )

    print(
        f"Source-eval positive="
        f"{yeval.sum():,} "
        f"({100*yeval.mean():.3f}%) | "
        f"Target positive="
        f"{ytarget.sum():,} "
        f"({100*ytarget.mean():.3f}%)"
    )


    # ========================================================
    # 4. TRAIN-ONLY PREPROCESSING
    # ========================================================

    pre = make_preprocessor(
        Xfit_raw
    )


    Xfit = pre.fit_transform(
        Xfit_raw
    )

    Xeval = pre.transform(
        Xeval_raw
    )

    Xtarget = pre.transform(
        Xtarget_raw
    )


    # ========================================================
    # 5. FIT XGBOOST
    # ========================================================

    model = XGBClassifier(

        random_state=CFG['SEED'],

        n_jobs=-1,

        eval_metric='logloss',

        tree_method='hist',

        verbosity=0
    )


    model.fit(
        Xfit,
        yfit
    )


    # ========================================================
    # 6. BALANCED SHAP COMPARISON — 3 SEEDS
    # ========================================================

    for seed in W2B_SEEDS:

        src_idx, src_n_class = (
            balanced_indices(
                yeval,
                MAX_PER_CLASS,
                seed
            )
        )


        tgt_idx, tgt_n_class = (
            balanced_indices(
                ytarget,
                MAX_PER_CLASS,
                seed
            )
        )


        # same number per class on BOTH sides
        n_per_class = min(
            src_n_class,
            tgt_n_class
        )


        # resample exactly same class count
        def exact_balanced(
            y_local,
            n_per_class,
            seed_local
        ):

            rng = np.random.RandomState(
                seed_local
            )

            neg = np.where(
                y_local == 0
            )[0]

            pos = np.where(
                y_local == 1
            )[0]


            nsel = rng.choice(
                neg,
                size=n_per_class,
                replace=False
            )

            psel = rng.choice(
                pos,
                size=n_per_class,
                replace=False
            )


            out = np.concatenate([
                nsel,
                psel
            ])

            rng.shuffle(
                out
            )

            return out


        src_idx = exact_balanced(
            yeval,
            n_per_class,
            seed
        )

        tgt_idx = exact_balanced(
            ytarget,
            n_per_class,
            seed
        )


        shap_source = (
            tree_shap_values_w2b(
                model,
                Xeval[
                    src_idx
                ]
            )
        )


        shap_target = (
            tree_shap_values_w2b(
                model,
                Xtarget[
                    tgt_idx
                ]
            )
        )


        imp_source = (
            global_importance_w2b(
                shap_source
            )
        )


        imp_target = (
            global_importance_w2b(
                shap_target
            )
        )


        st = stability_w2b(
            imp_source,
            imp_target,
            TOP_K
        )


        W2B_RESULTS.append({

            'dataset':
                name,

            'model':
                'XGBoost',

            'comparison':
                'source_vs_shifted_target',

            'seed':
                seed,

            'n_per_class':
                int(
                    n_per_class
                ),

            'source_sample_n':
                int(
                    2 * n_per_class
                ),

            'target_sample_n':
                int(
                    2 * n_per_class
                ),

            'source_positive_rate':
                0.5,

            'target_positive_rate':
                0.5,

            **st
        })


        print(
            f"seed={seed} | "
            f"n/class={n_per_class} | "
            f"FSS={st['FSS_Top10']:.3f} | "
            f"rho={st['Spearman']:.3f} | "
            f"mag={st['Magnitude_similarity']:.3f}"
        )


        del (
            shap_source,
            shap_target,
            imp_source,
            imp_target
        )

        gc.collect()


    del (
        Xfit,
        Xeval,
        Xtarget,
        model
    )

    gc.collect()


# ============================================================
# 7. SAVE RESULTS
# ============================================================

w2b_df = pd.DataFrame(
    W2B_RESULTS
)


display(
    w2b_df
)


w2b_df.to_csv(
    OUT
    / 'W2B_HDFS_BGL_shift_SHAP_stability.csv',
    index=False
)


# ============================================================
# 8. SUMMARY
# ============================================================

w2b_summary = (

    w2b_df

    .groupby(
        'dataset'
    )

    .agg(

        FSS_mean=(
            'FSS_Top10',
            'mean'
        ),

        FSS_std=(
            'FSS_Top10',
            'std'
        ),

        Spearman_mean=(
            'Spearman',
            'mean'
        ),

        Spearman_std=(
            'Spearman',
            'std'
        ),

        Magnitude_mean=(
            'Magnitude_similarity',
            'mean'
        ),

        Magnitude_std=(
            'Magnitude_similarity',
            'std'
        ),

        n_per_class=(
            'n_per_class',
            'min'
        )
    )

    .reset_index()
)


display(
    w2b_summary
)


w2b_summary.to_csv(
    OUT
    / 'W2B_HDFS_BGL_shift_SHAP_stability_summary.csv',
    index=False
)


print(
    "\n✅ W2B selesai."
)

print(
    "Kirim dua file:"
)

print(
    OUT
    / 'W2B_HDFS_BGL_shift_SHAP_stability.csv'
)

print(
    OUT
    / 'W2B_HDFS_BGL_shift_SHAP_stability_summary.csv'
)


W2B SHIFT SHAP STABILITY — HDFS
Source initial=402,542 | Target initial=172,519 | Target after purge=170,643
Model fit=321,806 | Source eval=80,736 | Target=170,643
Source-eval positive=2,773 (3.435%) | Target positive=2,336 (1.369%)
seed=42 | n/class=1500 | FSS=0.800 | rho=0.945 | mag=0.779
seed=43 | n/class=1500 | FSS=0.800 | rho=0.945 | mag=0.783
seed=44 | n/class=1500 | FSS=0.800 | rho=0.947 | mag=0.786

W2B SHIFT SHAP STABILITY — BGL
Source initial=35,000 | Target initial=15,000 | Target after purge=12,331
Model fit=28,014 | Source eval=6,986 | Target=12,331
Source-eval positive=537 (7.687%) | Target positive=753 (6.107%)
seed=42 | n/class=537 | FSS=1.000 | rho=0.989 | mag=0.827
seed=43 | n/class=537 | FSS=1.000 | rho=0.989 | mag=0.828
seed=44 | n/class=537 | FSS=1.000 | rho=0.989 | mag=0.832


,dataset,model,comparison,seed,n_per_class,source_sample_n,target_sample_n,source_positive_rate,target_positive_rate,FSS_Top10,Top10_overlap_n,Spearman,Magnitude_similarity
0,HDFS,XGBoost,source_vs_shifted_target,42,1500,3000,3000,0.5,0.5,0.8,8,0.944700,0.779438
1,HDFS,XGBoost,source_vs_shifted_target,43,1500,3000,3000,0.5,0.5,0.8,8,0.944700,0.782642
2,HDFS,XGBoost,source_vs_shifted_target,44,1500,3000,3000,0.5,0.5,0.8,8,0.947005,0.785883
3,BGL,XGBoost,source_vs_shifted_target,42,537,1074,1074,0.5,0.5,1.0,10,0.989189,0.826533
4,BGL,XGBoost,source_vs_shifted_target,43,537,1074,1074,0.5,0.5,1.0,10,0.989189,0.828465
5,BGL,XGBoost,source_vs_shifted_target,44,537,1074,1074,0.5,0.5,1.0,10,0.989189,0.831991


,dataset,FSS_mean,FSS_std,Spearman_mean,Spearman_std,Magnitude_mean,Magnitude_std,n_per_class
0,BGL,1.0,0.0,0.989189,0.00000,0.828996,0.002768,537
1,HDFS,0.8,0.0,0.945469,0.00133,0.782654,0.003223,1500



✅ W2B selesai.
Kirim dua file:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/W2B_HDFS_BGL_shift_SHAP_stability.csv
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/W2B_HDFS_BGL_shift_SHAP_stability_summary.csv


In [15]:
# ============================================================
# W2C — CICIDS2018 CROSS-DAY SHAP STABILITY
#
# Source : 02-14-2018
# Target : 02-15-2018
#
# XGBoost + TreeSHAP
# 3 sampling seeds
#
# Metrics:
# - FSS Top-10
# - Spearman full-rank stability
# - Magnitude similarity
#
# IMPORTANT:
# target labels are NOT used for training.
# Labels are used only to create class-balanced SHAP samples.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import gc
import re

import xgboost as xgb

from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import spearmanr


# ============================================================
# 0. ENVIRONMENT CHECK
# ============================================================

required = [
    'DATASETS',
    'CFG',
    'OUT',
    'make_preprocessor'
]

missing = [
    x
    for x in required
    if x not in globals()
]

if missing:

    raise RuntimeError(
        "Object belum tersedia: "
        + ", ".join(missing)
        + "\nJalankan bootstrap/resume terlebih dahulu."
    )


# ============================================================
# 1. CONFIG
# ============================================================

CICIDS_DIR = Path(
    '/content/drive/MyDrive/ercyris_siklus2'
)

DAY15_EXACT = (
    CICIDS_DIR
    / '02-15-2018.csv'
)

if DAY15_EXACT.exists():

    DAY15 = DAY15_EXACT

else:

    candidates = sorted(
        CICIDS_DIR.glob(
            '*02-15*.csv'
        )
    )

    if len(candidates) == 0:

        raise FileNotFoundError(
            "02-15-2018.csv tidak ditemukan di "
            + str(CICIDS_DIR)
        )

    DAY15 = candidates[0]


W2C_SEEDS = [
    CFG['SEED'],
    CFG['SEED'] + 1,
    CFG['SEED'] + 2
]

TOP_K = 10

SOURCE_EVAL_FRAC = 0.20

MAX_PER_CLASS = 1500

W2C_RESULTS = []


print(
    "Target file:",
    DAY15
)


# ============================================================
# 2. LOCAL COLUMN NORMALIZER
#
# Only for matching source feature names to day-15 columns.
# It does NOT alter the scientific feature values.
# ============================================================

def norm_col_w2c(x):
    """
    Normalize CICIDS2018 column names consistently between
    the confirmatory 02-14 checkpoint and raw 02-15 CSV.

    Handles CICFlowMeter naming differences such as:
    Flow Bytes/s       <-> flow_byts_per_s
    Flow Packets/s     <-> flow_pkts_per_s
    Fwd Avg Bytes/Bulk <-> fwd_byts_per_b_avg
    """

    x = str(x).strip().lower()

    # -----------------------------------------------
    # preserve "/" semantics as "per"
    # -----------------------------------------------
    x = x.replace(
        '/',
        ' per '
    )

    # -----------------------------------------------
    # basic cleanup
    # -----------------------------------------------
    x = re.sub(
        r'[^a-z0-9]+',
        '_',
        x
    )

    x = re.sub(
        r'_+',
        '_',
        x
    ).strip('_')


    # -----------------------------------------------
    # standardize CICFlowMeter abbreviations
    # -----------------------------------------------
    replacements = {

        'bytes':
            'byts',

        'packets':
            'pkts',

        'packet':
            'pkt',

        'forward':
            'fwd',

        'backward':
            'bwd',

        'bulk':
            'b'
    }


    tokens = x.split('_')

    tokens = [
        replacements.get(
            token,
            token
        )
        for token in tokens
    ]

    x = '_'.join(
        tokens
    )


    # -----------------------------------------------
    # semantic aliases between raw CICFlowMeter
    # headers and checkpoint feature names
    # -----------------------------------------------

    aliases = {

        # Flow rate
        'flow_byts_per_s':
            'flow_byts_per_s',

        'flow_pkts_per_s':
            'flow_pkts_per_s',

        'fwd_pkts_per_s':
            'fwd_pkts_per_s',

        'bwd_pkts_per_s':
            'bwd_pkts_per_s',


        # Down/Up Ratio
        'down_per_up_ratio':
            'down_per_up_ratio',


        # Raw CICFlowMeter:
        # "Fwd Avg Bytes/Bulk"
        'fwd_avg_byts_per_b':
            'fwd_byts_per_b_avg',

        # Raw CICFlowMeter:
        # "Fwd Avg Packets/Bulk"
        'fwd_avg_pkts_per_b':
            'fwd_pkts_per_b_avg',

        # Raw CICFlowMeter:
        # "Bwd Avg Bytes/Bulk"
        'bwd_avg_byts_per_b':
            'bwd_byts_per_b_avg',

        # Raw CICFlowMeter:
        # "Bwd Avg Packets/Bulk"
        'bwd_avg_pkts_per_b':
            'bwd_pkts_per_b_avg',


        # already-cleaned checkpoint versions
        'fwd_byts_per_b_avg':
            'fwd_byts_per_b_avg',

        'fwd_pkts_per_b_avg':
            'fwd_pkts_per_b_avg',

        'bwd_byts_per_b_avg':
            'bwd_byts_per_b_avg',

        'bwd_pkts_per_b_avg':
            'bwd_pkts_per_b_avg'
    }


    return aliases.get(
        x,
        x
    )


# ============================================================
# 3. SOURCE DAY — USE CONFIRMATORY CHECKPOINT
# ============================================================

X14, y14 = DATASETS[
    'CICIDS2018'
]

X14 = (
    X14
    .reset_index(
        drop=True
    )
)

y14 = np.asarray(
    y14,
    dtype=int
)


print(
    f"\n02-14 source:"
    f" {len(X14):,} rows | "
    f"attack={y14.sum():,} "
    f"({100*y14.mean():.3f}%)"
)


# ============================================================
# 4. LOAD DAY 15
# ============================================================

d15 = pd.read_csv(
    DAY15,
    low_memory=False
)


# ------------------------------------------------------------
# Find LABEL
# ------------------------------------------------------------

label_candidates = [
    c
    for c in d15.columns
    if 'label' in norm_col_w2c(c)
]


if len(label_candidates) == 0:

    raise ValueError(
        "Kolom LABEL 02-15 tidak ditemukan."
    )


label_col = label_candidates[0]


label15_raw = (
    d15[
        label_col
    ]
    .astype(str)
    .str.strip()
)


y15 = (
    label15_raw
    .str.upper()
    != 'BENIGN'
).astype(int)


print(
    f"\n02-15 raw:"
    f" {len(d15):,} rows | "
    f"attack={y15.sum():,} "
    f"({100*y15.mean():.3f}%)"
)


# ============================================================
# 5. ALIGN DAY-15 FEATURE SCHEMA TO X14
#
# We do not depend on the old clean_cols() helper.
# ============================================================

day15_map = {}


for c in d15.columns:

    key = norm_col_w2c(c)

    if key not in day15_map:
        day15_map[key] = c


matched = {}
missing_features = []


for source_col in X14.columns:

    key = norm_col_w2c(
        source_col
    )

    if key in day15_map:

        matched[
            source_col
        ] = day15_map[
            key
        ]

    else:

        missing_features.append(
            source_col
        )


print(
    "\nMissing target features:",
    missing_features
)


if len(
    missing_features
) > 0:

    raise ValueError(
        "Feature schema tidak kompatibel. "
        "W2C dihentikan."
    )


X15 = pd.DataFrame({

    source_col:
        d15[
            target_col
        ].values

    for source_col, target_col
    in matched.items()
})


X15 = X15[
    X14.columns
].copy()


X15.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)


y15 = (
    y15
    .reset_index(
        drop=True
    )
)


# ============================================================
# 6. REMOVE TARGET FEATURE/LABEL CONFLICTS
# ============================================================

hash15 = (

    pd.util.hash_pandas_object(
        X15,
        index=False
    )

    .astype('uint64')
    .values
)


audit15 = pd.DataFrame({

    'h':
        hash15,

    'y':
        y15.values
})


label_count = (

    audit15

    .groupby(
        'h'
    )['y']

    .nunique()
)


conflict_hash = set(

    label_count[
        label_count > 1
    ].index
)


conflict_mask = np.array([

    h in conflict_hash

    for h in hash15
])


n_conflict_rows = int(
    conflict_mask.sum()
)


print(
    "\nTarget conflicting groups:",
    len(
        conflict_hash
    )
)

print(
    "Rows in conflicting groups:",
    n_conflict_rows
)


if n_conflict_rows > 0:

    keep = ~conflict_mask

    X15 = (
        X15.loc[
            keep
        ]
        .reset_index(
            drop=True
        )
    )

    y15 = (
        y15.loc[
            keep
        ]
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 7. TARGET DEDUPLICATION
# ============================================================

before_dedup = len(
    X15
)


hash15 = (

    pd.util.hash_pandas_object(
        X15,
        index=False
    )

    .astype('uint64')
)


keep_unique = ~hash15.duplicated(
    keep='first'
).values


X15 = (
    X15.loc[
        keep_unique
    ]
    .reset_index(
        drop=True
    )
)

y15 = (
    y15.loc[
        keep_unique
    ]
    .reset_index(
        drop=True
    )
)


print(
    f"\nTarget dedup: "
    f"{before_dedup:,}"
    f" -> {len(X15):,}"
)


# ============================================================
# 8. PURGE EXACT CROSS-DAY FEATURE OVERLAP
# ============================================================

source_hash = set(

    pd.util.hash_pandas_object(
        X14,
        index=False
    )

    .astype('uint64')
    .values
)


target_hash = (

    pd.util.hash_pandas_object(
        X15,
        index=False
    )

    .astype('uint64')
    .values
)


overlap_mask = np.array([

    h in source_hash

    for h in target_hash
])


overlap_n = int(
    overlap_mask.sum()
)


print(
    "Cross-day overlap removed:",
    overlap_n
)


if overlap_n > 0:

    keep = ~overlap_mask

    X15 = (
        X15.loc[
            keep
        ]
        .reset_index(
            drop=True
        )
    )

    y15 = (
        y15.loc[
            keep
        ]
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# HARD FINAL LEAKAGE CHECK
# ------------------------------------------------------------

final_target_hash = (

    pd.util.hash_pandas_object(
        X15,
        index=False
    )

    .astype('uint64')
    .values
)


shared_final = np.intersect1d(

    np.fromiter(
        source_hash,
        dtype=np.uint64
    ),

    np.unique(
        final_target_hash
    )
)


assert (
    len(shared_final) == 0
)


print(
    f"\nFINAL target:"
    f" {len(X15):,} rows | "
    f"attack={y15.sum():,} "
    f"({100*y15.mean():.3f}%)"
)

print(
    "Final shared feature groups:",
    len(
        shared_final
    )
)


# ============================================================
# 9. SPLIT SOURCE INTO MODEL-FIT + SOURCE-EVAL
# ============================================================

source_groups = (

    pd.util.hash_pandas_object(
        X14,
        index=False
    )

    .astype('uint64')
    .values
)


gss = GroupShuffleSplit(

    n_splits=1,

    test_size=SOURCE_EVAL_FRAC,

    random_state=CFG['SEED']
)


fit_idx, eval_idx = next(

    gss.split(
        X14,
        y14,
        groups=source_groups
    )
)


Xfit_raw = (
    X14.iloc[
        fit_idx
    ]
    .reset_index(
        drop=True
    )
)

yfit = y14[
    fit_idx
]


Xeval_raw = (
    X14.iloc[
        eval_idx
    ]
    .reset_index(
        drop=True
    )
)

yeval = y14[
    eval_idx
]


# ------------------------------------------------------------
# Source fit / eval leakage check
# ------------------------------------------------------------

fit_hash = set(

    pd.util.hash_pandas_object(
        Xfit_raw,
        index=False
    )

    .astype('uint64')
    .values
)


eval_hash = (

    pd.util.hash_pandas_object(
        Xeval_raw,
        index=False
    )

    .astype('uint64')
    .values
)


assert not any(

    h in fit_hash

    for h in eval_hash
)


print(
    f"\nSource model-fit="
    f"{len(Xfit_raw):,}"
)

print(
    f"Source evaluation="
    f"{len(Xeval_raw):,}"
)

print(
    f"Source-eval attacks="
    f"{yeval.sum():,} "
    f"({100*yeval.mean():.3f}%)"
)


# ============================================================
# 10. TRAIN-ONLY PREPROCESSING
# ============================================================

pre = make_preprocessor(
    Xfit_raw
)


Xfit = pre.fit_transform(
    Xfit_raw
)


Xeval = pre.transform(
    Xeval_raw
)


Xtarget = pre.transform(
    X15
)


# ============================================================
# 11. XGBOOST
# ============================================================

model = XGBClassifier(

    random_state=CFG['SEED'],

    n_jobs=-1,

    eval_metric='logloss',

    tree_method='hist',

    verbosity=0
)


model.fit(
    Xfit,
    yfit
)


# ============================================================
# 12. FEATURE NAMES
# ============================================================

try:

    feature_names = (

        pre
        .named_steps[
            'columns'
        ]
        .get_feature_names_out()
    )

except Exception:

    feature_names = np.array([

        f'f{i}'

        for i in range(
            Xfit.shape[1]
        )
    ])


# ============================================================
# 13. HELPERS
# ============================================================

def shap_w2c(
    model,
    X
):

    dm = xgb.DMatrix(
        X
    )

    values = (

        model
        .get_booster()
        .predict(
            dm,
            pred_contribs=True
        )
    )

    return values[
        :,
        :-1
    ]


def imp_w2c(
    shap_values
):

    return np.mean(
        np.abs(
            shap_values
        ),
        axis=0
    )


def exact_balanced_w2c(
    y,
    n_per_class,
    seed
):

    y = np.asarray(
        y,
        dtype=int
    )

    rng = np.random.RandomState(
        seed
    )


    neg = np.where(
        y == 0
    )[0]

    pos = np.where(
        y == 1
    )[0]


    if (
        len(neg) < n_per_class
        or
        len(pos) < n_per_class
    ):

        raise ValueError(
            "Sample kelas tidak mencukupi."
        )


    neg_sel = rng.choice(
        neg,
        n_per_class,
        replace=False
    )

    pos_sel = rng.choice(
        pos,
        n_per_class,
        replace=False
    )


    out = np.concatenate([
        neg_sel,
        pos_sel
    ])


    rng.shuffle(
        out
    )


    return out


def stability_w2c(
    source_imp,
    target_imp,
    k=10
):

    source_imp = np.asarray(
        source_imp,
        dtype=float
    )

    target_imp = np.asarray(
        target_imp,
        dtype=float
    )


    k = min(
        k,
        len(
            source_imp
        )
    )


    # --------------------------------------------------------
    # FSS
    # --------------------------------------------------------

    top_source = np.argsort(
        source_imp
    )[-k:]


    top_target = np.argsort(
        target_imp
    )[-k:]


    overlap = len(

        set(
            top_source
        )

        .intersection(

            set(
                top_target
            )
        )
    )


    fss = overlap / k


    # --------------------------------------------------------
    # Full ranking
    # --------------------------------------------------------

    rho = spearmanr(
        source_imp,
        target_imp
    ).statistic


    if np.isnan(
        rho
    ):

        rho = 0.0


    # --------------------------------------------------------
    # Attribution magnitude distribution
    # --------------------------------------------------------

    p = source_imp / max(
        source_imp.sum(),
        1e-12
    )


    q = target_imp / max(
        target_imp.sum(),
        1e-12
    )


    tvd = (
        0.5
        * np.sum(
            np.abs(
                p - q
            )
        )
    )


    return {

        'FSS_Top10':
            float(
                fss
            ),

        'Top10_overlap_n':
            int(
                overlap
            ),

        'Spearman':
            float(
                rho
            ),

        'Magnitude_similarity':
            float(
                1.0 - tvd
            ),

        'source_top10':
            '|'.join(
                map(
                    str,
                    feature_names[
                        top_source[
                            ::-1
                        ]
                    ]
                )
            ),

        'target_top10':
            '|'.join(
                map(
                    str,
                    feature_names[
                        top_target[
                            ::-1
                        ]
                    ]
                )
            )
    }


# ============================================================
# 14. FIX BALANCED SAMPLE SIZE
# ============================================================

n_per_class = min(

    MAX_PER_CLASS,

    int(
        np.sum(
            yeval == 0
        )
    ),

    int(
        np.sum(
            yeval == 1
        )
    ),

    int(
        np.sum(
            y15.values == 0
        )
    ),

    int(
        np.sum(
            y15.values == 1
        )
    )
)


print(
    f"\nBalanced SHAP sample:"
    f" {n_per_class:,} per class"
)

print(
    f"Total/source="
    f"{2*n_per_class:,}"
)

print(
    f"Total/target="
    f"{2*n_per_class:,}"
)


# ============================================================
# 15. 3-SEED SHAP COMPARISON
# ============================================================

for seed in W2C_SEEDS:

    source_sample_idx = (
        exact_balanced_w2c(
            yeval,
            n_per_class,
            seed
        )
    )


    target_sample_idx = (
        exact_balanced_w2c(
            y15.values,
            n_per_class,
            seed
        )
    )


    shap_source = shap_w2c(

        model,

        Xeval[
            source_sample_idx
        ]
    )


    shap_target = shap_w2c(

        model,

        Xtarget[
            target_sample_idx
        ]
    )


    source_imp = imp_w2c(
        shap_source
    )


    target_imp = imp_w2c(
        shap_target
    )


    st = stability_w2c(
        source_imp,
        target_imp,
        TOP_K
    )


    W2C_RESULTS.append({

        'dataset':
            'CICIDS2018',

        'comparison':
            '02-14_source_vs_02-15_target',

        'model':
            'XGBoost',

        'seed':
            seed,

        'n_per_class':
            n_per_class,

        'source_sample_n':
            2 * n_per_class,

        'target_sample_n':
            2 * n_per_class,

        'target_conflict_groups_removed':
            len(
                conflict_hash
            ),

        'target_conflict_rows_removed':
            n_conflict_rows,

        'crossday_overlap_removed':
            overlap_n,

        'shared_groups_final':
            len(
                shared_final
            ),

        **st
    })


    print(
        f"\nseed={seed} | "
        f"FSS={st['FSS_Top10']:.3f} | "
        f"rho={st['Spearman']:.3f} | "
        f"mag={st['Magnitude_similarity']:.3f}"
    )


    print(
        "SOURCE Top-10:",
        st[
            'source_top10'
        ]
    )


    print(
        "TARGET Top-10:",
        st[
            'target_top10'
        ]
    )


    del (
        shap_source,
        shap_target,
        source_imp,
        target_imp
    )

    gc.collect()


# ============================================================
# 16. SAVE DETAILED RESULTS
# ============================================================

w2c_df = pd.DataFrame(
    W2C_RESULTS
)


display(
    w2c_df
)


w2c_df.to_csv(
    OUT
    / 'W2C_CICIDS_crossday_SHAP_stability.csv',
    index=False
)


# ============================================================
# 17. SUMMARY
# ============================================================

w2c_summary = pd.DataFrame([{

    'dataset':
        'CICIDS2018',

    'comparison':
        '02-14_vs_02-15',

    'FSS_mean':
        w2c_df[
            'FSS_Top10'
        ].mean(),

    'FSS_std':
        w2c_df[
            'FSS_Top10'
        ].std(),

    'Spearman_mean':
        w2c_df[
            'Spearman'
        ].mean(),

    'Spearman_std':
        w2c_df[
            'Spearman'
        ].std(),

    'Magnitude_mean':
        w2c_df[
            'Magnitude_similarity'
        ].mean(),

    'Magnitude_std':
        w2c_df[
            'Magnitude_similarity'
        ].std(),

    'n_per_class':
        n_per_class
}])


display(
    w2c_summary
)


w2c_summary.to_csv(
    OUT
    / 'W2C_CICIDS_crossday_SHAP_stability_summary.csv',
    index=False
)


print(
    "\n✅ W2C selesai."
)

print(
    "Kirim dua file:"
)

print(
    OUT
    / 'W2C_CICIDS_crossday_SHAP_stability.csv'
)

print(
    OUT
    / 'W2C_CICIDS_crossday_SHAP_stability_summary.csv'
)

Target file: /content/drive/MyDrive/ercyris_siklus2/02-15-2018.csv

02-14 source: 330,726 rows | attack=45,029 (13.615%)

02-15 raw: 1,048,575 rows | attack=52,498 (5.007%)

Missing target features: []

Target conflicting groups: 0
Rows in conflicting groups: 0

Target dedup: 1,048,575 -> 884,658
Cross-day overlap removed: 28732

FINAL target: 855,926 rows | attack=51,314 (5.995%)
Final shared feature groups: 0

Source model-fit=264,580
Source evaluation=66,146
Source-eval attacks=9,011 (13.623%)

Balanced SHAP sample: 1,500 per class
Total/source=3,000
Total/target=3,000

seed=42 | FSS=0.900 | rho=0.998 | mag=0.707
SOURCE Top-10: num__fwd_seg_size_min|num__dst_port|num__fwd_act_data_pkts|num__init_fwd_win_byts|num__totlen_bwd_pkts|num__flow_duration|num__flow_byts_per_s|num__bwd_pkts_per_s|num__flow_pkts_per_s|num__bwd_pkt_len_max
TARGET Top-10: num__fwd_seg_size_min|num__dst_port|num__flow_duration|num__totlen_bwd_pkts|num__fwd_act_data_pkts|num__flow_byts_per_s|num__bwd_pkt_len_max|

,dataset,comparison,model,seed,n_per_class,source_sample_n,target_sample_n,target_conflict_groups_removed,target_conflict_rows_removed,crossday_overlap_removed,shared_groups_final,FSS_Top10,Top10_overlap_n,Spearman,Magnitude_similarity,source_top10,target_top10
0,CICIDS2018,02-14_source_vs_02-15_target,XGBoost,42,1500,3000,3000,0,0,28732,0,0.9,9,0.998319,0.707198,num__fwd_seg_size_min|num__dst_port|num__fwd_a...,num__fwd_seg_size_min|num__dst_port|num__flow_...
1,CICIDS2018,02-14_source_vs_02-15_target,XGBoost,43,1500,3000,3000,0,0,28732,0,0.9,9,0.998319,0.707544,num__fwd_seg_size_min|num__dst_port|num__fwd_a...,num__fwd_seg_size_min|num__dst_port|num__flow_...
2,CICIDS2018,02-14_source_vs_02-15_target,XGBoost,44,1500,3000,3000,0,0,28732,0,0.9,9,0.998319,0.707875,num__fwd_seg_size_min|num__dst_port|num__fwd_a...,num__fwd_seg_size_min|num__dst_port|num__flow_...


,dataset,comparison,FSS_mean,FSS_std,Spearman_mean,Spearman_std,Magnitude_mean,Magnitude_std,n_per_class
0,CICIDS2018,02-14_vs_02-15,0.9,0.0,0.998319,1.359740e-16,0.707539,0.000338,1500



✅ W2C selesai.
Kirim dua file:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/W2C_CICIDS_crossday_SHAP_stability.csv
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/W2C_CICIDS_crossday_SHAP_stability_summary.csv


In [16]:
# ============================================================
# S5A — SUPERVISED MICRO-BATCH INFERENCE BENCHMARK
#
# Models:
#   - XGBoost
#   - Random Forest
#
# Batch:
#   - 5,000
#   - 20,000
#
# 1 warm-up + 10 measured runs
# Four confirmatory datasets
#
# IMPORTANT:
# Measures MODEL INFERENCE ONLY.
# Preprocessing is performed beforehand and reported separately.
# ============================================================

import time
import gc

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier


# ============================================================
# HARD ENVIRONMENT CHECK
# ============================================================

required = [
    'DATASETS',
    'SPLITS',
    'CFG',
    'OUT',
    'make_preprocessor'
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:

    raise RuntimeError(
        "Object belum tersedia: "
        + ", ".join(missing)
        + "\nJalankan bootstrap/resume terlebih dahulu."
    )


# ============================================================
# CONFIG
# ============================================================

BATCH_SIZES = [
    5000,
    20000
]

N_WARMUP = 1

N_RUNS = 10

S5_RESULTS = []


# ============================================================
# HELPER — deterministic batch selection
# ============================================================

def fixed_batch_indices(
    n,
    batch_size,
    seed
):

    if n < batch_size:

        return None

    rng = np.random.RandomState(
        seed
    )

    return np.sort(

        rng.choice(
            np.arange(n),
            size=batch_size,
            replace=False
        )
    )


# ============================================================
# DATASET LOOP
# ============================================================

for name, (X, y) in DATASETS.items():

    print(
        f"\n{'='*78}"
    )

    print(
        f"S5A MICRO-BATCH BENCHMARK — {name}"
    )

    print(
        '='*78
    )


    # --------------------------------------------------------
    # SAME S0 TRAIN / TEST
    # --------------------------------------------------------

    tr, te = SPLITS[
        name
    ]

    Xtr_raw = (
        X.iloc[
            tr
        ]
        .reset_index(
            drop=True
        )
    )

    Xte_raw = (
        X.iloc[
            te
        ]
        .reset_index(
            drop=True
        )
    )

    ytr = np.asarray(
        y[tr],
        dtype=int
    )


    # ========================================================
    # 1. PREPROCESSING — TRAIN FIT ONLY
    # ========================================================

    pre = make_preprocessor(
        Xtr_raw
    )


    t0 = time.perf_counter()


    Xtr = pre.fit_transform(
        Xtr_raw
    )

    Xte = pre.transform(
        Xte_raw
    )


    preprocessing_sec = (
        time.perf_counter()
        - t0
    )


    print(
        f"Train={len(Xtr):,} | "
        f"Test={len(Xte):,} | "
        f"Preprocessing={preprocessing_sec:.3f}s"
    )


    # ========================================================
    # 2. MODELS
    # ========================================================

    models = {

        'XGBoost':
            XGBClassifier(
                random_state=CFG['SEED'],
                n_jobs=-1,
                eval_metric='logloss',
                tree_method='hist',
                verbosity=0
            ),

        'RandomForest':
            RandomForestClassifier(
                n_estimators=100,
                random_state=CFG['SEED'],
                n_jobs=-1
            )
    }


    # ========================================================
    # 3. FIT EACH MODEL ONCE
    # ========================================================

    for model_name, model in models.items():

        print(
            f"\n--- {model_name} ---"
        )


        t1 = time.perf_counter()


        model.fit(
            Xtr,
            ytr
        )


        training_sec = (
            time.perf_counter()
            - t1
        )


        print(
            f"Training="
            f"{training_sec:.3f}s"
        )


        # ====================================================
        # 4. BATCH BENCHMARK
        # ====================================================

        for batch_size in BATCH_SIZES:

            batch_idx = fixed_batch_indices(

                len(Xte),

                batch_size,

                CFG['SEED']
                + batch_size
            )


            if batch_idx is None:

                print(
                    f"⚠️ Batch {batch_size:,} "
                    f"dilewati: test hanya "
                    f"{len(Xte):,} rows."
                )

                continue


            Xbatch = Xte[
                batch_idx
            ]


            # -----------------------------------------------
            # WARM-UP
            # -----------------------------------------------

            for _ in range(
                N_WARMUP
            ):

                _ = model.predict_proba(
                    Xbatch
                )


            # -----------------------------------------------
            # MEASURE
            # -----------------------------------------------

            latencies_sec = []


            for run in range(
                N_RUNS
            ):

                start = time.perf_counter()


                _ = model.predict_proba(
                    Xbatch
                )


                elapsed = (
                    time.perf_counter()
                    - start
                )


                latencies_sec.append(
                    elapsed
                )


            latencies_sec = np.asarray(
                latencies_sec,
                dtype=float
            )


            # =================================================
            # 5. METRICS
            # =================================================

            mean_sec = float(
                latencies_sec.mean()
            )

            median_sec = float(
                np.median(
                    latencies_sec
                )
            )

            p95_sec = float(
                np.quantile(
                    latencies_sec,
                    0.95
                )
            )

            min_sec = float(
                latencies_sec.min()
            )

            max_sec = float(
                latencies_sec.max()
            )


            mean_ms = (
                mean_sec
                * 1000
            )

            median_ms = (
                median_sec
                * 1000
            )

            p95_ms = (
                p95_sec
                * 1000
            )


            throughput_mean = (

                batch_size
                / mean_sec

                if mean_sec > 0

                else np.nan
            )


            throughput_p95 = (

                batch_size
                / p95_sec

                if p95_sec > 0

                else np.nan
            )


            per_instance_us = (

                mean_sec
                / batch_size
                * 1_000_000
            )


            S5_RESULTS.append({

                'dataset':
                    name,

                'scenario':
                    'S5_microbatch_inference',

                'model':
                    model_name,

                'batch_size':
                    batch_size,

                'warmup_runs':
                    N_WARMUP,

                'measured_runs':
                    N_RUNS,

                'preprocessing_full_train_test_sec':
                    preprocessing_sec,

                'training_sec':
                    training_sec,

                'mean_latency_ms':
                    mean_ms,

                'median_latency_ms':
                    median_ms,

                'p95_latency_ms':
                    p95_ms,

                'min_latency_ms':
                    min_sec * 1000,

                'max_latency_ms':
                    max_sec * 1000,

                'mean_throughput_instances_sec':
                    throughput_mean,

                'p95_based_throughput_instances_sec':
                    throughput_p95,

                'mean_per_instance_us':
                    per_instance_us
            })


            print(
                f"Batch={batch_size:,} | "
                f"mean={mean_ms:.3f} ms | "
                f"median={median_ms:.3f} ms | "
                f"p95={p95_ms:.3f} ms | "
                f"throughput="
                f"{throughput_mean:,.0f}/s"
            )


        del model

        gc.collect()


    del (
        Xtr,
        Xte
    )

    gc.collect()


# ============================================================
# 6. SAVE FULL RESULTS
# ============================================================

s5_df = pd.DataFrame(
    S5_RESULTS
)


display(
    s5_df
)


s5_df.to_csv(
    OUT
    / 'S5A_supervised_microbatch_latency_throughput.csv',
    index=False
)


# ============================================================
# 7. COMPACT SUMMARY
# ============================================================

s5_summary = (

    s5_df[
        [
            'dataset',
            'model',
            'batch_size',
            'mean_latency_ms',
            'median_latency_ms',
            'p95_latency_ms',
            'mean_throughput_instances_sec',
            'p95_based_throughput_instances_sec',
            'mean_per_instance_us'
        ]
    ]

    .sort_values(
        [
            'dataset',
            'model',
            'batch_size'
        ]
    )
)


display(
    s5_summary
)


s5_summary.to_csv(
    OUT
    / 'S5A_supervised_microbatch_summary.csv',
    index=False
)


print(
    "\n✅ S5A selesai."
)

print(
    "Kirim dua file:"
)

print(
    OUT
    / 'S5A_supervised_microbatch_latency_throughput.csv'
)

print(
    OUT
    / 'S5A_supervised_microbatch_summary.csv'
)


S5A MICRO-BATCH BENCHMARK — CICIDS2018
Train=264,580 | Test=66,146 | Preprocessing=3.281s

--- XGBoost ---
Training=1.045s
Batch=5,000 | mean=1.456 ms | median=1.444 ms | p95=1.531 ms | throughput=3,435,215/s
Batch=20,000 | mean=7.847 ms | median=7.164 ms | p95=12.105 ms | throughput=2,548,867/s

--- RandomForest ---
Training=4.084s
Batch=5,000 | mean=35.096 ms | median=35.031 ms | p95=35.542 ms | throughput=142,468/s
Batch=20,000 | mean=39.442 ms | median=37.400 ms | p95=47.547 ms | throughput=507,071/s

S5A MICRO-BATCH BENCHMARK — HDFS
Train=460,028 | Test=115,033 | Preprocessing=1.317s

--- XGBoost ---
Training=0.897s
Batch=5,000 | mean=2.264 ms | median=2.130 ms | p95=2.899 ms | throughput=2,208,051/s
Batch=20,000 | mean=6.579 ms | median=6.575 ms | p95=6.613 ms | throughput=3,039,988/s

--- RandomForest ---
Training=9.694s
Batch=5,000 | mean=34.711 ms | median=34.615 ms | p95=35.351 ms | throughput=144,045/s
Batch=20,000 | mean=41.708 ms | median=45.514 ms | p95=46.007 ms | throu

,dataset,scenario,model,batch_size,warmup_runs,measured_runs,preprocessing_full_train_test_sec,training_sec,mean_latency_ms,median_latency_ms,p95_latency_ms,min_latency_ms,max_latency_ms,mean_throughput_instances_sec,p95_based_throughput_instances_sec,mean_per_instance_us
0,CICIDS2018,S5_microbatch_inference,XGBoost,5000,1,10,3.281316,1.045201,1.455513,1.443575,1.531499,1.416285,1.571578,3.435215e+06,3.264776e+06,0.291103
1,CICIDS2018,S5_microbatch_inference,XGBoost,20000,1,10,3.281316,1.045201,7.846622,7.164258,12.104853,4.125466,13.408670,2.548867e+06,1.652230e+06,0.392331
2,CICIDS2018,S5_microbatch_inference,RandomForest,5000,1,10,3.281316,4.083596,35.095618,35.030577,35.541993,34.749960,35.617550,1.424679e+05,1.406787e+05,7.019124
3,CICIDS2018,S5_microbatch_inference,RandomForest,20000,1,10,3.281316,4.083596,39.442239,37.400440,47.547306,36.951106,47.600884,5.070706e+05,4.206337e+05,1.972112
4,HDFS,S5_microbatch_inference,XGBoost,5000,1,10,1.317471,0.897206,2.264441,2.129949,2.899149,2.082499,3.459340,2.208051e+06,1.724644e+06,0.452888
5,HDFS,S5_microbatch_inference,XGBoost,20000,1,10,1.317471,0.897206,6.578973,6.574646,6.612912,6.526621,6.615326,3.039988e+06,3.024386e+06,0.328949
6,HDFS,S5_microbatch_inference,RandomForest,5000,1,10,1.317471,9.693920,34.711462,34.615368,35.350920,34.129248,35.352782,1.440446e+05,1.414390e+05,6.942292
7,HDFS,S5_microbatch_inference,RandomForest,20000,1,10,1.317471,9.693920,41.707803,45.514225,46.006939,35.482061,46.045594,4.795266e+05,4.347170e+05,2.085390
8,BGL,S5_microbatch_inference,XGBoost,5000,1,10,0.032907,0.095803,5.189889,4.137197,11.453584,2.049016,13.596894,9.634118e+05,4.365446e+05,1.037978
9,BGL,S5_microbatch_inference,RandomForest,5000,1,10,0.032907,0.351538,34.675243,34.722414,34.991098,34.300733,35.005004,1.441951e+05,1.428935e+05,6.935049


,dataset,model,batch_size,mean_latency_ms,median_latency_ms,p95_latency_ms,mean_throughput_instances_sec,p95_based_throughput_instances_sec,mean_per_instance_us
9,BGL,RandomForest,5000,34.675243,34.722414,34.991098,1.441951e+05,1.428935e+05,6.935049
8,BGL,XGBoost,5000,5.189889,4.137197,11.453584,9.634118e+05,4.365446e+05,1.037978
2,CICIDS2018,RandomForest,5000,35.095618,35.030577,35.541993,1.424679e+05,1.406787e+05,7.019124
3,CICIDS2018,RandomForest,20000,39.442239,37.400440,47.547306,5.070706e+05,4.206337e+05,1.972112
0,CICIDS2018,XGBoost,5000,1.455513,1.443575,1.531499,3.435215e+06,3.264776e+06,0.291103
1,CICIDS2018,XGBoost,20000,7.846622,7.164258,12.104853,2.548867e+06,1.652230e+06,0.392331
6,HDFS,RandomForest,5000,34.711462,34.615368,35.350920,1.440446e+05,1.414390e+05,6.942292
7,HDFS,RandomForest,20000,41.707803,45.514225,46.006939,4.795266e+05,4.347170e+05,2.085390
4,HDFS,XGBoost,5000,2.264441,2.129949,2.899149,2.208051e+06,1.724644e+06,0.452888
5,HDFS,XGBoost,20000,6.578973,6.574646,6.612912,3.039988e+06,3.024386e+06,0.328949



✅ S5A selesai.
Kirim dua file:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S5A_supervised_microbatch_latency_throughput.csv
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S5A_supervised_microbatch_summary.csv


In [17]:
# ============================================================
# S5B — END-TO-END MICRO-BATCH BENCHMARK
#
# RAW BATCH
#   -> train-fitted preprocessing
#   -> classifier inference
#
# Models:
#   XGBoost
#   Random Forest
#
# Batch sizes:
#   5,000
#   20,000
#
# 1 warm-up + 10 measured runs
#
# Training + preprocessor fitting are EXCLUDED from
# operational batch latency.
# ============================================================

import time
import gc

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier


# ============================================================
# 0. ENVIRONMENT CHECK
# ============================================================

required = [
    'DATASETS',
    'SPLITS',
    'CFG',
    'OUT',
    'make_preprocessor'
]

missing = [
    x
    for x in required
    if x not in globals()
]

if missing:

    raise RuntimeError(
        "Object belum tersedia: "
        + ", ".join(missing)
        + "\nJalankan bootstrap/resume terlebih dahulu."
    )


# ============================================================
# 1. CONFIG
# ============================================================

BATCH_SIZES = [
    5000,
    20000
]

N_WARMUP = 1
N_RUNS = 10

S5B_DETAIL = []
S5B_SUMMARY = []


# ============================================================
# HELPER
# ============================================================

def fixed_batch_indices_s5b(
    n,
    batch_size,
    seed
):

    if n < batch_size:
        return None

    rng = np.random.RandomState(
        seed
    )

    return np.sort(
        rng.choice(
            np.arange(n),
            size=batch_size,
            replace=False
        )
    )


# ============================================================
# 2. DATASET LOOP
# ============================================================

for name, (X, y) in DATASETS.items():

    print(
        f"\n{'='*78}"
    )

    print(
        f"S5B END-TO-END MICRO-BATCH — {name}"
    )

    print(
        '='*78
    )


    # --------------------------------------------------------
    # SAME S0 SPLIT
    # --------------------------------------------------------

    tr, te = SPLITS[name]

    Xtr_raw = (
        X.iloc[tr]
        .reset_index(drop=True)
    )

    Xte_raw = (
        X.iloc[te]
        .reset_index(drop=True)
    )

    ytr = np.asarray(
        y[tr],
        dtype=int
    )


    print(
        f"Train={len(Xtr_raw):,} | "
        f"Test={len(Xte_raw):,}"
    )


    # ========================================================
    # 3. FIT PREPROCESSOR ON TRAINING ONLY
    #
    # FIT COST IS NOT INCLUDED IN OPERATIONAL LATENCY
    # ========================================================

    pre = make_preprocessor(
        Xtr_raw
    )


    t_fit_pre = time.perf_counter()

    Xtr = pre.fit_transform(
        Xtr_raw
    )

    preprocessor_fit_transform_sec = (
        time.perf_counter()
        - t_fit_pre
    )


    # ========================================================
    # 4. MODELS
    # ========================================================

    models = {

        'XGBoost':
            XGBClassifier(
                random_state=CFG['SEED'],
                n_jobs=-1,
                eval_metric='logloss',
                tree_method='hist',
                verbosity=0
            ),

        'RandomForest':
            RandomForestClassifier(
                n_estimators=100,
                random_state=CFG['SEED'],
                n_jobs=-1
            )
    }


    # ========================================================
    # 5. MODEL LOOP
    # ========================================================

    for model_name, model in models.items():

        print(
            f"\n--- {model_name} ---"
        )


        # ----------------------------------------------------
        # TRAIN OUTSIDE OPERATIONAL TIMING
        # ----------------------------------------------------

        t_train = time.perf_counter()

        model.fit(
            Xtr,
            ytr
        )

        training_sec = (
            time.perf_counter()
            - t_train
        )


        print(
            f"Offline training="
            f"{training_sec:.3f}s"
        )


        # ====================================================
        # 6. BATCH LOOP
        # ====================================================

        for batch_size in BATCH_SIZES:

            batch_idx = fixed_batch_indices_s5b(
                len(Xte_raw),
                batch_size,
                CFG['SEED'] + batch_size
            )


            if batch_idx is None:

                print(
                    f"⚠️ Batch {batch_size:,} skipped; "
                    f"test only {len(Xte_raw):,} rows."
                )

                continue


            # Raw incoming micro-batch
            Xbatch_raw = (
                Xte_raw.iloc[
                    batch_idx
                ]
                .copy()
            )


            # =================================================
            # 7. WARM-UP
            # =================================================

            for _ in range(
                N_WARMUP
            ):

                Xbatch_p = pre.transform(
                    Xbatch_raw
                )

                _ = model.predict_proba(
                    Xbatch_p
                )


            del Xbatch_p


            # =================================================
            # 8. MEASURE EACH RUN
            # =================================================

            prep_times = []
            infer_times = []
            total_times = []


            for run in range(
                N_RUNS
            ):

                # ---------------------------------------------
                # END-TO-END START
                # ---------------------------------------------

                total_start = (
                    time.perf_counter()
                )


                # ---------------------------------------------
                # PREPROCESSING
                # ---------------------------------------------

                prep_start = (
                    time.perf_counter()
                )


                Xbatch_p = pre.transform(
                    Xbatch_raw
                )


                prep_elapsed = (
                    time.perf_counter()
                    - prep_start
                )


                # ---------------------------------------------
                # INFERENCE
                # ---------------------------------------------

                infer_start = (
                    time.perf_counter()
                )


                _ = model.predict_proba(
                    Xbatch_p
                )


                infer_elapsed = (
                    time.perf_counter()
                    - infer_start
                )


                # ---------------------------------------------
                # TOTAL
                # ---------------------------------------------

                total_elapsed = (
                    time.perf_counter()
                    - total_start
                )


                prep_times.append(
                    prep_elapsed
                )

                infer_times.append(
                    infer_elapsed
                )

                total_times.append(
                    total_elapsed
                )


                S5B_DETAIL.append({

                    'dataset':
                        name,

                    'model':
                        model_name,

                    'batch_size':
                        batch_size,

                    'run':
                        run + 1,

                    'preprocessing_ms':
                        prep_elapsed * 1000,

                    'inference_ms':
                        infer_elapsed * 1000,

                    'end_to_end_ms':
                        total_elapsed * 1000
                })


                del Xbatch_p


            # =================================================
            # 9. SUMMARY
            # =================================================

            prep_times = np.asarray(
                prep_times,
                dtype=float
            )

            infer_times = np.asarray(
                infer_times,
                dtype=float
            )

            total_times = np.asarray(
                total_times,
                dtype=float
            )


            # -------------------------------------------------
            # LATENCIES
            # -------------------------------------------------

            prep_mean_ms = (
                prep_times.mean()
                * 1000
            )

            prep_p95_ms = (
                np.quantile(
                    prep_times,
                    0.95
                )
                * 1000
            )


            infer_mean_ms = (
                infer_times.mean()
                * 1000
            )

            infer_p95_ms = (
                np.quantile(
                    infer_times,
                    0.95
                )
                * 1000
            )


            total_mean_ms = (
                total_times.mean()
                * 1000
            )

            total_median_ms = (
                np.median(
                    total_times
                )
                * 1000
            )

            total_p95_ms = (
                np.quantile(
                    total_times,
                    0.95
                )
                * 1000
            )


            # -------------------------------------------------
            # THROUGHPUT
            # -------------------------------------------------

            mean_total_sec = (
                total_times.mean()
            )

            p95_total_sec = (
                np.quantile(
                    total_times,
                    0.95
                )
            )


            throughput_mean = (
                batch_size
                / mean_total_sec
            )


            throughput_p95 = (
                batch_size
                / p95_total_sec
            )


            # -------------------------------------------------
            # COMPONENT SHARE
            # -------------------------------------------------

            preprocessing_share_pct = (
                100
                * prep_times.mean()
                / max(
                    total_times.mean(),
                    1e-12
                )
            )


            inference_share_pct = (
                100
                * infer_times.mean()
                / max(
                    total_times.mean(),
                    1e-12
                )
            )


            # =================================================
            # STORE
            # =================================================

            S5B_SUMMARY.append({

                'dataset':
                    name,

                'scenario':
                    'S5_end_to_end_microbatch',

                'model':
                    model_name,

                'batch_size':
                    batch_size,

                'warmup_runs':
                    N_WARMUP,

                'measured_runs':
                    N_RUNS,

                'offline_preprocessor_fit_transform_sec':
                    preprocessor_fit_transform_sec,

                'offline_training_sec':
                    training_sec,

                'mean_preprocessing_ms':
                    prep_mean_ms,

                'p95_preprocessing_ms':
                    prep_p95_ms,

                'mean_inference_ms':
                    infer_mean_ms,

                'p95_inference_ms':
                    infer_p95_ms,

                'mean_end_to_end_ms':
                    total_mean_ms,

                'median_end_to_end_ms':
                    total_median_ms,

                'p95_end_to_end_ms':
                    total_p95_ms,

                'mean_end_to_end_throughput_instances_sec':
                    throughput_mean,

                'p95_based_end_to_end_throughput_instances_sec':
                    throughput_p95,

                'mean_end_to_end_per_instance_us':
                    (
                        mean_total_sec
                        / batch_size
                        * 1_000_000
                    ),

                'preprocessing_share_pct':
                    preprocessing_share_pct,

                'inference_share_pct':
                    inference_share_pct
            })


            print(
                f"Batch={batch_size:,} | "
                f"prep={prep_mean_ms:.3f} ms | "
                f"infer={infer_mean_ms:.3f} ms | "
                f"E2E mean={total_mean_ms:.3f} ms | "
                f"E2E p95={total_p95_ms:.3f} ms | "
                f"throughput="
                f"{throughput_mean:,.0f}/s | "
                f"prep share="
                f"{preprocessing_share_pct:.1f}%"
            )


    # ========================================================
    # CLEAN MEMORY
    # ========================================================

    del Xtr

    gc.collect()


# ============================================================
# 10. SAVE PER-RUN DETAIL
# ============================================================

s5b_detail_df = pd.DataFrame(
    S5B_DETAIL
)


s5b_detail_df.to_csv(
    OUT
    / 'S5B_end_to_end_microbatch_detail.csv',
    index=False
)


# ============================================================
# 11. SAVE SUMMARY
# ============================================================

s5b_summary_df = pd.DataFrame(
    S5B_SUMMARY
)


display(
    s5b_summary_df
)


s5b_summary_df.to_csv(
    OUT
    / 'S5B_end_to_end_microbatch_summary.csv',
    index=False
)


print(
    "\n✅ S5B selesai."
)

print(
    "Kirim dua file:"
)

print(
    OUT
    / 'S5B_end_to_end_microbatch_detail.csv'
)

print(
    OUT
    / 'S5B_end_to_end_microbatch_summary.csv'
)


S5B END-TO-END MICRO-BATCH — CICIDS2018
Train=264,580 | Test=66,146

--- XGBoost ---
Offline training=1.044s
Batch=5,000 | prep=8.589 ms | infer=1.589 ms | E2E mean=10.180 ms | E2E p95=10.582 ms | throughput=491,143/s | prep share=84.4%
Batch=20,000 | prep=24.182 ms | infer=4.516 ms | E2E mean=28.702 ms | E2E p95=29.614 ms | throughput=696,823/s | prep share=84.3%

--- RandomForest ---
Offline training=3.849s
Batch=5,000 | prep=8.417 ms | infer=34.839 ms | E2E mean=43.258 ms | E2E p95=43.686 ms | throughput=115,585/s | prep share=19.5%
Batch=20,000 | prep=27.112 ms | infer=42.307 ms | E2E mean=69.422 ms | E2E p95=75.284 ms | throughput=288,092/s | prep share=39.1%

S5B END-TO-END MICRO-BATCH — HDFS
Train=460,028 | Test=115,033

--- XGBoost ---
Offline training=1.653s
Batch=5,000 | prep=4.929 ms | infer=2.329 ms | E2E mean=7.261 ms | E2E p95=8.106 ms | throughput=688,653/s | prep share=67.9%
Batch=20,000 | prep=8.263 ms | infer=6.746 ms | E2E mean=15.011 ms | E2E p95=15.334 ms | throug

,dataset,scenario,model,batch_size,warmup_runs,measured_runs,offline_preprocessor_fit_transform_sec,offline_training_sec,mean_preprocessing_ms,p95_preprocessing_ms,mean_inference_ms,p95_inference_ms,mean_end_to_end_ms,median_end_to_end_ms,p95_end_to_end_ms,mean_end_to_end_throughput_instances_sec,p95_based_end_to_end_throughput_instances_sec,mean_end_to_end_per_instance_us,preprocessing_share_pct,inference_share_pct
0,CICIDS2018,S5_end_to_end_microbatch,XGBoost,5000,1,10,3.208250,1.043660,8.588810,8.996459,1.589498,1.618307,10.180330,10.101573,10.581897,4.911432e+05,4.725051e+05,2.036066,84.366719,15.613420
1,CICIDS2018,S5_end_to_end_microbatch,XGBoost,20000,1,10,3.208250,1.043660,24.181960,24.919459,4.516132,4.965783,28.701711,28.552180,29.613660,6.968226e+05,6.753640e+05,1.435086,84.252677,15.734712
2,CICIDS2018,S5_end_to_end_microbatch,RandomForest,5000,1,10,3.208250,3.848977,8.417166,8.637299,34.838549,35.065949,43.258238,43.327591,43.685585,1.155849e+05,1.144542e+05,8.651648,19.457948,80.536219
3,CICIDS2018,S5_end_to_end_microbatch,RandomForest,20000,1,10,3.208250,3.848977,27.111728,27.986333,42.307087,47.488450,69.422158,69.521369,75.283704,2.880925e+05,2.656617e+05,3.471108,39.053422,60.941764
4,HDFS,S5_end_to_end_microbatch,XGBoost,5000,1,10,1.265449,1.652850,4.929470,5.147716,2.329151,3.059287,7.260552,7.079561,8.105894,6.886529e+05,6.168351e+05,1.452110,67.893878,32.079537
5,HDFS,S5_end_to_end_microbatch,XGBoost,20000,1,10,1.265449,1.652850,8.263206,8.545781,6.746140,6.958251,15.011406,15.015571,15.333737,1.332320e+06,1.304314e+06,0.750570,55.046187,44.940095
6,HDFS,S5_end_to_end_microbatch,RandomForest,5000,1,10,1.265449,9.396868,4.820703,4.987212,34.689412,34.931231,39.512064,39.495273,39.784515,1.265436e+05,1.256770e+05,7.902413,12.200585,87.794481
7,HDFS,S5_end_to_end_microbatch,RandomForest,20000,1,10,1.265449,9.396868,9.440920,10.266221,45.508297,46.135166,54.951780,54.715085,56.153704,3.639555e+05,3.561653e+05,2.747589,17.180372,82.814964
8,BGL,S5_end_to_end_microbatch,XGBoost,5000,1,10,0.029187,1.397926,4.003382,4.149351,2.077465,2.117828,6.082669,6.038465,6.269079,8.220076e+05,7.975653e+05,1.216534,65.816215,34.153834
9,BGL,S5_end_to_end_microbatch,RandomForest,5000,1,10,0.029187,0.353343,3.728092,3.868804,34.481412,34.843065,38.211279,38.219111,38.531678,1.308514e+05,1.297634e+05,7.642256,9.756522,90.238831



✅ S5B selesai.
Kirim dua file:
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S5B_end_to_end_microbatch_detail.csv
/content/drive/MyDrive/ercyris_siklus1_confirmatory/step2_s0/S5B_end_to_end_microbatch_summary.csv
